# Fine-tuned NT-v2-50M: rescan the 15 genes

1. **Part 1** repeats Run B of `finetune_ntv2.ipynb` (same seed, same 8000 / 2000 rows, same hyperparameters), prints the test accuracy and saves the fine-tuned weights.
2. **Part 2** scores every base of the 15 genes' canonical transcripts with that model, the way `10_scan_gene.py --context window` does: one 600-nt window centred on each base, centre + previous token into the head.

Self-contained: the gene inputs from `data/genes/` are embedded below (compressed, checksummed), and every piece copied from the local scripts says which file it came from.

**Kaggle settings:** Accelerator = GPU T4, Internet = On. Then *Run All*. Download `finetuned_scan_tracks.zip` from the Output panel.

In [ ]:
# Same pin as dna-venv (scripts/setup_local_env.sh). Must run before anything imports transformers.
%pip install -q transformers==4.49.0 einops

In [ ]:
SMOKE = True   # True = 200 train / 100 test / 1 epoch, then ONE gene, first 2000 positions. False = the full run.

MODEL_REPO = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
MODEL_REVISION = "81b29e5786726d891dbf929404ef20adca5b36f1"     # the snapshot dna-venv has cached
DATASET_REPO = "InstaDeepAI/nucleotide_transformer_downstream_tasks_revised"   # phase1_common.py
DATASET_CONFIG = "splice_sites_all"                                            # phase1_common.py
DATASET_REVISION = "851f9946252e90c665cdb3cc3eedb78f1f26197c"   # data/label_info.json: dataset_commit

SEED = 42                # sampling seed (phase1_common.SEED)
HEAD_SEED = 0            # 03_train_head.py --seed: validation split and head initialisation
N_TRAIN, N_TEST = (200, 100) if SMOKE else (8000, 2000)   # 02b_embed_center.py --max-train / --max-test
VAL_FRAC = 0.10          # 03_train_head.py --val-frac
CENTER_BASE = 300        # 02b_embed_center.py / 12_repool_center.py

# Run B (fine-tune)
FT_EPOCHS = 1 if SMOKE else 2
BATCH_SIZE = 16
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 2e-5, 1e-3, 0.01

# data/label_info.json: dataset_commit

SEED = 42                # sampling seed (phase1_common.SEED)
HEAD_SEED = 0            # 03_train_head.py --seed: validation split and head initialisation
N_TRAIN, N_TEST = (200, 100) if SMOKE else (8000, 2000)   # 02b_embed_center.py --max-train / --max-test
VAL_FRAC = 0.10          # 03_train_head.py --val-frac
CENTER_BASE = 300        # 02b_embed_center.py / 12_repool_center.py

# Run B (fine-tune)
FT_EPOCHS = 1 if SMOKE else 2
BATCH_SIZE = 16
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 2e-5, 1e-3, 0.01

# Run A head recipe = 03_train_head.py defaults, the recipe behind the local 0.9405.
# The head trains on cached vectors, so this costs seconds and is the same in SMOKE mode.
A_LR, A_WEIGHT_DECAY, A_BATCH, A_MAX_EPOCHS, A_PATIENCE = 1e-3, 0.0, 128, 300, 20

# data/label_info.json (worked out from the motifs by 01_get_data.py; re-checked below)
LABEL_NAMES = {0: "none", 1: "acceptor", 2: "donor"}
MOTIF_OFFSETS = {"donor_GT_plus": 301, "donor_AC_minus": 298,
                 "acceptor_AG_plus": 298, "acceptor_CT_minus": 301}
TRAIN_CHROMS = {str(i) for i in range(1, 20)} | {"X", "Y"}
TEST_CHROMS = {"20", "21"}
# center_meta.json run_key: fingerprints of the exact 8000 / 2000 rows the local run used
LOCAL_SAMPLE_SHA1 = {"train": "d5b52e4e9707cde5", "test": "181e6bfb2b6dcc65"}

# Part 2 (gene scan), from 10_scan_gene.py
WINDOW, CENTER = 600, 300
VARIANT = "center_plus_prev_concat"
HEAD_KIND = "finetuned"              # new track, next to 10's "original" and "hardneg" heads
SCAN_BATCH = 256                     # windows per forward pass
SMOKE_GENES, SMOKE_POSITIONS = 1, 2000
EXPECTED_RUN_B_ACC = 0.967           # what finetune_ntv2.ipynb's Run B scored on Kaggle

In [ ]:
import base64, collections, copy, gzip, hashlib, json, platform, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from torch import nn

T_START = time.time()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # one GPU, no DataParallel
USE_AMP = DEVICE.type == "cuda"      # fp16 autocast on GPU only. Set False here if the loss goes NaN.
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None

print(f"python {platform.python_version()}  torch {torch.__version__}  "
      f"transformers {transformers.__version__}  sklearn {sklearn.__version__}")
print(f"device: {DEVICE}" + (f" ({GPU_NAME})" if GPU_NAME else "") + f"   fp16 autocast: {USE_AMP}")
print(f"SMOKE = {SMOKE}: {N_TRAIN} train / {N_TEST} test, {FT_EPOCHS} fine-tune epoch(s); "
      f"scan: {'one gene, first ' + str(SMOKE_POSITIONS) + ' positions' if SMOKE else 'all 15 genes'}; outputs -> {OUT_DIR.resolve()}")
assert transformers.__version__ == "4.49.0", "restart the kernel so the pinned transformers is the one imported"
if not SMOKE and DEVICE.type != "cuda":
    print("WARNING: full run on CPU will take hours. On Kaggle: Settings -> Accelerator -> GPU T4.")

## Part 1: reproduce Run B

The next four cells are `finetune_ntv2.ipynb` unchanged: data, tokens, model, then the fine-tune.

In [ ]:
def load_splits():
    # 01_get_data.py download(): the repo has no named config for this task, so the
    # task's two parquet files are loaded directly. Pinned to the commit the local run used.
    files = {s: f"{DATASET_CONFIG}/{s}.parquet" for s in ("train", "test")}
    try:
        from datasets import load_dataset
        ds = load_dataset(DATASET_REPO, data_files=files, revision=DATASET_REVISION)
        return {s: ds[s].to_pandas() for s in files}
    except Exception as exc:        # `datasets` missing or broken by the pin: read the same files directly
        print(f"(datasets.load_dataset failed: {type(exc).__name__}: {str(exc)[:200]}; reading parquet with pandas)")
        from huggingface_hub import hf_hub_download
        return {s: pd.read_parquet(hf_hub_download(DATASET_REPO, f, repo_type="dataset",
                                                   revision=DATASET_REVISION))
                for s, f in files.items()}


def stratified_subsample(labels, n, seed=SEED):
    # phase1_common.py stratified_subsample(), unchanged
    labels = np.asarray(labels)
    total = len(labels)
    if n is None or n < 0 or n >= total:
        return np.arange(total)
    from sklearn.model_selection import train_test_split

    idx, _ = train_test_split(np.arange(total), train_size=n,
                              stratify=labels, random_state=seed)
    return np.sort(idx)


def indices_digest(idx):
    # 02_embed.py indices_digest(), unchanged
    return hashlib.sha1(np.ascontiguousarray(idx, dtype=np.int64).tobytes()).hexdigest()[:16]


full = load_splits()

# Chromosome split. The dataset's own train/test files ARE the split; this checks it,
# the way phase1_common.py dataset_chromosomes() reads it from the `name` column.
chroms = {s: {n.split(":")[0].removeprefix("chr") for n in full[s]["name"]} for s in full}
assert chroms["train"] <= TRAIN_CHROMS, f"unexpected train chromosomes: {sorted(chroms['train'] - TRAIN_CHROMS)}"
assert chroms["test"] <= TEST_CHROMS, f"unexpected test chromosomes: {sorted(chroms['test'] - TEST_CHROMS)}"
assert not (chroms["train"] & chroms["test"])
print(f"train: {len(full['train'])} rows on {len(chroms['train'])} chromosomes (chr1-19, X, Y)   "
      f"test: {len(full['test'])} rows on chr{', chr'.join(sorted(chroms['test']))}")

# Seeded class-stratified sample, as in 02b_embed_center.py main()
idx = {"train": stratified_subsample(full["train"]["label"], N_TRAIN, SEED),
       "test": stratified_subsample(full["test"]["label"], N_TEST, SEED)}
seqs = {s: full[s]["sequence"].iloc[idx[s]].tolist() for s in idx}
y = {s: full[s]["label"].to_numpy()[idx[s]].astype(np.int64) for s in idx}
CLASSES = sorted(LABEL_NAMES)
assert all(set(y[s].tolist()) <= set(CLASSES) for s in y)

SAME_SAMPLE_AS_LOCAL = None
if not SMOKE:
    SAME_SAMPLE_AS_LOCAL = all(indices_digest(idx[s]) == LOCAL_SAMPLE_SHA1[s] for s in idx)
    print("sample identical to the local 8000 / 2000 rows:", SAME_SAMPLE_AS_LOCAL)
    if not SAME_SAMPLE_AS_LOCAL:
        print("WARNING: this sklearn drew different rows than the local run. Run A and Run B still "
              "share one sample, but Run A is no longer row-for-row the local 0.9405 run.")

# 10% validation rows carved from train: the split 03_train_head.py makes
# (train_test_split(test_size=0.10, stratify=y, random_state=seed)). Both runs use it.
from sklearn.model_selection import train_test_split
fit_rows, val_rows = train_test_split(np.arange(len(y["train"])), test_size=VAL_FRAC,
                                      stratify=y["train"], random_state=HEAD_SEED)
for s in idx:
    print(f"{s:>5}: {len(idx[s])} rows, per class "
          f"{ {LABEL_NAMES[c]: int((y[s] == c).sum()) for c in CLASSES} }")
print(f"train -> {len(fit_rows)} fit + {len(val_rows)} validation")

# Label names re-checked the way 01_get_data.py found them: donors carry GT at offset 301
# (or AC at 298 on the other strand), acceptors AG at 298 (or CT at 301).
def has(seq_list, a, ka, b, kb):
    return float(np.mean([s[ka:ka + 2] == a or s[kb:kb + 2] == b for s in seq_list]))

o = MOTIF_OFFSETS
for c in CLASSES:
    rows = [s for s, lab in zip(seqs["train"], y["train"]) if lab == c]
    d = has(rows, "GT", o["donor_GT_plus"], "AC", o["donor_AC_minus"])
    a = has(rows, "AG", o["acceptor_AG_plus"], "CT", o["acceptor_CT_minus"])
    print(f"label {c} ({LABEL_NAMES[c]:>8}): donor motif {d:.1%}   acceptor motif {a:.1%}")
    if LABEL_NAMES[c] == "donor":
        assert d > 0.9 and a < 0.3, "label 2 does not look like donor"
    if LABEL_NAMES[c] == "acceptor":
        assert a > 0.9 and d < 0.3, "label 1 does not look like acceptor"

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION, trust_remote_code=True)
special_ids = np.array(sorted(set(tok.all_special_ids)))
max_len = int(min(tok.model_max_length, 2048))


def tokenise(seq_list):
    # Token -> base mapping from 02b_embed_center.py spans() (slow-tokenizer branch):
    # special tokens are found by id, the remaining tokens must spell the sequence exactly,
    # and their cumulative lengths give each token's [start, end) bases.
    # Previous token = the adjacent token before the centre one, as in 12_repool_center.py.
    enc = tok(seq_list, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    ids, am = enc["input_ids"], enc["attention_mask"]
    ids_np, am_np = ids.numpy(), am.numpy().astype(bool)
    c_idx = np.empty(len(seq_list), dtype=np.int64)
    p_idx = np.empty(len(seq_list), dtype=np.int64)
    seen = collections.Counter()
    for i, seq in enumerate(seq_list):
        body = am_np[i] & ~np.isin(ids_np[i], special_ids)
        pieces = tok.convert_ids_to_tokens(ids_np[i][body].tolist())
        lens = np.array([len(p) for p in pieces])
        assert int(lens.sum()) == len(seq), \
            f"reconstructed length {int(lens.sum())} != sequence length {len(seq)}"
        assert "".join(pieces) == seq, "reconstructed tokens do not spell the sequence"
        end = np.cumsum(lens)
        start = end - lens
        pos = np.flatnonzero(body)                     # input positions of the real tokens
        c = np.flatnonzero((start <= CENTER_BASE) & (end > CENTER_BASE))
        assert len(c) == 1, "expected exactly one token covering base 300"
        c = int(c[0])
        assert c >= 1, "no token before the centre token"
        assert end[c - 1] == start[c], "previous token is not adjacent"
        c_idx[i], p_idx[i] = pos[c], pos[c - 1]
        seen[(int(pos[c]), int(start[c]), int(end[c]) - 1,
              int(pos[c - 1]), int(start[c - 1]), int(end[c - 1]) - 1)] += 1
    return {"ids": ids, "am": am, "c": torch.from_numpy(c_idx), "p": torch.from_numpy(p_idx)}, seen


t0 = time.time()
toks, seen = {}, collections.Counter()
for s in ("train", "test"):
    toks[s], n = tokenise(seqs[s])
    seen += n
print(f"tokenised {sum(len(v) for v in seqs.values())} sequences in {time.time() - t0:.1f}s; "
      f"input shape {tuple(toks['train']['ids'].shape)}")

cls_first = bool(all((toks[s]["ids"][:, 0] == tok.cls_token_id).all() for s in toks))
print(f"<cls> (id {tok.cls_token_id}) is the first input token of every row: {cls_first}; "
      f"eos token: {tok.eos_token}")
for (ci, cs, ce, pi, ps, pe), n in seen.most_common():
    print(f"{n:6d} rows: centre token = input index {ci} (bases {cs}-{ce}), "
          f"previous token = input index {pi} (bases {ps}-{pe})")

# One row spelled out, to read by eye
r0 = toks["train"]
ci, pi = int(r0["c"][0]), int(r0["p"][0])
row_tokens = tok.convert_ids_to_tokens(r0["ids"][0].tolist())
print(f"row 0: input[0] = {row_tokens[0]!r}; input[{pi}] = {row_tokens[pi]!r}, input[{ci}] = {row_tokens[ci]!r}; "
      f"sequence[294:300] = {seqs['train'][0][294:300]!r}, sequence[300:306] = {seqs['train'][0][300:306]!r}")
CENTRE_INFO = [{"rows": n, "centre_input_index": ci, "centre_bases": [cs, ce],
                "previous_input_index": pi, "previous_bases": [ps, pe]}
               for (ci, cs, ce, pi, ps, pe), n in seen.most_common()]

In [ ]:
def load_model(repo):
    # try_dna_models.py load_model(): NT-v2 registers AutoModelForMaskedLM but not AutoModel,
    # so both are tried.
    from transformers import AutoModel, AutoModelForMaskedLM

    errors = []
    for cls in (AutoModel, AutoModelForMaskedLM):
        try:
            return cls.from_pretrained(repo, revision=MODEL_REVISION, trust_remote_code=True), cls.__name__
        except Exception as exc:
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError(" | ".join(errors))


def token_states(model, input_ids, attention_mask):
    # try_dna_models.py get_hidden(): last-layer vector of every token. Same logic, minus its
    # torch.no_grad() wrapper, because Run B needs gradients (Run A adds no_grad itself).
    out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state
    if getattr(out, "hidden_states", None):
        return out.hidden_states[-1]
    if isinstance(out, (tuple, list)):
        return out[0]
    raise RuntimeError("could not find per-token vectors in the model output")


def pooled(model, t, rows):
    # center_plus_prev_concat: [centre token ++ previous token], in that order
    # (12_repool_center.py: np.concatenate([cen, prev], axis=1)).
    h = token_states(model, t["ids"][rows].to(DEVICE), t["am"][rows].to(DEVICE))
    r = torch.arange(h.shape[0], device=h.device)
    return torch.cat([h[r, t["c"][rows].to(DEVICE)], h[r, t["p"][rows].to(DEVICE)]], dim=-1)


def make_head(in_dim):
    # 03_train_head.py: input -> 256 -> ReLU -> Dropout(0.2) -> classes
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, len(CLASSES)))


def evaluate(y_true, y_pred):
    # 03_train_head.py evaluate(): accuracy, macro-F1, per-class precision/recall/F1, confusion matrix
    from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                                 precision_recall_fscore_support)

    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)), 4),
        "per_class": {LABEL_NAMES[c]: {"label": int(c), "precision": round(float(p[i]), 4),
                                       "recall": round(float(r[i]), 4), "f1": round(float(f[i]), 4),
                                       "support": int(s[i])} for i, c in enumerate(CLASSES)},
        "confusion_matrix": {"rows": "true label", "cols": "predicted label",
                             "names": [LABEL_NAMES[c] for c in CLASSES],
                             "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()},
    }


def print_report(title, m):
    print(f"\n--- {title} ---")
    print(f"test accuracy {m['accuracy']:.4f}   macro-F1 {m['macro_f1']:.4f}")
    for name, v in m["per_class"].items():
        print(f"{name:>10}  recall {v['recall']:.3f}  precision {v['precision']:.3f}  n {v['support']}")


t0 = time.time()
backbone, loaded_via = load_model(MODEL_REPO)
backbone.to(DEVICE)
HIDDEN = backbone.config.hidden_size
n_params = sum(p.numel() for p in backbone.parameters())
print(f"loaded {MODEL_REPO} via {loaded_via} in {time.time() - t0:.1f}s: "
      f"{n_params / 1e6:.1f}M parameters, hidden size {HIDDEN}, head input {2 * HIDDEN}")

In [ ]:
# Label -> class index, as in finetune_ntv2.ipynb's Run A cell (the only thing Run B needs from it).
to_idx = {c: i for i, c in enumerate(CLASSES)}
yi = {s: np.array([to_idx[v] for v in y[s]]) for s in y}

torch.manual_seed(HEAD_SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(HEAD_SEED)
head_b = make_head(2 * HIDDEN).to(DEVICE)
opt = torch.optim.AdamW([{"params": backbone.parameters(), "lr": LR_BACKBONE},
                         {"params": head_b.parameters(), "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
grad_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
loss_fn = nn.CrossEntropyLoss()
y_t = {s: torch.tensor(yi[s], dtype=torch.long) for s in yi}


def predict_logits(t, rows):
    backbone.eval()
    head_b.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), 4 * BATCH_SIZE):
            b = torch.as_tensor(rows[i:i + 4 * BATCH_SIZE])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out.append(head_b(pooled(backbone, t, b)).float().cpu())
    return torch.cat(out)


fit_t = torch.as_tensor(fit_rows)
n_steps = (len(fit_t) + BATCH_SIZE - 1) // BATCH_SIZE
gen = torch.Generator().manual_seed(HEAD_SEED)
best_b = {"val_loss": float("inf"), "epoch": 0, "state": None}
hist_b = []
t_b = time.time()
print(f"Run B: {len(fit_t)} fit rows, {n_steps} steps/epoch, {FT_EPOCHS} epoch(s), fp16 {USE_AMP}")
for epoch in range(1, FT_EPOCHS + 1):
    t0 = time.time()
    backbone.train()
    head_b.train()
    perm = fit_t[torch.randperm(len(fit_t), generator=gen)]
    total = 0.0
    for step, i in enumerate(range(0, len(perm), BATCH_SIZE), start=1):
        b = perm[i:i + BATCH_SIZE]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = head_b(pooled(backbone, toks["train"], b))
        loss = loss_fn(logits.float(), y_t["train"][b].to(DEVICE))
        if not torch.isfinite(loss):
            raise RuntimeError(f"loss is {loss.item()} at epoch {epoch} step {step}. "
                               "If fp16 is on, set USE_AMP = False in the device cell and rerun.")
        opt.zero_grad(set_to_none=True)
        grad_scaler.scale(loss).backward()
        grad_scaler.step(opt)
        grad_scaler.update()
        total += loss.item() * len(b)
        if step % max(1, n_steps // 10) == 0 or step == n_steps:
            print(f"  epoch {epoch} step {step:4d}/{n_steps}  loss {loss.item():.4f}  "
                  f"{time.time() - t0:.1f}s", flush=True)
    train_secs = time.time() - t0

    t1 = time.time()
    val_logits = predict_logits(toks["train"], val_rows)
    val_loss = float(loss_fn(val_logits, y_t["train"][val_rows]))
    val_acc = float((val_logits.argmax(1) == y_t["train"][val_rows]).float().mean())
    val_secs = time.time() - t1
    hist_b.append({"epoch": epoch, "train_loss": round(total / len(fit_t), 5),
                   "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 4),
                   "train_s": round(train_secs, 1), "val_s": round(val_secs, 1),
                   "train_s_per_step": round(train_secs / n_steps, 4)})
    print(f"epoch {epoch}/{FT_EPOCHS}  train loss {total / len(fit_t):.4f}  val loss {val_loss:.4f}  "
          f"val acc {val_acc:.4f}  |  elapsed: train {train_secs:.1f}s ({train_secs / n_steps:.3f} s/step), "
          f"validation {val_secs:.1f}s", flush=True)
    if val_loss < best_b["val_loss"]:
        best_b = {"val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                  "state": ({k: v.detach().clone() for k, v in backbone.state_dict().items()},
                            {k: v.detach().clone() for k, v in head_b.state_dict().items()})}

print(f"best epoch on validation: {best_b['epoch']} of {FT_EPOCHS}; scoring the test set with its weights")
backbone.load_state_dict(best_b["state"][0])
head_b.load_state_dict(best_b["state"][1])
t1 = time.time()
pred_b = np.array(CLASSES)[predict_logits(toks["test"], np.arange(len(y["test"]))).argmax(1).numpy()]
test_secs = time.time() - t1
run_b = evaluate(y["test"], pred_b)
run_b["training"] = {
    "backbone": "fully fine-tuned", "features": "center_plus_prev_concat, straight into the head",
    "optimizer": f"AdamW(backbone lr={LR_BACKBONE}, head lr={LR_HEAD}, weight_decay={WEIGHT_DECAY})",
    "batch_size": BATCH_SIZE, "epochs": FT_EPOCHS, "steps_per_epoch": n_steps, "fp16_autocast": USE_AMP,
    "selection": "lowest validation loss", "best_epoch": best_b["epoch"],
    "best_val_loss": round(best_b["val_loss"], 5), "best_val_acc": round(best_b["val_acc"], 4),
    "history": hist_b,
}
run_b["seconds"] = {"test_eval": round(test_secs, 1), "total": round(time.time() - t_b, 1)}
print_report(f"Run B: fine-tuned NT-v2-50M + head (n={len(y['test'])})", run_b)
print(f"Run B total: {run_b['seconds']['total']:.1f}s")

# ---- insurance: keep the fine-tuned weights (best-validation epoch, the ones just tested)
best_b.pop("state")
STATE_PATH = OUT_DIR / "finetuned_ntv2_runB.pt"
torch.save({"backbone": {k: v.detach().cpu() for k, v in backbone.state_dict().items()},
            "head": {k: v.detach().cpu() for k, v in head_b.state_dict().items()},
            "model": MODEL_REPO, "model_revision": MODEL_REVISION, "variant": VARIANT,
            "head_arch": f"Linear({2 * HIDDEN},256) -> ReLU -> Dropout(0.2) -> Linear(256,{len(CLASSES)})",
            "classes": CLASSES, "class_names": [LABEL_NAMES[c] for c in CLASSES],
            "test_accuracy": run_b["accuracy"], "best_epoch": best_b["epoch"], "smoke": SMOKE},
           STATE_PATH)
print(f"saved fine-tuned weights: {STATE_PATH} ({STATE_PATH.stat().st_size / 1e6:.0f} MB)")
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"\nRUN B TEST ACCURACY: {run_b['accuracy']:.4f}"
      + ("   (SMOKE: tiny sample, not a result)" if SMOKE else
         f"   (finetune_ntv2.ipynb scored about {EXPECTED_RUN_B_ACC})"))
if not SMOKE and abs(run_b["accuracy"] - EXPECTED_RUN_B_ACC) > 0.01:
    print("WARNING: more than 0.01 away from the first notebook's Run B. GPU training is not "
          "bit-reproducible, but this is a larger gap than expected; say so when quoting the scan.")

## Part 2: scan the genes

Gene inputs (embedded, checksummed). Scroll past the data block.

In [ ]:
# data/genes/<GENE>.json exactly as 10_scan_gene.py get_gene() cached them from Ensembl (GRCh38,
# 1,000 bases of flank), gzip + base64. Order and gene set from data/genes/scan_gene_list.json.
# sha256 is of the original file; checked on load.
GENE_LIST = [
    ('MMP9', 'ENSG00000100985', 'original'),
    ('SOD1', 'ENSG00000142168', 'original'),
    ('PCNA', 'ENSG00000132646', 'original'),
    ('HMGN1', 'ENSG00000205581', 'added'),
    ('BPIFA1', 'ENSG00000198183', 'added'),
    ('CPXM1', 'ENSG00000088882', 'added'),
    ('WFDC2', 'ENSG00000101443', 'added'),
    ('CFAP410', 'ENSG00000160226', 'added'),
    ('ERGIC3', 'ENSG00000125991', 'added'),
    ('YTHDF1', 'ENSG00000149658', 'added'),
    ('MTG2', 'ENSG00000101181', 'added'),
    ('TMPRSS3', 'ENSG00000160183', 'added'),
    ('CDC25B', 'ENSG00000101224', 'added'),
    ('ZNF512B', 'ENSG00000196700', 'added'),
    ('OGFR', 'ENSG00000060491', 'added'),
]
GENE_SHA256 = {
    'MMP9': '9fa60c86290f2124a1735a9fa0591b6d14ac4c97e6c8b0062a393e8606c33b80',
    'SOD1': '586ec122993cb1a049579a9f88637fbbf2e742243bc94a39bea399db1bf24209',
    'PCNA': '82d81a9471b1cfa8804d97ab51d4e4b15dc3e36f9460ccab447691546c082371',
    'HMGN1': '568297694fd24a91fd65abbc186621d12604f4f50f0c5751e8b28e07030d7cb8',
    'BPIFA1': '7bf3c0e3194b47caab6a88f19cac6049679cf3842f0c8f71d9b3f660ee6ccb4f',
    'CPXM1': 'a811830b1ea1ea6e3a9c5a7638293ee3aa24b6134cfdd15ec09278c383154820',
    'WFDC2': 'ded56bf2f4498f3fbeded9cbe0878f96e5a8e702c2dd1dff34144a7df5077a9a',
    'CFAP410': 'f3304f3f940b95df9c99b68ffcd18fd9e9f9fbaf22e508b760d10d847b9f244d',
    'ERGIC3': 'e57174478a80cd2d2c8f53bb27b9703e4d36e903695753a974675ae2383b50a6',
    'YTHDF1': 'd74e97a0905e53d495855acfca7707e6fe6587591b35a5af040de2685d5ae7cd',
    'MTG2': 'c8e548cc5dc4a95265a415e1272acebe8563e40456c4a92afd0379c8007730b7',
    'TMPRSS3': '3ffcbe975b457acd9644c6013f4ae37b4e64870241cdbae339c07bf4f04b2952',
    'CDC25B': '4515581658903c6ecd21d44860d8bddb5502c08f9b8a0f28747cba7b07ac55b1',
    'ZNF512B': '9fc1b861a392f4b91a490f6f9f94051516197ce8947cb517d9db9b827bdb0d11',
    'OGFR': '552ac5afa63db1c7195404caa14f2ab8240f478b3c94e9bd83ab9591c3a9b216',
}
GENE_BLOB = {
    'MMP9': (
        'H4sIAAAAAAAC/+1a2W5cxxH9FYHPgj0LOeT4rVAI+slBkNSbYQi0NJKISKRC0kAEw/+eW2fpGcqKESfI4sBDkZq5S9/uWk6dUz0/nD18fP/d3buzr56dff31H/Znz5+dvTncHl7cvOpDv/v9n8aqX+vVan910Wdfvr2/e9/nNqv++PB4ff+4fDzfrVZXV7v9cuhw+4oH1ruL3RrX3F/jWH+4fng4vP/u3cceYvwx326vepjvbu4eP3449MEP93ePh5vbFy/vXt3cvumTffvDy/ubD4/HaRWmtb3cbLerT6752bEOf727fVhOffMNZrxfXT3H3Per3fm3z5/h6P5qt8NRrFpH16vzq62O7rbreXS/2fDoenWx8tH1+lzXrrfr3Tx6cXmuo5fnlz66WW8veXSzXW/n0fONj15cbXx0u7m44NHtxfZ8Ht1d6Nrt5X4+7XwZg0eXoea159vLtY6eH4/uNher59Nl33771KBPfAyLnZ79ibuP52DoH87+rtN+1lM3Dy9eXt/e3d68vO7wfLz//vCb//5B//34/NlPzH61v9qstr/U7K+v3z18xu7I9P+Y3Xe28K/W7uf/K3b/f4zs/f5iu9n9KiP7v2b31f5ftvu6H77Z/2b3f2u8dy1+/e769s/NnxarLx/vD29u7m6flOVLUi+dOZbkSzGww19efHj3fZv7LMbIGiNGjew/kVERtfztExnLyVo+ZvWp5e/yr/oGnIzlHpzMzD6by2F+CAy4jDSiahlzecIy4nJ9X7c8IXuM7L89QPWA/K+fsdwbfdvySjwgdFvmWK5ZnhKJUXuYwLR7Un35cttyZvlZjixjV8+yj3CWfb4H7Ef0ID3NnlNy/v3q+/zqoXuGPfloU5QWNrAwjNx39xXFtfSlo396Nr3a6nX1KAXD9amBA7BsPxEPKhuj7dWPTa4Tjxn94qOw2LZZ8QMc01Zo72EpONW/XFcPOOCoXjHXwkVxMcH5lM5jqcHrNKKio12EC2BZvm9DJwPDJ+GLDgsMnLAIwkXLQGi1h7FMzQBuwgiFqcPlhTUiJvCI4ExG6lYOztExLkKvF1cIFX7mLw4hUhiqsEEy5D3dRDDpDrrdbsBU8G4w+BmsuIozpFtG6Kr5GVbCv4JpEBTMoY6NxFF4P37RDx+teSJgwjMqGp2+wFSZfyOZnKXQlCFw75AtYFdkMgOSaccwSK6TD2W8Y+7zzVBkw0NcFAMw+ewen1Mv3kgH025YiPMdz2Xkc26MbqwnmYsd8sqnGY4M5Bz0HQIRkctZMGCZCFgaHROcr+yFm+grhu1QiGNymkQxmHETPT1D4hiiIwSoxeCPCE0f1zO8S4vSSf6n/FJc9y3FYUc5+pDQJezGIkM4AfO0j0um0w3z+YKg0CIG7Y74KJkIWKoQCmUWJ+ScZTIS23SAKEtbIsQKBuLNDI6OpFRZQbgSlx29whBOAyCLtB40PyGC+UqgI+4NxDBrBuaKWaEu8MV47zdBFMqwY9pGuIHA0A+h3cLziDLS9TyYbKmE5mDlEOjrG0D4+I7f0IP9F/ggmHJoyrMYt1gAUzHPoAeQyvIoaDSkfpBNdKaKrBbU+F+MJhg76UCN0nFNfCrMs1KRRZibaUxEZZ1XvXLF6ycgeLCEUorbWyx9iYzG4UFgdDLmXCAMz5gV5uJsstC69NPJzHtmaDuTM1amAGWcN+ncyRBmp2hG8jpmTA0HTC8i2ywO9kINclgjjwR9Y6KKodL4SFyHR2Y287nkSUE304jEoQBoBPGjz8eYkfvkBfisIWeQ5aC2A9k5fbl3aFTSIBdYuKNC2DLzEqRHlTO1ahd1Eomj+eBeBhq9FuAxJiooQKx/BFDkeyQHRtzwfRJASlUJF+fgPMgyMWpiDSlP9xVkXe0l4leVYIi1mSWE9jSXodmPIedkwpyZ2wzQoyEZS2GXA6wzZ8nXOkTZBsAWxs8T3xJjmJV29jBi6GEwBAoS1ldifrQOQg/TSpO8cpQjH+hRzZRe6AjKVGVtn3PVCEsCiFcKc4Xggx8B+UlmT5RJuy59iNYIBzi9lpxpu5oOHyyETiXytHaaXMT4Gk+GLSIB0UwIiihg1jM6EgssIE+a4dHkpOVwNkyByOgAJcgMxpm8Q+Yh5MTcCSQYK8R+yuADqxTPqoDZ9yXjJ/3PfCCFYuh7JswpYTuBV78kA1ifKMxQrOjxKDuTcUnpuFx5PZRbhiniDQ1XDij6Igke5J8iXVEGEpY7YCPrIFVXUJ+FWdAIV3DqgRBGqvYqycXBTbnlqpQVUAuLXDaMXqTE5kLpZCqKArsrrS3srjxx1/iMu1gNWYNs1lN3pdSgSOmJu0CyUe2Jh5wTJo7KooxlbCpaAbwKa4LkyHHku0l9g7ignKZ3SPOoU4ejp4bXlxK7tCXvYoVK65AkufR8MhQbnCi1tRcmWk1NQMmoQmnCPGSlDOme8LVmYkMeHNKaRbcQcAu4A5PSZBhkJruUCOuu4NyFJqaL3RiwssO1dLCF33DGp/gy0iClJF1MytlDYltxBMdwTg5VrzD1V+kjIwq3OIJcletG1IQ0wDFqrE5hEWOhooCshWW4BKjDomOY+qtk1MQHjqsSacUsOjm177DKRUekbwPyoU4AcDhQqICxeEmTM/BQwlEC6QnYB0xSWcSeELWGEhxaxDELZlVSGAyiEhNVZR36O7m1pYbrxJAPSdQjBTwTbqT3rY8iaFKi8ZhKUTQSxjrhTm5qMAtKlTzYAylZZGrAEHmQ1iRIUgUOd4RC4rYmynK58hUNIKUjcSjRyzC3xODDJW6GNUmImuVR6vCwkORE61B2xdCwYuLhLhTlGHJSMz2VW5I5lUeBpYhRiyhE94PUTvhDOkB4F6ONqSO5oPaBZdD8F6Le1rtmYzkLY1iQKBh07BiCAOySTGWZZ48NiS61LskP+cfmnPoLQFeul/p/VifUv3S7Uj3NsBJh/0WtEOnZzEkTVOUFAnxf7nyo7SB4niAs9TY5zeePh4Q0QU2UM9RJHBZQZEdDdExcIK13TPfgMIMdzS+EBe9EYXPrzs5IZR+JezluHdJpjxECReZcQtzupSnCyONLy7WRMeMaQ6SGJiQpJEyHoT6J2SEwKhUVsilLyBO6noNVnkWHPRstTyU6Begqlap31NHQ0rD3rORcHskRFWGxj+ZYneKjZxXk50yHdMeIPWf1CCpcTAcb3epRBjs7g9GdzpNJK1OMn4Aj9ZSq0Jkxy1lOgSsUni1LSTfmCctvshrELDEcl0kG56gFKH1pypOuBJOii6RPFcTGFPUdFZAFgrGZIUeYsteGO81+gHCqROK49CQ7mXGiJh+78GoT5EmBD/UThmSGeNGTQEtNnZNk0A5yIrrZrSStd+RRK4p/xURka5JS3zrdnhURwZJzCND48Jnig7pMxZa0j8VZ6zCmh7viRUls8ghGqHgpCxHSeymz1D5BqLilxJQkFd6lG3chEuROnqq8+Jt4ibZhmNKu/sRUVTiWQ6pg6i9rb21Z5Gl3RV0I+UhFpGJei1wtNVOYkWopJdPQNcWdCPU1XOB5Y5VAOl3urWdLaKzyaxUpDlWGArY4yVTB1QU+oa6ceTvXRiqqbapUz6doXgH4qYFIjEIFLFnZZGm8iPLcs4tS3zxVuEIbceW2OV2AfpdCgZyRZvTuTrjdPHvZ6quF0t07TtZplo3ow5pjDPYktemm/TuRGQt/Cnv2QaTJdUekp50CcERh+XlllqPmUZW9LV6cJ4yQfRgTaMgWDk/ruXzX7OuxTaPdTXXwSATrGPTU7FQ4UelFcBuwrMBJ5rgx6IN+5ZP/5kvqJLQDxYUIBZ9s5BIhSmxCTC3SapmgwjLv6qq0i7mJOxSvcxO3xjH02BAf7EaU8k9cihEjwqBdJG3vap6kmfQzmTrodrrZczqNcFXwNPK4lzzUQMj8xJDH3WNhD34HN4ZBS8v0j4/TZpI2VyRicZUUksiU/ULvak82vB7tjoSMUAKj2WSJGa1mWeWtcfZqmTvD'
        'SKy0KW5zMZJSN4feJ/m/W76squaFxYouDBvWOAQykX8RSKlsbcBxO04IOI9Jfg1FITUS0NEbT+T21EtSgtoUL9JHVQtr1KA1tWtPCCbmcBtYdcqh5xJ73B05jmEQGyLIavtJ+ZoXEFTVwLFU196YSAs0jjSNRLFqKuUF84ena24AU0WGmWBKhKqR5RarO1qkVtRuyAhW3zIlidmfHjb1mOSoZidG8KYM9Ga9uoQYGiyDhUoAbyGcMy9ofNTkHMf35feoCvC1EObTl744kFShp5eWxEp5G3RMmZYmngQaefuT7cB02yD57QF93SKsXcgf3ItHkWOmzV4dQuE4VVNtBJmInfS62z6Syimp/GRnUk/m3o6loh0UbA6HN6hCU7ZUnhst/9QdoAvS9+wLIgSevOrUvrTj3I0FLhF0VCVHme5T1nPbJoSk+g6M5AwhjEVkMHLclPVXb8K9A9CpcrRL+kWe7oAzQNSxIFVTbLqhFO7gDTbd3E0o9fzcAQsXb1Eg7RIWiROXxBZUTj+Xvosy29j8jot2bfjVIa0R41QIWeeP+ipzJ1ZfHiDIzf1haDbpWSEfkYWRw5UR6pjxLBDaHpKPtIWiXZ8SsRn8mlSpVqkdzC9F0Uz9fbvXh8eXbw+vXrzWF/XfPj5+ePjqyy/vDw+PXxxu8fX7L+7u35z9+DfSY/pw/i8AAA=='
    ),
    'SOD1': (
        'H4sIAAAAAAAC/81c224cVw78FUPPRqK59+SNYBZ82wU2fAsCQZbGtrC25JUUYI0g/759qoqnx5J8kTNybMSWprunm+Qhi8U6jfxxdPP+7YurN0c/PTv65V8/z46ePzt6tbvcnVyct0P/+Ocvcdz+zJbz2XpoZ89eX1+9befmuPjm9vT6dvy4mK1X2/VsMx7aXZ7zwHrYLma45voUx9qH05ub3dsXb963W8S//fUCd31xcXX7/t2uHXx3fXW7u7g8Obs6v7h81U62r9+cXV+8u53MSpg13zTL7lzzyXvt/nd1eTOe+vVXWrxdPKftw3L22/Nn7eh6sdke4+h6MQzrOrpeLrc8ul7Nhjq6ma8GHt0sNqs6Oiw3s+c9BL/99qGBH8asWbB/9l74pnMw/I+jO0FYDNvtdvXZKF7cnJydXl5dXpydtuV+efrmZvdAQNZrBWQzLJ8uIH8+f3bPj+VmPTtef9KPE//5l5PLq9uT893Li8vd+SP8emihZ6vyYLZZ/GVvlx/xa0rSx6zP7fXvu78tXx9enuPtcvlEyzPem88/3hwP39avYbNZzIeDlY+saJj5pBZvD2XxatYLfr0X+dW3ifzi+FB+LA5ZGfNH+zH7G/wY/5Efi+XqSVdpfijvOn5tN9tvW+fb+bCZbb7VKm2PJz9Wh/dj+DuqZr08uB/bb+XHanisbfPj7xJh54/2Y/aUfvyF9R9G6+7SvdGIm93438nb3fnF6e3ufKQSZ6fvH2FkG0HuJsLBoaSZvjm46cPw5bU4X3yNQ/PhEw4Nh3do+6UOfdb08fiDps+OV6vVdnFw07fCuuPF6vhp+tR8/hGHNmObOrxDi+Gv0qP19quqhQ4tD+7QcvblDh20A9OhJ6iW469CrvljTR+elDx8zuKxkjefsG17+Ep+BMMZviZPVtvVRxzaLkd4+i7nm+2jkqb5sf4u/dhbj+WmEmz4SKuYLWar9fx79+POejTt8OWb08v/NP109GL8eL17dXF1+YGMOFB61ZlJQtxKgd399+Tdm9+b+UfuYREZ418LG386/uZ41NPHUxbZPpn5+Mnb2fDxQ7s0DYfHEzkez/ZrOzL+YviJL+DuiUc4vtbu0T47z1l7WHtMe3L7F8+UKe3BvI5G8qrxM+0bL8J17TE4PprcrRzPjb+Pn8a/7SbjU5tHssxMf5v94/FoPyJoeotF87rd0MZf8MX2o51uT23HEBScbjfAJ4Qk2rXtsOFLeHCLjszH0fbN1B1xEYKXvE/wC9H8GK8KBFEx9WBAGcfxMELczGzfGT1odgWe3Z7ZwtBvhSsNweQyYf3aPdrT24/2JdzOcd6w7M24tkrNPf7At7EMbeH4uVxvQQyDGwELvH0pdRNHTsERPry5Vb6ZZZ3iKrRYIbfaco43QsrgyYgSlzCxvOFykxZrEfAU5AEt4JOQGQlDsLZ4FNMXK2tco1TE2iU8n8h8b6bgUUwaD5odCAJzA3nIpwbWoGWkYXUQoXYr+oQ/htTEjXHXwI0z+ZNPoUv6OjIdoWX9Yt20ILQOBdLMDl6ANWF2KKFQ9DrpLDBnpiAxUAM6Y/QC2ICwtDMpq5EIzi8HPQxmR6KwgtDRbE24qIfg2ZksYHOED8FMmp04wnPBANBDLk+tTEsTfpeXA0xcUJEyl7d2JJDzN2QTfm8xd1yXAJrmLAGB8EW7eQdAC0EGF0QtCEzuxxAsz+xrxyM87EhB5rAjn5RQ/E2JQT+66XUci4b0YSAjtNDEUCyzLvZ+UHnmujGyxbPjcGb0h7jcTyyx/FBU8IUKHeKVtWSJw8gGgA4D2+oNLhCk4TYjCpSmfVp/uCT39BsQu8XQkvZi5ZKVggLGSiZS16qOkhehIRnPIxeAe0zglnXGn8aTzRPBETsYmxxbB5Io2VCYKEbgY9IwQxgEmpKsNvYqpmwIHyzULHCI0Ufc+J/BYq2qLkkAGXuYCtjUgHjr9jjnVcnO18Cn1VbzzNiqdI1wN7EyaI8E/PZQI9wR/9DaEwb2yiPs0unsPZA1b0JWHk0a0aBc3QA+JGMJNCAApRiGBw0NtkYYrMzj7TOVTAgtFsII8QBdVLxwPtkVieEeRDOucrUudTq0SCdus7Ulc0mtwoHcre2iDyYpD1qZ9fbPluaplq9r1MRYHUwDJABRKWA6Q4JqQLQRaQTU2NUZH9wOJSZOwvwvKsM8kDHGRhtqjMleSM+Fz7IdfauWggmZxPSgnyg/eYkFZYkZeyruRIqSegCzH+gog6O4QTI1ncnoXNCeSDyMODovTQaGN3Y2VC4paIpx7ZiavLoWgvCK9igUI4FULrFegJgGWgeaW609CT2ipq0KwsmLgLm83sVyUF5BUuRqowJRE4KBf+wDDjKODZVsAPwvCKSivTCeBC5BGpwVrtbF5aeR1UmACDIiCR3FHMgWiL1pVR40kGhthFfGmIaoYrFMgQyziknSELJedl+mJSio1/qZCc+xen2N1AHJiHkDZhO7EG5Sley8tfIJhlT6obYRXq4ghwj1I3jJumMp8tZMpiBVrjJg7Sp5iEF0W54SQll51i9lGeCogxvLAaCU8s2UoUEijju69cbDLuVFMGAtMxux47VRdMMLgDVmIULdc8AWe68lfQANYRGrhzgpqvuE3kE+QzgsSNL8oWp3ApvANBQ6zXlpRXmJ16Az6BBYg9C0VexYy0HEFZFnzpsXBvBEFLCyWM10ayUvXUuR5GoBYCJqdpruUL85zRls4JgHaiYz7/Mp8o/zUmYvVyVpasRil2UPJ8QBG1lyzFw1K4DP3gczZSyTIpnbhEGiglwl/Gctm1oGg1jdhgmFDCPzRhZwIsyi3kk+JY4QGvlqoiXnSq0jh0B1pvCiiuqbwjPWBlZPVIPsVBMCAtjCIELm7KcE5eyA6Iy5tAPA3cRLos+CZUapBxxXaIbUA3xP6sC+esBQC7bKTCxOTuqBc1mNy3RXPfAP1IP0STywKanJEKRCmGrX2NiZaZGa72CWJjSVvchgEJ1M+R41MVcjrD+qAisWBpy23mL74BSyTdQz1JA5jYfmZRFA8cmpMO8FMYT9Hwli9iD6FwQxJw2mxv8auKP+JQr2Q0paobrEGfEcCSmlIlmVms4HKYkGfKpFhU4CglRUUgumoYbzNYBUjA2AQIjOIl3yErQkJman7pmc4Ag8JHTqsBMDFvgrM9g0Fd/CbdgZLGwLsVOvlp91GytcFi/QLN9td92JsB5Zio36Z1S6ReEL4CNd+K+CJ/6aEJnNhRhtJfOB5Xr1BhPMmLocZEO2dj5KLSX2GrsUrigZTB2Kgp6z3/LLztkk+V22HtQGO4PSq0TKpC9qfTVAqXa8hgLyxof4IuuLax+cx7KPc500itqCr6ku1Gjv8UWUoJGd3eWLUTSBshUDIqqs7jXRWrETFxV0oUQghTVadhPvUkZNtqrfrNxi0BSzzumIfEhCHLdUyRp7AknmJA9yhiH+swN090wtnG2iglFsgV3d2dE4MAFCQm2GNCaVWiIKLK4oiiu6yRlBQ5CE7O4JZTvMu96/punRZL1zoq2eXCOZUof30czbH+8mmWoiRxE1NJNf9SyUgin1wkqY+eTYEntjS+yNLRQb1RzujC0i7PfSsJQu78LLNLZUHu7NLUrD4qzQSPvcokdS62MVRJfePzG1WJ9aQKC0c3AnbiWyulC/txEpWJMIwgQLq0lOhFZ0SORWUolLRKxuySHVUkONYgl5W8FzNk/J6Vyl1PTOZGefCW2FuPQXClBUdaj/q3MVOetTvVX8sze8OkNwEqRJW6WOkRIuWDque1jHbTE2Pg2Jk2L8HOaJxcadBEupESx9kY2U8I4eWMWc4suip9Ulai2MoJVFNjl/aQ15j75aqlqTPqAeLjGY6jf1xGpsUnW8xlnJ0NojKaGHWOHF3kNV0Ude9WqVi4Quk8Iq6LFSOk1bPczm6lAkvTVMhUYxKpdsXq7epuk5lTrseVnJIcU45SXzLvTNaiVsyCbBSkIBmZOckwwiQ8kQ'
        'rHoEybG6iAiq9eEuo5IOCBFdMhDf0M4K2iAzgPkg2lnWhOaQGmAriYsuaOdK4zILm/Fl1rCHUdLRhpxktqkZ3v0TNXDs9fN8EEjzY/pP3tN/4r7+w22wfSAVUYwP+rk90M+9I6nJB+5jeilASmJtZBWWcoFErXVE+sGEpqEJouOy134ryzeK/1CsluZjajeUFkTkCEUpabIkKFKzPv+R63QpXCpH9MnFuh4hKsxcSal41FaK5RRwFKUve6KeWgzdyGmS0dC4pfGQKE5yTqJSBDzUG3yPfGBtpTFJadZIVcmvYqD8KYHY9uQl7u7U7lGKhhGvnLBX7DOq94ncm74hOSFK0g0q3NrbjP250ovTMKF6o9JkH9rvVOl4tbTpMFumdii6QEXyA6rltTuenb2gn6jdinIV4Z5aVUgFM21F18AnCzVn1ZgRGshq1iB0mRQiqk7qLUaFz7SDzUGZS6b9DSsh39TNQju/XizSJFaLHkZ2WZ/NlBuYeo2BeaitEcsij6A+NfOJThgZBWdCoXd2NZL8gaAJ/tP3pVUsVQFaDPDLYpEke3qDwiVJsuOqZpwEQtxJE5R4kFGtd+WhFT8jq9TU68ocwps6Pedezf8cZCoRuMHstSmRcoMzOPuYCJO2TTRmE/E127uEVjb51IYDNkZ5vQnkONClXg+ReEpRKErcELthvkCj0k48CkIC3rQ9qCZGvEufNnxqy0aTZ5bWKcpD8a/PqdYpU22sRJEzQXgxrlI3U5SZjnMi5Y6PiTyJgFmXq6hXp7hfH5Y/+I3TtUxmFinvNOdqFtf+mIvwlSioAUvioStbrTOHELF1zX5si953HVlt1dDom9eBNFVbVbHatV4s0mYBiUbHgAILSqCiEVKa68Wi3poUmg4UWZyacxw7ieZuRS67KKhNysicRPf6t1qSyEmXRydWXaOANq2Fr8ycopjiRzKqxL5Jy+Q7LoqDlQrHOatkYbFOUwMgDkRt5aU26LhwWjalcfF0vcdERGLeSWllYhNeuwArY0vnYkXiyozaxGM36MIhi5iMm9vbyaGOLSyKoLD2vTZgiwkGGUSvvkri0Dym/WC9ZCRBsDYs+dZP0ZPaANLMbxp2uP0nfahe1QqTiEy5s28j6EU2FQnBnPO74Msz+q2ssMSsHOtPFv7WnmTNK/JBEnaf1UzTlwAiSrxIK3ARqqq0amuWYxQZogim3hVLSXtS4FUqhIXa8ZJWYqK5NbRYST98ZY25VyngulFtDlZpepZizyalL0kaMpFV7UIIIjVIskvUvsM0skrv7PpIvZTAJlN7+SUXuvhn1rtlzq0rzjvazS9aSR5QlSIRJ/X6Cud4GSKDokirls+9LjDlY+0CpIinq4eUHpGll9eLgkJjiSKmZBO2qG+5XlLQdobCnVZrFuIlXqfdSwgvGauyzq3eCPT+ekXq1axOC7k1T12k9OMSScVXfHoJg3Oc1WtM9YJCp/y2t8U70V0XvhDOyDX04HopkluDhGlWX9bbklV+9fZFt8FLT6s9AVGHZNZo76M2K7I+iUvw0RH1LmJWhfY+0Ft81ouvLgItKZorl7WXQkpdL0Zo0kxFYsocr90Mq9cFC+WtWiA3LqQQIWBcwKgtC2nMekm3lJXUKyiuISr7Pgtn//6uAdGx7FTc1fvrnU6mG4ekjPZW9Mvd7dnr3fnJS/3vH17f3r67+enHH693N7c/tDf0375488PV9aujP/8PuQKo41RCAAA='
    ),
    'PCNA': (
        'H4sIAAAAAAAC/9WcW29cV46F/0rg50y3riWp3whiwLfGYGa/NQLDsZXEGMf2WGpggkb/9zmb61v7VNmOOzGMSVJRKqpT58LNzcviIpV/PHn46cdv37x68pevnvxH/jWefP3Vk+/vX98/ffliHvr3v/5Xnc3X+eXF4eowv33+w7s3P87vLs7mx4fHZ+8et4/X5+dXd9eX25H71y/688XN4fKsz3j3rA/92/n26dnDw/2P3776ad6h/jN/uLydd/n25ZvHn97ez4Nv3715vH/5+unzNy9evv5+fjmvf3j+7uXbx12q0VJd3tydX12+d84n73X/v29eP2xf/e1vS+Dtl+vLw8U3X381j11fXd3p2PWNj91c3Zz1sZvD4Ypjt4dzHbs93Nz42M1B197eHg4cu7u+ue1jd3fXt998cyrqh8o7+tJ67CtPvuoF/OPJzyrjkxp4+fD0+bPXb16/fP5s7vrju7/f/w708s+vv/rogg5nv3ZB3z179fBbr+icay8O12d9bfvCx1d5e3N9uD586VVeH36Hq7z5I+/l9uttn3dxdnZx/olV3v5Wq9z86PNWua3o/OITK7r7w1nnJ1Z0d3l2dXX921vizfnhc6Q/3J5dHf6AfnR7cfHLo0Wv8uYP50e/Liaeb3Hk4uL9VW7Lebjffp7+eP/i5bPH+xdPX9w/f/bTr1ju1RfY1LtfuNy7208s7faLL+2L2OvFL1za5c8s7fb64nD3xZd2+PxA8nk7tC3j5uyLL+P2C0eU689d2vn/a/T4cvnp/Oxuu+/178d1foHI55eXh7vbP1xS8opmUfbdq2ev/3sTYotcs2Z9d//9yzevj+uzS9VnfLFq3Ftq3Pv/efr21d/nMp7EfFXFGDG29/kho8Z89VdDv4/UVzkPbSfOk7af7bV92i6N3E7IeVH1bebp89LtY8zbpW6us+ZP3z5Tt5wn89puvV24vW+X5HZizrtUn1mpL1rA7VMLvD1zfjFP3Y7MB2yn9vto4frxc3nZQsxvot/nczfh+qbVQoXuOU+az+uT57134cZ8hnQzL2316ClccSLqXMF2jmSxclrmrFYEUvSSWoohKaYMLcx2oqUI7cZcxtRXRm9A/2f0Q4YWE9qC7WbeumhdSgGRfWrvXGpfg+96J1u47ZSWcfRN+gZ9ZJ4819KbNB9arY+Wvzdf+15SzGiJekv6ePYzttXMz73Oublz24IH9nOGVhK9ZYkuZaR9Xu/vlEkSYbHZqpCSs/q5Lbc2fV5bMjk2UXvf+xdLnfORWkmLIaOaSmzRW15Zf7XF9j16w2Vi0fLqrlO2foIMvh+uR7X+ZLhzIdhKyr+kDKlMDlLtndjpstboLRwtcu9n71nJ6FvK0KrlFvPwfGRZiMH62yb1pKHL5mIwspBLtdlG33XadrKxqH6uSAoIgkT1Y6WFVv806H5WyDdi2B2qLEGyXOk3ZUM2Jd2r7zJkORa01TzNywY43WZwm6mK8tmF2Y8+WR41dAUBT7vb2zwUAGPgGTKbtsB2x7bOzLG7RDoSVdi4pBGFn/bzIcfnZrIx+WNJwLmdin6y1P6mZLrz32SpkYRb1DDPZ8m9LXMBLUThCSmVyg9LAnVswTU6YCimsru91lKgUKBS3G1nJ4xFG9tUQCthrmYeiBPZcOTo0zDukC2t+DGIWtknRf+zzEuuLc+X3bWFYz5S+JAbK+Zor9sxnM7Y/yzU03tMUJwX6xaBqfcK+m6hoyGfC2lHFlvSDzY/ZESkMgWDFn6POsS7lkbe3l+0XXYakWBt1kWClL+ngh2XEirbteXXSnZEXnJC8iNdztWuiDZf5c3hJnKl6FiUJAfUkh1XhoINCafjUTg7kP+xX3YUtXdkUhaX0yp7BalbmmnrTTl5yPyk35LeCdIj+OykoETZ+Z9Ft5jT2fp7HDLwo5av9SwxFBeLIF4E2DbsziSFDoukU0CKUlCpFRCL2KFF2MhrJS1ZQhSi4mrCVvIOSRngL/TXu7oiJJnYt2vbJXZ0PMmjWJ4D4EaoSCJ+KVchvMy7cDYlXTJqx8j0dicxM2TCste+MRd3CFyhKIbW2gbTfkoCVfDGb2w7SvHt/oVH9SX6l3gYsWxYoWtHg+XIhUnVB2gQB0nCZCliKSQLDYYy6J5fj3BYggb7qUdoUEHKQLAWEGzFNWhQbBhKRMOhGkdUXHXSEa4E9RZXTrllkCUI1xkn0Qs4TBEM41xpVQYwapl6SXs6LbV8TAYQK1cXuiHm9jp609siRlgzA8gq5w6Fg5QZYgqkO1RQBLJSvBWSkrlWDecgoQj5WvBcLCqEmIV1EoNrMKKUndqslkIYlvIitRXVzxEgTcHtCAOZJJWXYtAQAjLUG0JRykeync6mOKBlllkk2NexZyR4qgb1EfYjDyRZOn8YVfX+KKfnQiHSLklRv6YArsy7bTJLyJf8JgCEj3aZ0nhOxo4RgLeCLNUeo+eVY7p2nVCgbe8UmKrW2rd6w6Vb0KS20UBLsFoiF3ZS5F9BK0Aj8glotBEPFxRKGwT8zL12UQ2byme4WmozhoQdlKAUqdpomUvWykMSAr3kgp6hxapmJS+QJ4OIyiboR5BQ37cNY+LaGwCdtItNdfxocNgJAltnec4jRfAGIUm1NkPMxQmVcNEY2GAKzxxIxZmExFp6ydzTNA9VxNR2LpiO2oGfRkQggADLCGnXMGBnH4lWygTaBsF1XQNuEI0QApdJ8ZLegRJYc+FBvpXBqDrtJDEWlBOOUirZLLVO8EAjAJIzAGulfCrTrl2zQ5Fz40InAJR02lAJrr1MUImsvdBvdaE0MNyWRLG5gqSoipLsSNoeYdsZQJRxIqeQyQk0qbHM2CYixNxBspUNy6D4qJtSnCj7GFArsar+TCsbvmBgO8MUQ1vmUEoT9BvpClXJZpl1xaD0COpxRd3ARMoOLOieynRd6Mv1B8W/4q5SnA1IKbxPajRQ1EjZe6MUS2gxyTE6oyT+5VKuXEJFrXpNYo4j74k2EBJ5kfIpj6u/ZENhz5SlgOYQAilI6RpLOKeThK0E0y2RIXa7adaSdP6mys+JZezuRHkgHs3pr6MC3s1GtWRhogFzBhlqNwtME4uJAYI2X9DaFvqe+sTbjm1cNlTL29JW3MC7Dy9VhKvGJI307zV2eAsNIqRZYimJzjJbLXjAqM2llAmXXFaWC/Jly3qMqyrLUbHDXtsIpV9/SwndJVYvP8Fdi/vSW6edItjbzfQA5QICTayvKXnEKYbum5h2a6sjse7YzyrehZn4lKp0VCzVcPRWTBuEwE6KJZzTTtYxLYuaScFR9K+hnfy5ZIEyv9DWuOwVGauCalC2CyTmSnqiJCQVpBv1Lal9EKkFltlWVQMloK8s206TqzQIe+WieaWkIAUIX2uXxCBCkgaEsaqT1mv78JBltx6AQUkOUtoru3U/UrA7xLnIVyVOQQnVMpVBSe7vhx4kPJVgaBjP1lnvVUF7qEgWQpZvawNybX0/OCVsg7YMmJTwAkUAtMHIwTg2fCMnY5htKiPBYpGBOHJD7hJIrKIyd32e4hRF8ppVEZl/TNLnwKSGOSMUADk69neQh+7a9am31wBtFMly1Rzp0gTwX7lY0UGVIhzZhpmSG4dQRBpGQQVcbWAL2SHlN7sCSicDo0lK6kb9gM72ggz6Kqad1i/ALmWTodpkvYxIdKuOQ2ORZ1LE2NEdPLlLEkV2U4aqJ0QMwQHy7M7qCR9ZsGRpAm3s+cM1PHxgumgtIyCqpiqSb7hPIMzs3VE2kReqiJDQ4dbUestVKx31kXJnDsZxHynoI5XY2zQD+R5zkLGYg3IdftJHKmrAE+ZABkgaOWEOKPP2H15mEvKDllIdt5QEktVSErFFxWuxFEnHxxpLqXjtWIIfuV2TLrfSEXZviqh6FAyvo26SwCy2UWOvcUP9HvErFeb4F2ea2A5UCETcTrxrqYr2BdEFlWwaBwgLdQ95XDv2GxTGgLxcjYZhQrIVNkwEf/TVkRL6FtipqCmr'
        'gKEKNkz2IUpBqnMAlOeLqUuwrE0I7w7xiwnzo85PuAYHZg1TwwLNubYt3AZJW1UIL0e6dBSQh1fY7W51OhQTy91XvPWUb9u7r1p4rrLAfddYbZ9Ox0WzJMd+CgXiSetV4AXKJk5br3sfulwKCSJ0emhsoi1129WRBDYwaDOYPHIuo3fZIu4NYEq0GvYQNCbXyDDch10rl9K7zXzwvr5QxXDc3o7xcQWLHE7TebSOjgjNcliCYqJj7bAkVl+g8Xh1nXWG+fL3wtJHwtAiNHdwuMLQcBjKsLhFGBpVP9vf3pUs1CZsQJG1EwvEklqUDbXr2LtE7hfTG9lLcXhLgZtBlQ1Z6limJOl+VjIlUDkcpMpkeNISVrsBLn2ACVR5HSEGrTfMfe7RZrGM8vrCd2i5udJ1R8On48nq4B4x+sPwLeG40smZxCnQAZde9V6OFB6rsRiWoNNNn2/Q/3fFZE5L+DuQQJThan6M1SwAufF8Oqymty2l7jJqR8UmGwloFaulFysQxOqhCd+WmVZyShqDFLnEDXZjOZrhNM4IAklXaqE54YNGn7EQ0tjb9FIETDON4Q6IRd6jpFPWhG0COSl61CISYH6ESUFqR7wbJQYeLRY0YMdp0KTnEoY6Svt4gmImfGjqAKQMzJMok4hVENKBV0hykQd6phYmvo+lQBpM4eUHmZ9aYRim0AZavXG8wBMd4grVHYGATwZaiuYY6DA8SCEMvrdZRI4SQySeqKMAKQrdqKpheggCzJRbUPCXdQ/mL/sUmi+AuPmLRULAZjb4K9g083C504USQ3WP+HaLIQ5SbNMSI2kfakpirNqL9ubaO30hwEQHy6T73sZcXUzS9tgpb7+sMXoUGtWgvSi84IblyDVDAr/qZnjQzQ0DdrF2UFI0EOgLQVY4jK3mKf0cwgbjN4Zhns1YAx7O2Ey12TJK7c6hURJZVeCECWtZtOJN2EOJ0SI1zZl4JyqHZiPmqT4HN4pO1CQAEwMFT6wJpREmqUIxT0oEM8YaPUlTpjszr6Q7XA3QDA9YDSgx0+C0jsaaJMClxmIAxprCM+cYhjeKGCQY2pJuO6/xHRO2vYIKdzlrh9GKvZK8DGdtp7QZKtegjYdvTKrbWet9Z81jZ+VksT/wXHLW3U4XtU+3qL10fMJNjf5hlUQ1edYgiGBH0SI+4qYWsE7ctERzupYap5R+wF1RBAV9xKNX7v/JfeAg19ABpFztqNmzZoXRMj/HCA6gFoLYihFXo/ommN6iQ0HqISiONZ1HxkIha6jD3Ww6B6LJZMeKQWQAhWQhOwEp+qiyOI4Xnfyd3xnujFF+FeOttZiEo04WFaQavAbzSuJJ9yKhn5I5mbIfM6yz8hamtmbKSBSDIojF/HyuCcwXykwkp6/O1ZlSpJN5AP2hv9iUAcmZOxBmZouwT3+kTidm4siKaeWK/Nkd1hMjsSDyKuyTkTSReUJnTHjGGkhkwMAd5dUuU7KMj6nmyLPFr9beRhgr/zkNWzVjb4Z5fuLIv8MTz+XeLrNY+4SNzcYZiBbaPvThSZwsDywSBIXTE0QJi7eXl2MFuF4VCV3WD9Ie4Ff6++V0bnI51rSQSquiHi0HCw9junG9+G4nQGhWOtGKlQzuBZ3ZAeofHgbNlUQWT86eUEJC2C9zG3s7kBDmkQXCwDCnl8EwGfMGTrACbNhAMkPpdl8yyyDbBUXHcnhPTkKGi4mw++XHbSxWcO4m2/vZ4wOop0nvGP8a6pHxj6CeGPU0ZpfGXQl/iOVqOEnQk3u/72sspyiQROujvJAY68nn3NUUMNPL0iHSVWt6TED1VRkhhaTizwLcj6Mqx/lVFgEzmE33oEW5PaueEFPufis4P8h/M3/Mkoox5o8NZOcsQRHnqMmVNpaejhF4VQ9nrJZx13h0UUwlcad9AmLU8OQh+aa8mzRDGZRttkatXcyToUoT/s3R0o0WO5FMsqq0h5ZJBWlZV2G5q0WZVARD/WLFAbpgvVTqXp27v6mjY7ZYKmJAXdwQ8E+jUaUebK2emnpTmozrhNGPoBmXwzbs0dyk81NuYMrRVCGonWIGyCynipNahCityzgaTmVovsoDsiaJgVRu2uXSCP633tSxAgRz3lB7zA8rr6iv6YkBPa2lznJnDvHYmb1XDuLAmwubcjd8VWb9tIJRbktw262MZJN+bqghq9a+Y0XSAWxjUFErOjXNoYt/SA9jqgcmFaspXWsPgh63jmjviv7e8KrRVvoxtW/xau6FOD17MX09N6IrXafYd9yrLfc1VzvD21xeaOhOtCOhevmTGUYk3Y8rWnpyWIEzx3i6Ax03M9yREsPKUKAT1k6St6ILzglc5NmD4TCx+NkoF3VrxhD2U9FVZrUGFYTPFHPdqSnCOY0UBp7lJ8OGLYKl3KahIb+SOwVdMUENzbR2QLFoUNvUaotB/eS+YEYsmSwDCguXwI66fTAoPnWEEnxA2TLR4nLbyGRnljTcJWKXaYoKjleszjyE7NGUN8xXmeWi5CHRFQmT2ubkGLOPCGtHpck2gr+eGcyBJH+LMVarJhhSMbc8PEgTO/G71wfwVXIS5glZPY08WIAxao1LeFqNUWZdJ9MqS5KejpcbDf9N0mp8D/d+6N4AQhvwZNg/3PRhF2lPBXwsxIlIVM+cBe2R4ZkKd2yWhYTpHmx1IelkOgwkLNgzbIj7PCBUDWOjjrmDycpFL1FVJFWXMrMRHxNkx6za8Z8GnA7g8acB7035R568r+Ews+LblfMPSL+7f3z+w/2Lp9/xfyD64fHx7cNf/vznd/cPj3+afx7747ev/vTm3fdP/vl/PepvANdIAAA='
    ),
    'HMGN1': (
        'H4sIAAAAAAAC/91cXVMbVxL9KymevYlmRtLM5O1W71bvy+Zhc99SLgqDbFNrgxdI1bpS+e8795zTV4IgGdsSYBMM0nzc6c/Tp/uK/HF0/fH9q8t3Rz//cPTPf/kvzdGLH47erC5Wx+dn5dg/fvnVZ+WrnS0WA86evr26fF/OtXh7fXNydTO97cZu3rb9OB1aXZzpwLhcdLjm6gTH/tZM706ur1fvX737WNbwf9vbbijrvDq/vPn4YVUOfri6vFmdXxyfXp6dX7wpJ8v916dX5x9u1nJlyNUNs34+3rlm51qr/11eXE+nfvsNInfN4gVknV6ML1/8gKOLZr7k0UXbz+Po0I4tjw5dV68dJrV1dLqvHl3Mte6w6Pt6dJx1PDrOhuHly9ti3zIl5No8u2HV6d5b56DOH0d3TNMOQzef37HtxaT8avo+fr86Oz+5WZ0dn61OTz6Wq86vj09PpvPnpyclHl6fvLtefZW9FkM309FxMRzCisM4Ni9f/vnih78ov46LncFwR+ebq99XzyxE7lNuPiz7eXNHuavVzcn5xeTQ84ubq8uLvXl0H4qM4xZFuqYbZ88uRPuu4BhezfuDBO7YzGdbTDK27TB7JN/eb4p+UvQ+Uwzd/kwxtmMjU0wgdb8plot22bZ7j45u+MroaJr5Qw3Vj43W7cdFexiUaJptybXs2r3j/zIM9WnzHSp2HgKRwP/hc/F/axZ1X1kAPqnHotvixGbs+mHfTuyH4UuceCAsnG3Fwumivt3pxGP7+6/HF5c3kwVeF4R8uA2G2exJiMywbLdp2zbj4bSdP9jj7bw/cCFYtt2OtO2fTdp+TkGbLxUjpenZ5uHFYjE7kIcnlrU/1vP1cD322/jN5OVDRfnYD18X5Xe0vV+DeTebL/cMyd3sq913+LTdVru28Lf5cj77S+36Uio7JXO/t2z+XEXGphm6w8Rs3zyV25bbMnSiqsPyQNp23eH58GKbXk27PBD69vPQYA96LdutfKidDLinhOrH8ROG3CrGcuos9jV+GKY2b5+5OjZTYLffIoO4xYjHz+EV0Ln7jnSe4iuGA+MuP8+/I503BiLDLj8vvic/t1Xnnfm8fI46zxcP17lf7NDuKfqdL+UMuyJzeOaR+VA9xmejx4bE27qRqT604/itY8KaM23i4DjbqvPUL31HOm/gYL+8V+dm+jeMzbPYKnkWY+9msdhhqP3vGjT9c5qu3G+SLVM1mqR7FibZFjuH2mbbaZJ90ce2X89u+kemUtu1W3wjDn+MPbJ+F1gsn4ehDjbo6dsdyvdPqPzTjSy7bYnTjP1y/4kT5f0BJjl8MkxwtUP5/gkpxtORibbdYZLhebCuRzZJsytKxv2nSHcIfvUosdNsN1TfPCGWfOk+0aG42Gyxw1AHYOzD/urQoT75MHY7TDJ/JrhzKOWX8x3K778Iz5aHSJxHIvTFJI/NUw+49bjN88txfujBTjceuCvrthaEot1TjnC2qRlI+ZmfYO12qNk+JY2c10Q93FxhG2GcgvtbH01OTeH83q24xQ6dm29d5/nwOePYZqIX3/42Mz49W/4K5fW7k4v/TAKVmcCLspn/5vzy4tYfpDT82x6dqX+MspjpT3xW/z3+8O73otRRxpe7J0vJp5eWbHpffpkld5vOpZTdyg83z9M7m85OJ3Iqv6a30/vpdbmjLFSWwPHpRfl24/XTz+nN9IyERT1xAS8XGG9PiU+dlpseUqTJCfLhrum/6VqXkJbK8tO7svp0OZ6e+Cznq8SF83QPnl6WLY8riqyfXmSSANSgXFfkzsUSfDzXnQ5M2hcVykIJSzgkLNbIRaMiSlEtFdHKD9xfHp2hXCqnkoeJrN5RVi/3mU6VZ2Qq5+UZxrvhhkRhM1xSFsfzsDLFht0NdivH6NoEjxncQu2TjALzZhwqa5anwEqyEC6HVdzp2HJU+jAIdG+RFL/KLTBqMW6xQbEz9DJIVpTIiBQrC04aweHwEFSE4QwKQJBcwqLYIsu5RUSdgiRlBdm/qMgwTvqipRHJDlMWWyGu118lvIq6tLkhABiaSoISOYjDEoQIEMRuOakkYLDhDiUBQiIzqmoS2D1JwNC7HYZ5Qz4EbCSCh6SRCEVSl6TQzqEdHl7snjKtI5+5Xmfe64rCxGRDEMBxSQlYnuQIaxjAYClcidhXhCEfYX1kBXKFwVa+nM9CSggamFmMdPrX4SZTRFFkOM34OjOPXBGmsIZQwJryYKQErKugRPYbL8umsE/ELqPwuJA+U+ZjPVzt+MG4lKzJGbDIWyJfQmTxkQoVHIUHHMeSywiMQGcsM8y1pG5IshlkRY4jq6CtvBji0hOJCA1xHFmdlLNydBJCCSeIUHwEwM2MeOpCe1gZayXGa1L6RQinrGxDahqNSGUQLZlortAjeNGLruy0vJGgDO+0kbRMDqAH08RYQOpdQgiItGGxSJWkiEbo6g6sAgmTwhZG9Zr6SZCVZVggL34RZmABVB0KAixD4iJtTJEJPxAWiG1KRqQUvS3HKGKz1VpHX8EvWegbJnHaG/CeKRKzm9U68x+qYl6bQ++QAbAhfiPQEwoGIgcVyqiyQAFyIBphYwU9DY8aiNPMHHjOTEGDNYigtBORyoH8KAweUVELJOsMEJ84KDQiGmbBFA6gdiRp5jUfnFFsLmbA/MhEiiy5jaBVzgAinLoyW4t/WNGdkFpiQsZmGSYOIRkYdQWFAVzlEi6WwrVZuQm24QRCgK9qcl6jigWRAMJSwTA1AaPWUfgBHkTkwWbGMu68KygSnUv4AXAqiKEly3ZkpxN+5BHibhABJBfyhAaFTyMqWThwAcM6SWBmTJZhs1b0yCayJ5ZTFkAju0C+W44qCHmJHYoCXlt+sgymLOpH3IcnsrTxWkgNZYCHzQQdFrgQUEW4ICkk/iCeoiJwRVZyZTqcCzidzpdYoXOAKc4IIjSLZUJelfnsyhrc5056ByOwclqQ27tfHuZjAdx8UTE1k1IhU/BGhIo55rgA5udpq5xWeJkrUSKl4htehkvEGQKORW0tK0UyazIxGs+0rOVE0kJKpL67hHPWXle59mgBKJ7YprgOsuIWMVOBD4EYnVYLs2Dbcw6OTA6cKoZnkaWwdLJQw0QHqLBJ9pTz7ThDzSRzl0V1GWof6qSL/bBOZeZLEqQn0nZlmG9EqIkTiKRlQgnqS/ZAZ/JuvUMlAfF2eVw3qxyG9xh6WdQwmEH8QLVM6jDgINYytR01tDxKehbdEwGTltksmgu6J4v3GwsZIU1EHX2LCXFTxeBodKA2ux3KZOsk5k3iyvQwI4kFlCBFXNryH1gMuTwisBxwtnEs1AUE2OxEhWMisSHN7Cez+k6vHSAsxFc6gJPCNhohs0eBWuqTFEReXZaV7SyJibwozK1Fo3nl81iAkpr8pKIX7RRJIz2CYFE75Go1svAYfIwjAJEIsm4mRRbTwdVZbHOzf8vKXjAMJUS2WiRq6SP0s86qLRPuoNk1Ck425pGLTo4LCdS4UB0xQrXcFpkDEVxIAdByMl2TkQOjC6xDOPR3WUw0EZ5kFFJv6gM7eFaAmguiyhSiMARjoWcNwKKm5ka1XjMBi/gW7ggSYHiyLpJUBTPTUrkMHsfsUVHM8HASV6XwrGh0O38kEu6kwhgNX85BapSeTG1nk+6MTAnAbsmDaSCNVBUsnMv43Sgf6g1zCoivdTnXGQbtH5wVTA3xFj2nmkiyXWamZhP8zrW+Iz1Zy1HqXYCnqUKxmZ4B4wqhiXQO8pFVGJLgFVGjzhISrikHWIrqo7FHg9nE3BSjanBUPVyjHalPOwRCMzVVRKNUB7/ygEu1REosceDoYPkqB9CpCtDtHBuBZnICQtrKpi3qo2YEeEtQr71ePF/trOiIIEzjJcKkzJqC4RlHRKgGSR08I0rkMr6isAuRk1zNhtCZJutxBgmrCC7jT51s8gB8JxA4ey/JlYK0G4dXLjBO4PmJ40p2JWjiaC1xDJJ6tfoWmMBRn7pwdgLGqWa0'
        'tEFdLCanAeIxpKV1YxwJpuziJuII7NqZlpz4xXgoq/khouOxVmeuzCkKQv9Ec2bV+JqqET+SkB7yexAFVX+vqRwMho1qqoGBmYiA1pVo0cvdX5YrL/P1pDFmnsZJogU9yupqslJfDBJWWMcOSx9DnIkQTWDWVIfzZQ41iI6V8rrChOim7GDwsUSyOMeUS2gaHI44z6BJMQDluGKtg8Wk3SJC9Ow6EYgxUbBakCOLB8sI6lqzxkcA0pj+5qhaMaMI9m3SW2PW6HgJcRo1BD67XAutOVqvA9mN4arma5wDMRo1kmfi5mjCkoYUznG7ulCXRVnPKs5KH80gyUSSVRIUkz2ND1h2xYvVw1iqg0DCiQWS5xh/6Bb0+MC9IJEepZtjJMWyznllQYxRV4argVHPwi0NDc8Yf7WYZCawWYxv1L6qC9R0IXOozOZVyRsBvZ7zZU2bCCoxlcbuSAxJODyMqW+0nr7uLkPjmOiKbXCAQuqNXsIJB7Xv1PCRiBR9RzR2GH8pEixrD4DzM35zKMSmU3gek2xXJyCGGQyYAax5XkCQKLJFGuRaOljRTcWMiGGaKpEYmlvtuJELYBE0P85yGmaMSme7wBmBQMEZAoHodJt6WdOI3zXlMXFpE6Vy4bMFHeSEAQpaUAxzjX7AO1AZ1B5oj4ZEulxv3MBgknvEjFdEcVe35CampczmnMY0oYmpkdXpIvoiGYXTDQ16jVNTx8OpPvIlK7c4g9VPRRmnqaBcsDl3VhIUcz3CTA4gTqB+G+9AVedoLVKEd5grj0UEhbFW7wQouJI0WTyJpuYyCRs/nA2FIBpiMi1iq4dysolAYUC2JfkITMlCI8mOV66XtDV4I7dfczVJyCIrb7izcJQwN5zkjEksil/5E2e0vmJFA0TTBI0hQ6ak2DdVk2pkOpesghsEfKImfPQXqLbXVzzONXJ8y+557R4GNz0XdrcQeo2SeJMIiYxVbcFRK+NExDVSpPVJbzmjA5WMJTmidg2ZN77kq3iohgOc3FWnmozAEWSYQXu39JRl0zY1kjimmIo1ASq3v40Ig3qSNQflXpZyUkaqZSNtbMazmcpeM5E5TYbDOVwYViqz+eHGvtdeiwNvVg02+V73UjQQ9WBUrB+xiRdRwv0Bj/1fbekx0utY1AQ62YNUyVHae2X8Vux08xCEe1UwDWlfFpFPgqPNf3mjgdIWMNtxlqY7W8AeW8BJSBh7KCw5AXDUwVPsCq9dfHsLmNnhatxz7DqrZ9auXXRl5NCx95u1q6GuUfjDLUBmaWxD29rsIUOqn4YAN9Xujag/40WNn5lG2bI05+K0lOoVN1ld+9Nirbxf79RhkHa7RiEyUoC6SI42Xrxuf2tLE0U4xpFR9Un42IrGQY2GPNWPPzinpNpMcA+O6+qqIm442nSvc02N6zKxd2NPXrMQfVgE1k8q9mndL6XYF+RWtXoP7sqQP3ES4Kl+BsBi08ZVMhKLH2RVCCSlisLDgotlRm/dKdZGhoq7JFA7mDzX/UnxnMSEdH5liyGvNsxSnRVq+L7eXlI/HHPrFCoQ4F2fnRGlMj05WwrzqyOsfY3IdPlc1uvVzenb1dnxa/2/jN/e3Hy4/vmnn65W1zc/ls8pvn/17sfLqzdHf/4f8QqG9yJZAAA='
    ),
    'BPIFA1': (
        'H4sIAAAAAAAC/+WbUW9bxxGF/4qhZyMhRVIi8zYdtIO+FEW7b4EhyBJtC5UlV2KAGkH+e+/Od2YvJdgx/BI4iR1Z5L2XuzuzM+ecmWV+Pnn8+P71/e3JDy9O/vLPv//NlicvX5y83d/tL26u+8W//uPfseh/lrvtcrvqd6/ePdy/7/dOF/3t4+Hy4TC9Xa1OV5vd9ny6tL+75sJ6tVou85mHy7zW31w+Pu7fv7792IeIf/m71bYP8/rm/vDxw75f/PBwf9jf3F1c3V/f3L3tN/vHH68ebj4c5mW1XNZqsz7dnT975lfH2v/v/u5xuvXjj6x4d/Yy13622CxevXyRV8/PdudcPd+eL+vqdrHZcHV7ulzX1d12seLqbresEdaL0xVX14vVdlNXl9Nrri7XZ7u6errYsIb16XI5X11vz3V1c1ZrWK+WC11drRZnr149NfzpXnTLju8eb8vi6b10yM8nz517Pnn39Iu7c/N4cXV5d393c3XZA+nN5e3j/hOOljm/G0efroajd69e/fLyxSf9s1z9Wf1zdvpy5Phn/DOS82v8c3j4af+Hy9NPuWd7tlpv1t9I+Pw2nvhMIu02m812+8f3xJdTJj2x+7Yh5TeJieVit97tnntiMvlxP/138X5/fXN52F9fXO+vLj9+hUu+MRjZfAULnX3OUZvp15/eUXNufc5Ry9VqctWfla6fZVxXjm9uL+/+01X55J3p7cP+7c393RMRuUbQ684sINfS9fv/Xny4/am75cSbNwv3FhEe5hYWzaZ/pyvTLbf8057+Y60/0oKX7egZm4YxD7e8OT3S8tc0zjRy9En6dYad3vl01Xmu35semR70/nife/o13cl/pmlcH+4TeB++9fFMy3TvY+Vg3teuEfrDNr1v3cS+iBzUGXoas/+ZDJhc0A2INL6P09fT+nq6fbycZvIcJg3vM6YFhhV9Qktr+iDeV+u58r4Ixz7PURmQF97tavlgLii40FfMx7oZLkv6UC2Xnwbl4i1NbLM9PM5m5PrTpvRqvyevdvt8mJibnovufvX0eXojImfuY2GXpbvCM15yB43txv19M9JnbFofLNfcX6QjGcTS27lWw4BQnPXAinKiabvTk8YOEJg9lHIlittGsEZ6rF/sbjJe94BoGVQZYB540nPuvNqfCtzSDXUjZvu9zIXcA5eT2T9m7IHjbGb/WPqsR9PIHc/1EKmWU6WF/YN9jRjpRHO3jA9aPuvpJFKQiYzl9ZUYy+9LC9zgsqmPPO++swWEY/987nimYZj+sq2KA6Iur2Qc5kfybr6JnJ0gjcxfV6QSGdOaSImKSQWVkzYRyub0Rzq4kZLOmvIpbXbGXLow9x/PkPDB71xNa1HJ0qfis5lRuXN9CtLbMlTSPbkl06a3vNp9VeZmGmcqsZcZas20gD6rEefphkwMZ1o2q7YmB2iKehKiKXKIYcNFmiYNSbBR9GWyB6mmzcyb2tTW74OsLKD2Ki0kqUi9UGimJwGc3E+yp99zI2UTAVgPkIHbjXsYkk8LTxKvQK9+NzM/oy9yM+IYfr0uReWJCUjJZTY977kASlnF1Ao74ie3RUgiOEmchVOE5UlBQLfIJfcyAbHIhvzMTcMRiqTMsMzSNKvFgA/ZntFe/m2EOIjaFw98BiTK7ofoCVvIXcPyzGGc3Gpvk1o8mCVYBmiaa003aPeM1eaHGiAOI0GUUBXpkhhCWoGPxKvoxDvwZwZ72ZGpxSbjatgaNhZU5h02z8E4qLQDPKYDdhIK/SHYR5mN/ID04K7AstQO4ii5P0bEi8l7wCVhoyESnL0Swdjknoa5CxmHYhznXY5vmjVICgbJnIbkUq6EpgDbA6FB0oARSkxyWFBPlJpgwomPJuZkvwXGIozkymbC1QRHUr3IRKElPEZSJeonXc26hf1Ko4kdxJiyWsDCIpVCSWPolwTIACdJZ0MnZZohRUTZGX8O/cMSQmmpPRCTqIc0je0jywJ8yhCDU4I8hMZMswSpzKbBE8Rn4NJOOmisDEPwn6RLKEdYQkM4zIWLCJpA1YinTfg4X09qxdtkSmTQI2uMzDXpuUSuBA12SwD0dX/Ja8Wp6C4gaKLMisLljBCdgDDSEF4bZ0O5GfQ0OAA1bAUfJaM0aCPQW6tEgDTE6YEwAgBQcSEKaC40I6ZZa46vgQKdOaS9F35lRBmbK6VQvA5thCDVhX+JCiP4THbLOnS5AlhIJVw9eiddqAUUFaKI0NUlidxKrbdWOic9Soq6CEwudR8fR1hE01axNOxErlWKK6OalEdCajOeGDMr6LQlRskDk6o+SXxFzJnUFiJakhbgTQxFMLqKDkbzeoLAb1KmQZoCFZB2lUOGHjcqoywKwov9Mi/DBMZiXQo9eAm+LeASRfG2kcqDLbM8iSreVBGguVR3ySGmXypTlK9NVVk3EkkcRd7klVgzqFFjJAsminuIu/IgVZOoz+sHGigFIPganEYOw4HkNVbAA/DRMRllVCsJUpJJ1AqYbFTbc1FXqSb4H4xLRpREg9SUq1VdjAuupDUElGcEuNAU2lbqk7iAH5sa40dZX95V8Gra9DzJ7OhjwnK41auoc6knQkbqflRWmFdrKGCRWkDoIwepr4CpKrpdpXYVN6grlXXNx63ISgzca5SVeDbH8yFUc59za5qGsor4cnFV/Ul0DcZO/kLl5VpHGhrbRfWh+toKOMgAkLIwxqgLUNpsULNRUKscK7YRaLZZYkvMWBUOFkOB+gg7oqjxrCN30cUFlEKWoPNj6GvpLZFYqqwcJPtIihf1O5qqEpWpKqWrbUN5gueLkKLqp1b7B9Q22YM08xEIqr4rgMg+ak22MkMxalg1FOTgUUy3wp9Q8KmiIqXEMFYSSuoEOWzUzOkJzAXlVJ1QmdLCoj6RLE99lh6JskhGixmeytumjkYb6jbUUFD1YQjpX5e3JjUgeRsaYFa3/lzd5iSNavGZ0FXrDKd4qPBX2WsFUJD6kBuIdwhHaYG/04K5b9SEB6E1NfSH1E8pQMCf+lWMNoOdGntEHo2d6mVUM+oY8LJUFawX4Ek3yMdR1bRVjKjxU+lhBZgqb0e/qhLjq368kskGCIesYzGslTUdjz/ezy+/+AN4UekJ0RtVPv03m5siUS3LpGcYukI/41TKl9SmZje1fMgxkYIasUAFXQe1I7KIoqYMV/2pJkMz4UCodxqU+UWn9HrSJNUzolZVwuoskbDVyqDOkYYCyugeoP9QSXTrBAWtOhauOgvFUh2fHJFOIvpPuqqJ9IJOk2kYleuoFh6HWLwKqdycQW4NL0jDkQZNDUpR/mhVgVbV7UQLmkwqpUnlQB2e3CdIbqxg7hurD+5qN6hWwvklOkNtqKZWr/CgkKkOE9RUUZdJgovGwJDp0Xwwu4SDV/uWbpr4wEu+q7/SVMNYHRaMnmZTdyOEBSpRolXda9lWjKbDiGpWjop+7iKrmlHzRa2Zin8b2N5nUg2qtlTBtVXnsBQFnQ51E23IRpXa1X9IIC1VS2dSqCQIpF4RPIOJ8myBcUFVtYBHjWSjBZRyQrOH5NJc37vVaQYZECIwsQOconYmAgphxMkBHYEQT0vxmbrI0tfqucM6rnyx4nRrItbBh141WjF0kyqipxPz5KwbO0SgOjuQNDHljYqnagrV6QTrxahchknwyv9ePW/gp2K8iUcovsQfMy9KXBDktHNMkVgprTadYqKmGwSneELLqKHRKoOYwUKtmagRVDhggSJwAFQrrRY2a9WmXvJIe3p9dVDQqvmieBvnEiiCzDUrghxlSJ30QC8qQptHdQr0cLF/JX0VoUkxhYnEilAHQE2PSZvQo2vKi/knbWH6DNYczFSyK0VVw1b3HaKX1KkuOcaS/DrkEqwpdBTUQyC1uQ6TZ1SAgy9RNRfBz+mbSZiABdU9MIW1SowqLJuPYA/VWACbjWKEAFIUsWHmRwc59Vd9EZUswoM2H6vouabDWo2qd/iX3xKYo2tSsPJkAH2+jn5zi/UKBjn+'
        'fElgH+c8Y0QdUs0nbvJN1aheJbEq2jjuEKgL5nPWcQ6mnqGKZWrBxsLMOJD06n1bVBcgJ2kavLKVmgzBUdVBFa5qvaobqgI0ZiOqFSCaly4mXFQeE1/QUIz+jU6U1YjBBaFmcx2ZRp27U+SqsSAiKmZWu1FJojM06romvSeRhxQugIhRWoI4WUwFNW916LyOWlt1RPttne2iEFSdIWVslMUh+YYqkdtozsIEVHtmdWyoc/VRmFdcCAMqAzlISLcGsRc6hNTXCUynVhAHDK2DcQCFXkERRKIneivUJGoV+DpsqAN05alJlVatr+MgGuKK6FB21tEF36YwnXnmKrQor543/7XRg5RYzgajz/VOFcg6cpDgtzrYTX6rU2KVoToucH09Au0A6atBLoVp9X2V+o6KuqdSW6ZOoZT7UM46zSSL6uSDr7XonFVn+3J4T8+xTuEiZ44c+pJFTYVLK0fpSBXn0J0k/NjzpgYIp2VzUsNZdRKu3W2cTtHyo/1ZJKZO5dF2HeGi9L/gmrcu43iwVA5tH8F1depHc1QsAmmi8tv4TgzBf9SUkgejf0/rzf5w9W5/ffFG/zvCu8Phw+MP33//sH88fNe/+/b+9e139w9vT375PzmZkqjmMAAA'
    ),
    'CPXM1': (
        'H4sIAAAAAAAC/+1b224cxxH9FYHPir33nfVboRDUU4IgqYcAhkBQ5EoiTJEKdw1EMPzvmT7nVO/SMulYoRTJkkTtpWemu7oup05VUz+d7N6+fn5zdfLdkxP/2z//Mj15+uTk5fZ6e3p50cb+/Nd/xAR/hvHPrF09f3V787pdm03a193+7HY/fp2tN4vJajOObK/bo7NhfGa+xB23Zxj603T8drbbbV8/v3rbZoi/+6v50GZ5fnmzf/tm2wbf3N7st5fXp+c3F5fXL9vF9vzu/Pbyzf4gVUKo+TBZTZa/uOfBubb/vrnejZe+/x4CrxdPIfliPn329AnGlvM1x1bzjcaWszXHlovpqsaWmw3Hhk0f2wwzjK2mq5pvNVvw2dW4SI2NE3JstZxobD2ZrDC2nmwWNTbdcL71fDGrsc1qwNgwWQ4aG6ZL3jfMNrXusJjpvnFDNbaecY1h2FCW0UiLyWiW9mE1Wz97dleVd4zbdHV88WDnVdvO0SUo+KeTe431oIUud6fnZ9c315fnZ80t97c/br/a7Tft9vPTJ+8ofFitJ8v171X4i7Or3a9ofLX5QjU+LN7R+IhrD2h8+P9rfJhN/1g+fq/GN7Op8tJXH39X48P6oPHFY2p8/jlqfDXpGl9PPzcfX3yxPj6C2fpDancxKnf4bLT76frpdDIdlqvVLzQ5qmy3HX9OX28vLs/224vTi+352dv/XqXr2f+s0l91zvdwxMeha4vFA+pbP7r6lh8v3j9NlQ6foEo/JFw+kkqn96h0JJez5aOrdPFBVPpxVLWcP6CqR8fDzzCBfwSPXH86HvmR1XJP9U+1DJ9goL6X970/YX8kNU+GB9S8+VxSzAetMH+3Sjf3q3Q+eXSVTt5R6WqhsfV69kW1DJebeyj8dL5YzuZf2ykfqGhq7fUXV2fXP4xKmo76Hr/ebl9e3lwfd9rnPEbRhd5ln+o0Zfuv0zdXP+5wfmLuluO/iMwIi7QcXyI8cMHax/HNM9rP+HF8NXwYB9trexTfxqfaTO2nbsT1TMNcnNLT2vOYdhzLzHbD4W2UYry/zRnRZDrcKDHbo9lEMuNaFHCc07Q8BZOQ7b6A8OPuEouPX0dRIcz4M76MS+EJLNAkbx9wpX3iwhCxXW6T6BuVkG0gsNtR8Kawpso2CumaaG2u9to0hM+GmTG/td208eCOsBEIb7iIt/YYbuMC7RPvSyiiLcdHsxmUkus+itl+YF+ae7zQZMes0kHbQJm0rR4UEsaQ3fA1mwNwOuND2EKbsW0j4B9QOM1qHhp0bjkce2kXwqUjuVNbrXkKL2Nf4ygkwRJUOhY2bgYuYfAT4yrBieWMzeZ4hLukPWB9uJjTI9qQLlL6hJppQcgEfWNOfDRO0ualo7fZqGJYBPcEpaPnISKoD+0MFsC88IfAjCZl1XsTre0QNwS9P5pKsB4UgEBpSghuEBpoMxmdGLJgy81P4age2BU8l8pwCsdQT8jG4G1/TTHBePBmnmaDNpkzzoK2C0ZPc2SHhuW8eJCTCg4EGUFXg4ztckh8aJLWlXrpP+GFPklsCKM0oW3AMvSDwKS0izNOcFvbgSGyHK4JcJO9eTNEoDckQAJOIO9t4U15DIBmiQ3KGG0MlqIpIQHESqeBgCTQelIpeIVaC1bDD+9eLi97YHGYFfsAFrYIA1CnRAFgebl6m0gi8yojMRRSMjWwkNjNaBBiMwToxMBlAhK9uzsFdBVQhWIfZk5giNCeCCEEMMwrrKPqg54WADN4E4Qv4AnFXf8DUyvE4euG4OKt8HVjmAKvGRuMpUDcJ5E9mUFSqQxZiCPwLD4EbErK4LKaEU+TWcWCzwa1i4tNGj6BtQgTJiRM6ZbKh/8i2Fz+k1opiEpBk5qyFnC5jBVKwcp9VLZiP6Oc0pVahMCpJEDrKudBj8pH9BRjIOINLluSEjSVdAA9KWhUnAiiCaAOI2j3hGxAs9FzuM8AABhivJmGgCakkEMEYVC2SWW+7rxBr2/x0Mb5BFMZIuXIAEbK4uQSCPYkN0gTPrb9cComCAIKaQN9U6zIKqEENA4s86IjzUcKTlNpPPg4I4nB1RGF6JPQQwGp0xsJI+RGzd2cWiPymvU0qjyoBO+SmpYmqgFclLJNkYmM48AupXSsSxMFolz8K+TAydURAuVh8ExRAONYCLKClFJT0I5RwE4/p5sjTQWV3xYWraL1cIuRlrpQUPwVOoUCTbZK6F/kDnmrZG1rJl1QZDBhPQoSuqBv3BPjg5kKS8sRRASDw/Anqr7thZGTUWpELiHqKXzJEojgNBWJVM+FdKaKVg04gTNIvCorIf9VMmjbDxNJUbDAHj3PeelczJfhm4xEJ+I7H4RPiDkpPZlcDw4jDaUmN5Fu0juybKEC0mHCU1hkBGlTKritNKIURG5rXBuSZU9/ISJOK5mrZLH6GlqgkzizorgSHjCRSnhCabhYWiXAqnGEMPBlIRbJhvcdFAgVKVJhQCZVOOW1DvRCV06tUAonS2HVkYplphJxTWGUvMFEuYU/UVEd8j4ndgtRk1phINKAdEyXu8LuRoKFyHOTvgTO0EBGZTpko2ReToa8EofIqATxXlWyFGOQhpfVZV8r8saQp66rKAXlCq5Kcs1Ek3zCZXzCGFHIuheUd0EaRLMXz5PbcVtyVxLTziXFWkyOLVbWuQNcWkSAaVQlpJAyWVkkrYNoQy6lgujkgCxXFpR/IrostYQ4KG1HiVEBUasmkIlDijNmJUU2y9nuoWb8pzqsirraqRdwkbyk6h2XLknhq9IXkkq6rmvitzFMCmkZUYWuiB7RbmNXQJuEvhTdReNqagsl1GSaIiEhA1FpJS8U95YhGXFVd9J3mGEZoqBX1VkI2sdURbIu6ywc83q5eqjCJa4roaW8zhWGzDMpHCFukGOaGh6MLgNEU2RXGWWh5gm3RBtncSxCAMkxI0He6iEqA3xH+jTZz0xYQWKjylKWqhJKxERx6arL1YWgZ5nQrvKdyQfofNQGiw2xLVUF0lWqjCyOoqpNRSwRmenes1SSplmCjYkqZyqQoveB5AmmjES3Z9RW2lG1WXk1VBAhHKhK92J8LPNVmVT6FGSnYlQdFFHjSPEs+SGgoloOQXMJs4IcSXIeDO126LXA50U8+K76I7KXFOTNSsp5lI9YEjGDWoVlRteM9ssekAnhyf6yCLmi1Q7tP+woSTmcd6otIkypNlgweaYoFveUR/ThGG+U9wXcBerVdwzRb4APHZBxJNcqOBLn05ZEmEpIcdGqf8gczQtPiJ1qMTq7kdKMWIfiUNW9okR9kSLr7O8RjkNaUF60UOfBinlGRVk5uRpNcEXBCeFPFNrLW5S4SdxEFpUUsjOecHWY1DxlHnP5HwCBVX+SM1RvkWlHqcpLObQYE2j1W45aJ+VYkBTtTL1W36M+sHmgRhR7IyxOnPwfKUesMlVEqTcSir1Q1RPFzmgNNU3lzZhHnCWqece2COvf6g53RiDosZ4BTdY9FJTqLlX7RH1Xt17hp4LehVLRm7JVuBMejA1ycLSQaCKe5OZYPr3qfvbHxCKdVILtUji2qudyB9lcTWtnAu5dfWfLhstpTfJKlvSuO9mmJ8Sg1nZW2+zQdr7OHMz4ZHCpmEjWaIqZ7GUhHdSqgcByivxXGZ6dF8/e9RfLVD1KN7EsECQR8r7sUQNVBaAqoUIzi+rnVmMiqzVBpg5X7N2DOuJQaIeeTLp11bmH3WT2ZaUtF7epTuudv1KPC0FVxRtrH7WZe3ef9IRfUtyHeFi9Q9Yq3fGcRxtiEK6IcFF+rNbBNauPUQ3/6mxUNxaEX5yu+hzyB0Zr9i11T6ojHEkiGt5b6kybdNxaNw8kTQmNDFA+kEc9OzT/ir+rDlIDr4qASuhqQqt/pBTPI4asDK1cLp2EVQ/Yq7jyqitM5bha+aaOUrLlr7a+Hy6p06+//WIftLpwuPUwT31TpXznzn6I0J+wLoPF3fvrLqu+i9ZRK8b6cseSq/7uIqmlEizE'
        'jN2X6oof+ijVW2BDKbUIi0CwSpf7qZxHm8qiTM+gVWfKy03RXrVqIapRIXvqfCDUCWMWjaIeWKAYEn2ByKXJUYppUZ5TeG+5RTUzqxeUPd1X5ClHq8GFjp3olqunlqrhy7EgQs3p/RAg7wx4PwjpUFb1R/SRpEbTo5MoUo5eY3ML8murcoh3EI15FkfLkN+KSBYskEep6QHsrpaK9s0NoT4VPbXO55PxKC5EaFMJwaQKS/I0tirOOhdhmq12b+iso8ggoaCvRCy2qvbI/ZjzKZkp93JA/VcieeiwD7HItporKamhLrovkeFSbAFCeh7Qkkkj7YrnsPJyHZqQgIKpkkawKVUUk+lOSq/ygpo18cLDGX0/LKUOUBIeAFktS2N3Uue2qqUloMtD1B7QAVVIHeoUssaog9M6hmMKNGWrZAWFiMg6iSuKVycp9Cbz6mok6YmgvPp4XhGcvY+TVQlFnfZWwa3KtRdZddilA0ihWvF870eXqZNFho4yDIvmcDVaunfzVNZ7v8LVTmJq4Skr7Um3qAMW0TFjxo5+CKAKgFwFpjD2C2GeXt2GarZUPwYBkCquerVCnbA10EtjHs3q9w90Vp2pCFKr0VXZ6DcGOq1gYyHrbAMf2m/yvNjuz19tL05f6D/1vtrv3+y++/bb2+1u/037favXz6++ubl9efLzfwCoNZloKzwAAA=='
    ),
    'WFDC2': (
        'H4sIAAAAAAAC/82cX29cx5HFv4qhZyPhcIYcMm+Fym6/7cOmgH0IDEGWKJtYidSSDLBGkO++03V+p2fkyF45ouNQskjOvdO3urr+nDpV47++ePzh/bf371784asX//Xvf8zzF19/9eK7m7ubl7dv5mv/9h9/Gmfza3O22e228+rr7x/u389r52fz18enVw9Ph193F7vL683l5eGlm7s3euFqc7Hd9j0Pr/q1zeGXV4+PN++/fffDXGL8Z36/vZrLfHt7//TDh5v54oeH+6eb27uXr+/f3N59Ny/Otz++frj98HQUq1qs7f78cn/5o3t+dq2b/72/ezxc+vOfW+L9xUHA/unq8uybr7+ar+7PtlfX/er+7GJ77levr3fn/erV2dn1jlevNtv95mtv9vybbz4W5SPt9LNOr36kqPOPrrWIf33xk9v92T3ePr58/eru/u729at5sk8Pf7n5J+z8b19/9XcCn2/2u/OLXyrw21fvHj8h8W53/dkSXyLx/nq/u/q0bLvdfrP5sfEdhHi8Ofx9+f7mze2rp5s3L9/cvH71w+cLub+4/FwhNxuOYL85v9r8AmVvPr2h7fbw3stfU9n/uCHsLs/PLs9/VraX+cc/vby7fzpo/O3t3c2bz5X1oL299bjb7L9c1u3u/Gq/fRY97q932+t/UGNXV5vd9tfR2JLq4B6X11+usevt1cXFc7n59soedHF2gRQHv9pfPWO42v7MPi5/g3D1UxLvP0vi/b+OxNfbz7KVq99S4v9Htut/UgQ9XPgJKa4uD7b/W2gIz/vlfjMlvnz2VLq7+OxUurvakwIuLr48XW3O9ufbs/Pn3lDD48/c0P7ijA1dPUNO04a2z72hX+Z++y8QfffrusP1xTOiXkn8/O6wPf+U6Jvz5xR9sz3fbP+VMuDHEs/y6u27V3f/PYvJg7yHXx9uvru9v/uo0rpSHcqVY5V1Tjl68z8vP7z7yxT0RdQYI2sc/lRmRh5+Pvw+vx1+rYyaV0Ycfo/5ehy+VURU9PUx/81+5+GlyDF/PNykb9FvHYdFDv/GfP+YXzH0uLna4VrOh1RfivnWuebh9sMtMW+Zd8RcLOaT9NIU+PCt5qXSA+eCIakPv1quFnM+YfTfgwyR2l1vu2+Ifmk+sXfbAo8WN/qO+ay5t9KGh36eD5x3900tW1+NXrP32ssdROwb5q3VT65s2ecj5n77Yh/A0BOntqYG5v5b8pryhV6fb51X+24J1KqYl4cWKSmpXxxp7fQyc4XSSbeae5s5X2pR16nMw9byMeUfesbcr671Xy1gzWmtkALmfz6EFnhKHn20xfFL/j6O5JvONFtFOqj5Dm2V86w2xWmNLWhhGWOqtGSxo2WWilt5fUQoYV466HuuMAWqlnlaVugRPCuG7GTqe8rSFlJSUfSmW2Wy2DaQ48ZynlGbQB9PL95KqD58aTJYZrTTxLB5pQ44et2DQNNAQm+UtQRWMu+R+/XbE29sM+idjN55yOS9qfm2liekknn4w2tPVWsHUmvhJilT59yloN5Ha6TdvkInFbpPsaHNsO1E3hLy2L4gw2jPRrhywGmP7hiEbUxFDjk/jtYBJWXncu9svXP+8qj2tZz3j45rji8tzJAiDj/LDtvto3hVvimF9nJHQ05f1Wmf/NwehSFVH162ItrvswNjDYJgR4yBgh13dW6tj7YLztMxs61sKkMR7OiA2mDbnX7o6CP9Epv7LOelSgX34Hz6ySMdw4f/jLIB51C4VaTud4de0SL9yNFO169EH8BQ1Bx91pKqD11/c50O93YC0uoKAdWHosjRYWP0+cqSU67Rd9na50rR34YOSOG0j8zHo1QgP8UcgsjUEZGn6MXEuZOM0BtRhO0DDSzb4YyQQXjsPUUoqxEUuX/6XNoSFYY7lpCHe3mJguO0ldpWO+BmP0uhV8v2/nSmemfIoxw4+3Zss2Sw7WpKsYrVsuYRpMfCXBVySpoDKwShr/zADEJIjBWv9KxQsHBS1eZS/kkS7sMs7FAm22sVpoUJS0xnTj1RD23fKzYq+3b86OzetjrsWp0rdAIzwE6DbQtKcABJp6+mwVESEXUmilWdP0j9HUhtiUNp1UYrv1rnGycZEGMe2F/ngiHj7guhy4qEuqOjvcy6SvFKCbjPT3qVDiWUDzdIBr1/v1UBRMcpu16xAWGnW67jL0G5BDvJ8VNnIvHAi0lESofdXJqITiatS4V1aY1Q4EhODBtOc7KRPgSFjjoioHSMxFKcDFkm0LoTprLm1FkVMLUwx1oBIwSNGizoVuGbwMxIU430OhLIJxwl5QihN5SxSShlpHKSgCv+z25mFC0yLDhDCY3kE+HU2DmiNSibaLMi7WURXzrpjjTo7EQnK5m3Nn5SQgS4adm2e4N1QcmS6SiuS1iiFtmkHa7Rh21YkU34p81m6Jna1TRXvXccFVuESkV5wQJBIQfYQInaGzi0dyUIGgArgHEfIC4xb4wylupTU1mQTgeOVQ1odIg6NJ1YqFxx/TIEepSBCEMhqAGapAbQoRfWmivyZRpqgvcca1HJCAKnTljBD4cVFIuVewbFDWCkBFAboykVastCsG0hxWGEbIXg7cpHiQDRldCUFmKJPcjeA63J+BrVtGZb8/17+0fUQhED8OhzcV7GEIOULLcMFVsyBNVbMhNB7EZEbTxB3aVqrXCkXi+ENSkIhHhVSrTHKY4LrZeNxCeowlZhQIhNaY8aZUiGkhEOPXg4SLeZKtyoSIyQYvpr8LghYCvNdaieQbXdK/wDiDtS5ViomJOiZY21gLtgtTCnqkFJBYzvokRofeVW6TopYzsZgqY7s9rPVrRVqDwWjDKrABG0dQ0fQEPCoc22nPLFpN4MRLcudU/ae3iODEQm3hagQpmytRbkmsvJbiR6AyWlbzu/q2rpCbxGAdw5dcjulhcZeudCBm00SXWjpFhUESyUX7iQ7rVKmkUI6kcdf1HTRpCmFcM6UBfalp03THYJqaK31ZHHDIAHtFZDFTK4MC3uFLS9t9J1npRKRlbms2+3r3fwJbXIA+FRZFRKnMQhamg5pfkIBYQyExBUoLiQIgkOprMPmUpBFIjDwdxCRWFRDkoxAUzVgo2HHSTbgKXCMH5XJFBCrRUau24cMs8g7iXbFf9QFNFUn70R4aByoJa3u1Tu+CgyiqVgeUQ5FfvuK/wq964g2Syrc0nWqun6A/OgppG8RX1Tikgs1WvIc7xgyCMT/UgktD+MgyLCZdIAccjtxc1QDPJqh51IyDK4RYMZEFrVCtrEPGsk4DyKysRobKx8r7yWjjr+ZyhPBOybVBjm28Lps3URJjVkDZ2NgBHaXDNQQ6E8TRBxvCr3qeBFJsjfOUjHRZFtQyYknD9smOEC0nAPh0AMIbWSawhnyRs6FisqIfWi8UYbsyy7oEzsXgir41n+UHZ2FGty1+mH5L8AlogwQDu6a0a4iCKEqf7V+VNl88BOVU1CigbGD1RvY8HwBQ9EtQoZm6tzQCkDEEB9uRIpwdYkUg1z0RTRJyFCiKNTL5lHwSUDzBuU/JGATkXC4FBMYeH3KmKUgGqR6WE0sXI0bwZSG/cFJBg2VaL8BEtwn1NjJBCU7ICzV+AZY7GSovXJGzKdgIoyWycyGvJvxIo6BB2FIe2hFRtyvuX+yfuU0GKFfRhm1J8LNaogyPVAaGrrRcWeAhClbkCANeYa8OYK0+gC+tp1l3EJ9UA5Dg1KxGGUoAOnLhEXaX8t4xASuDCw46hMjARGESp9gx5j8QI4mEwAx9P5K5CGc7hilTEX3pOu0+U4nHAEHqMKP5yate9BjhKZuzhVJxupcXUwxO8vZkCxAx6ztDLMBqc03Bsw8jrJN8caoYyawt2hWliOlEkeHotlFm1MV0Hp/ciwk8lq1QPHUGfP5JzHMSW5JaGwNDAUexXUcDo8nRisjGpl0AVZg3jo0KitZFEXre5Swb4DXI4hV9TBUU/KGETIgsw7xQah2CIK1mHXsNQsvWxWJAx9'
        'LhUhdALp8ynp4GxAqxPEPxxtgg6XT0YZ4OgSjhnCJrUIPHpRMEIBeFBJJOxBEUGcDgMYMiJUa8gYlOlddOG98AwLUg69c8UtBbYg3AxDFTEJsUqfMRY90d4nJJiYCGSnanWjvjgWoqNgxAeERwIuY/UeKqxcEKRYShkY6Y6EDVcMRz+AUE6H/Vg4gIC4U2LFIjkunbdig2xIBb4lFlSjXyZ4V04OCfpEOhl4AuILjswVcRqNe4OKFJJtOE1Ir0p+gzr0xLdszRyh2JcoE3dLCHD/qqho1NHiUoFkIwrYH4jc4XS8SsRwTlg7UQNKdCSKSbgMMLIOMFyUls0X/qNP1GmA5pCDjVkLXbXRQwl24B1AcHcWKhYk7vdJYwLWbjcCIXD9RSWGG5cB6ZM2ZjOXRcvrBP8ROsiLLgIJd8r/uSBnYtZKkPRSxfRRSAxbtNFI0qgjII7VZTV6kH+XwZtMyK1bdauHO/RiM2qFXdpjomFWxWccJ22sXCm0eVrWD2CquNB2MCz6OM3g3JQO4WJ/AUByOkU9ek+1fC7CAJVyW+audFounjNNWugUKwBlgqDlZgsJTi+5jU/Cj7Gqk9WqJ2f5Jupuc9wZK1nqrnBYWjzyWNV1OdZJMGEQ3S8gEUIbRSeBstsjKAXvnSAtz1CMlbsDBkESUnyFeQnOvEy8qoBd6o91liamnJ5pKOCtCqV57PqoPyG442yr+jPLZ7Pq3cWx0twC/pgFhdWNo+UYAoObwbSCulxdh9+F14AUSjbHkY9YTlw8hw5+0TMYGtGpj/AN4WhFDLNyCyEwX8JMjbn/4UI6TeFR8wXEfhId6bkUgzxyZBahPSC6+4QpcQkFK1Jl6Krgq87zkcUnrS5/Qrc6LPoBNDQBOAnaYwqnSYtjBzRMlw26HPSCET7h2dzUHSsuuUFIKySHifthKxJpxvzHsTmf9FfSOx1OWUAS7JvNxGr1ArjsO+iEFuWxvyBviZMdddO125UwRcP4T/WvlyiYQWD+ih/FGJTOJOSYHtESqW+CUoZn/lijBxEePRkQ7mKQgeBH3aaZZLxabKDhr92MoLEa6N0SZWrNrXmffEEuM/lgvl8kDt0AYLAE4I6MNUlCZgvnSjWMi4EomR7TK4PusWMD1FEcB9Wg8E+GlcJHBmx0oQ4FpMGBFmCYkpLppblIO7S6wJFulqrYIT4pIa22UWFaaU6hyoYZrl7TMzyrHc5EhycYtAFjVg8DRngNTWoUUyzGBIoFwXCfe+1mqNzDJWasDmQV+9TEHIplzIIhrDr5wqWEi+1s9MjpwUJoJUGUEbd2lXCXeGlFs2fhRL5mWhh+PDbROxKI4FeHMO1b4WIcMkyFkqYIwKtqHXIwSXhPB0E30GQ1rovVsMqlcM5FpfIKi5HuiBJtZcRRHpljbGVlZ4bHGIkbghgU61BNq13RW1WJwjiliNUwU2GwG7FomDVxNjz1oSzHNJ5brpB8Y/0pkpYifGggZBzRX64uKKSVm6s0Z13fEXpA+bKs5X1uaoXAIdpmAkRwa5Sb0+k2StEoFpYWDi9U4Y4uhiDAFkiswAV0WaOIrrBzccIyDMUOc74KpApFVjXRQKg/GSYMU1lBKZerdqOGBrnAU2mIRnh1ERAAPoaHgxa6EmUECsCNbRwFBSajC5oeYtbK6Y/pwlqtHmpdlYPqpyoaqfNEkRpgCKK3IoHTPKnNXC8zzJpT637XYGCwjAUo2OWErrE828VYE5QYwAh0bGuGeV7zssPjIOq+14IINIDS3JnSvWdCxir9GRA2DguNWith5HHuw0lIIFi1WS52UfigYs10Ahoc8dU3jtWEYXzQUXOsCV36L5HOzXUks5lZ7JWHKm1zd0AxjzB7itwtG1mNBqmYYtVsVpj4SQC0cyFoLjSHTn5N5jXCkxuqsU0VC+xouiXo/whCRB5HFCH/bf7cr7GqNRmbxwJD6aTWyYaHu+QXiW844HmAWOkZoKckk+7ggCB1c4r6TNul07fHpskWqnE87AqnVBqdEJUfP/4SvXgqE5VzrIxTWOySyHovQ5u0f2nrHpCONZmocUS4GQusOQmpZqiyymD2qMpz2us/JosK8ORZTVgGysoBHSzi0wzOij+qPBaTr3FsNdYCDghIYOxCM+cETaMrQ7WhKURO3M3UVbl2MAl/GCCPY6qqG1zRxDFnuWBPD5WYu3FCGCbm/HGCJGFWHptvGi9TogH8lDsQpTitkF1uWFW6S0PTMFYgVRvCCcy1gZRFi2JlLRzXn2lwrhwEWujMYOpbQD3GGnAYnidWgB2uxX50U3ziplhlNuQ1H/WQjMKcruf4FAJQd6z5qWFXKFYengBmFJ8+MgV5nIRkI0KoWOgwJiTG6sKvfEjGSPMfBoquHV0HrYmwRbaP9cEHz5N7EI2Z+nTQ90AfLdj0zMpSacI0rMY8MyryERNPnp8YdQKIGfnAAwmPcmo14z1A7YnVcFHlT5sw0ss1GlOY98LYw0MR2vBK/8Fb4LHQ5PAs+OI1EwSg6ESQ88cekuY6nli5BmuMEBW+AXOD41EOFL5YfMA4NjIZwGFCJ/lICQP2AIPjfBjkdwEGE4zA4JvwnYvg5GMuzKEOLSodQkSMRb4Vc2kQCvpHOFYoi8+m8KEscVtQM6SHsRrgq/gEiq3RqrRFuG3skdzwJ8Lc3QSUnYzJLtrbpC+cRvnTUHiHiMcjuwtm9WTSafedRmquamIAg/M4d+BFarFgnmqqxVh4hhC3UT/KnzJj5XI95QVN3iNfuP+YkGR0CPkcW7nJAQWnTy/BcYanMwRbSDFIx6cyqGQ1BqVYlms+JuhvaThG3SO7cLoX7VI7SfsxP+z49ubp9fc3b16+5X988/3T04fHP/z+9w83j0+/mx/pfP/tu9/dP3z34m//B72lK75PRwAA'
    ),
    'CFAP410': (
        'H4sIAAAAAAAC/9VcW2+cxw39K4af3US7Wu0lbwTb8q0oWr4FgSBba1uoLbmSAtQI8t/7Dc85/CRHlu1EVhLZllbfZWbI4eXwcJKfnl69f/v84s3T75489b/bP1eLg6fPnjx9tT/fH5+djqt/+8e/42B8LdYHy+V63H3x+vLi7bi3XIxfr65PLq+nX1erw+V2tzqcLu3PT3HhcLdabuqZy5O69pfF9NvJ1dX+7fM378cY8S9/fbgd4zw/u7h+/24/Lr67vLjen50fv7g4PTt/NW6O969eXJ69u57XlbWuaY7tYvvBM/eOtf/fxfnVdOv777nk1bNa68Hhcv3Dsyff1+ft8pBXd4udri62K15dHixWujq9yKvTAEtdPdocbXH1aHvQz27Wqx2ubtbrja7uFgs+uzvcHfzww21hPlDw6vbdG7qe3r11r4T86elHFXavls6ujl+cnF+cn704GeZxffnj/k+hu5+fPfmFyKvNwWK3vlfkY//rv4/PL66PT/cvz873p3eo4OXJm6tf6mC33S2+QAdc67Sc7UfWulsfwrF+l7WuP3+/jg7v36/1dknrWO+WB/fv4rQEPrud/n5EM+vlZrX8QDOX++uToYPjs/Pry4vz36qK1fqzjXR1sF58rpHeEG+zXd0t3uFusztaf6lf3i0d4vDHpfuyjf7Njrk6+JjMy6Pl8vCryvwrjfvrybzabNeb1cOY8Ygjy3sFWa0/sozdbtLC0QfLmKa72k9/j9/uT89Orqf1nO5fnLx/yD34VQnh09peLO4Rc/34Yq6/jpgH9+3m5uHFXD3S0rcPv/SjxwwGy08Jv7xH+N2jBf3dHyAALg4KEjz4hh8dfA3hPy3mcnOPmJtHy22fkG656KvTaL/Dhm+/3oZ/ra2dRLpHoN0DC/QpRProSWVxsN5tHzypfGG99FXg+GfUjBB++0eBpr+n+y4W49uDaOJweXiwfowVb5brwz9K6P3Vu7T4IpkHADz608v88X0erNjLNyfn/5mWOVnkoLou96/OLs5vEWQbMJC80+TYNASIyP1/j9+9+fGqqMfpyz3N6qd5ZgY+T1fHT4v0rM91Z/ps9WSMu+PbeHv6ZHgmaqSod8e16fnpvfpZo/n41dJyGtOnZw0zjZmnl7PGGwOOdyJqqhyz2Rglp3WOgYzjZz1Rr1gNbVxQ1sdp1PFJj08PQRCsv94aA4xRhjzT9LUk/DYk9ZpkesVL8CxVWD0yVj79sfEsZuoRp6tYz7hiiTep52nILEGx4MAtK6Va6ZYrnoYb2oPexqod34byhtBDV1YzZU0fdRdDjI9Zuq7FDRHqzax9sRorJCZezJoJappu1Et1cygP80dKaaVfvIgtqG2bvuOyjQ8wnTHBNJqXdXD4soChatrVMJm6Mz4k12U1fS0VexFQRNmglWQ5vmAF5vjU2xkYJ2WkDom9trrGkwQJ+0qabNbyMWhAz9BVYF9rS2GX45OVnmqumqgeq/XhssMf+Bp20WEZZf/wgZJ1rMG4YE4xNI+NMLhJPTSW6xjP8TsnLcnhisNKc3jLtBTovmwyy8DLgsfQ0wNObzfsK9ZG1wysWpbs3KjEuoeL1S8OddcucJD6gVBhZdz1BpVePp8l8B3XhtMn7ab8LTFbKRND1wJhp7MBz3YCb4fTlfXVM1AZwotztoAtWcjFStTxwvjutXCn4cG46p0RvYZsWDfWn1RxKZBGzDA6B8eh8RGkoA+ss8yDuqsAVXPDFUrgchZaesmRCD+BFxi2A4tOTGgJ5614N0KTwUW85kHoLXmsQhg2B9uFuGcMR4zF45trPZEdPhHhsQh5B+MRTAmmB1GpjoSmK87hXnQspTqQJiBKzYJMYTDdEqHWZvRY+Cejnv5V2Fe+Cog24mDqrXGz8oOX7yAEYtaKQGZl4lIG97O2LOH8xpSCrMcQx5wD47AyO4ybiShvtGyjT4bTFsfDZXJMeFAfb43FDIdNJFtESaiybnL5qZTKJF37Zz0m/BsputyIwQvbjVATSlxwZ2fwHBOXk1WetTlidHYffkW7qwiobUlkPGRARkVsajKMIZ5gt/CIAIRV+kAUr3Wk9jrgtkh+EIWeC3+FISMNpkKHBz0GccIVPBD1lDYbdNAv4Ehandv8NygC87eSN3OfrNOcAdxgzYboFe1TDLzWciNcIDIxusHLDT4MbQRmQFYleEHcNM4DZViZLB2i/A3JhGbqpS3unJSFIMQMzS2mYVGF3sArmTcMgEqqoYTWgKvkhr84gxlwDD/U9RqOozlkCOle6M2Ix7J05kQRGJwRTGmI8PC+B5IOS0zbH6As2IkTETNXuLAWUdmMrgBPldyoAIBCuLhSkGJdWYAxijNyAoZV7rkxQCBzc7cMIZ/O4tq9iIaaSUdEHg3kCNqyBXEsX7eswQlKksbDlIPoTHc2E5KzXkDlNPkCbzMOIWWlMZErDjATOKEHwRKSiRfgQangQrFSE8BRzOnQkXkc62eEIaBwuIIxfsy24t6uiwCMkE+sAlDTYE1InrbvdLpyKaqSOQxlS4mAagFLGtta3sQdLR2GtxYdkQjVBUITQkrblUJEcE3wPFhNGiurhEgeCmkz5gsVWKyuGHNbwFa/4qUBIyEn0HsA1mYAEO1ewFuqVxB2Zf/QZyCMIDcAIjHH17qtFJ+mmhMlEMEkVFt521KQzYQ0sZVGf0sVk/2ioofR5QCzNDZwurFUjc6GgHrwGmPqUaZDWoW7BpO+p7OaoIEIt2BTaTeNg4hLGHWqeGQJU6GFVWxwQv7RbaPxUdeQF/DXVNf1qgApanhUmgB9QAEI1YDblImRXxHJMmSS8Neci/WOXHP2RDVnqGYY1QEFoTfg4t7VNKXhnMcFDCb4owURI8PLGWpY/KvQQI5zE7b2ZAplrEsjPDNTQQRcPIZMwEsLGZcXRRGoosqFEsZfN4z1laMoinrIgM50t2BZqsAHhGOhVD5ksHakfdg1wH2Ggr4xvmDXHGSDYwsRlRBpkVZQCSaqH9ZATPcBf6a11kKCCGqGMthnjRAKNp0mGSXCGTmZBBFPmLMRYDi2IHx6Iw1YKaoMFHy1mYAsiG2VzIgNWbFhu8tivLfcWFlwe40JFigYcLueuo0Og4VUad+pOJW8MCiIBHdvKgWMRi0GdA01a70GnykCJF890FUOQmFXzd6IadgNoj5zMjMuJmkuDV6RcqRhXHRqImsCthlflyDi7vJGYkLS6VxKMwB+hqYIPQXFM3x+KFA1scCAURIZm4blUks6yEMAaS0QJmKUwpMK1Ygr4Q1quNmhMMLam2nFGKOAdZGnicqdexqMBnMGpYLAlTARwl6xDVhGqnpnbok5xXHf01AQmoq6yI6h2HfyXtGlLreACZyoxEzkorMaCzqNC3tYKxZ+VlaKMN0xX7wRODTWxjJ6IQBvq0zyABW6iBtpyo5QJvyPgpfZS6EV5bix6DOyE6FqiVUEtAgQYWANQEVR5S7SGfjS2kCYlZIRmDgTu4m6heiguVhxEFikUAbrUMJ1C9JzSI836i9iSeZqnys+FkAmx7GbxpSaMVhAJpZGB3MIAQjNGpVDEpuqlEd6ZX2m0JtkGTxV3KJkYWSDx3L3DfyGEUeQJTUx7dwulL2kSEFKidxCHvc5rytBgI0dQ1VxCfFYKbPMF+yjk2YXxg0ojCU4hUwoQ+wgd0QgrhymTFZ0TLASNsLTEB/OKvc2XhYz7DQ8RrtKfkHSzuVRNPAWw0XhMtPO5pwCmhWwADQ0fdOkKKTiRmMD5TYTPGhZTix62El8ceuBUlAJ/uJ2f8W9T+at2zQe1MANEI0gsvs5KNiikIzI8aTjdQ0r82CGIo3lKktEFhFsWGepiprVY4JpA88QlgQNFvgcXQ3woy5SXOkDPQy/kUoQQc2JDlloZ9PkVE50AnLWTbDAJJ3vIi1J42TaDe7Xu7nF1OpzByWSBSQ8PVEeyweiyyzU5Sn+bJ7KWJwRNLBv5TOVxNaWOggBhh/5AE/jCqoWdEkYd7Az'
        'LE9Y1gWJDgQcbAZxvLN0CPUpDJhBEMlYjBfMVT0DgEzozhgG4I0cZmpggcE3VPnFDnK7TXRYNh0XotNSXDM4QYWnVHUfriSQocWmuP+mC5AqREepMZdCBNY5sls4ynU0IN4A5Zrq0yR5ApfdYcFGcIT2UgImRPcGwfjJrpz4x6IbRs3NIgWYgpzytpNuNJvdi9mW5RubUiQl5ir1JrhnC8aYM5LVHAp2Y7KBU6dMmt0ydnpmjKVFoS4XWBVnIX+fKcLKHamEgMK2i0W6qfrPYFPYc4HEBP4YMxEu6MuzKpOJihUOwZLCFyoAQnj2GtTkbHrDRFeWB1pD2RTho+5lQ0u+I74w40/2js3vVOnIViygP+yOJp+pXqRqOBQCGSpm7Ab7mq4+H6JftfhQOBOriJUAvYJMy4IRe5osvIINHAO9mOTyukVPl0m2OxCDkoBPrQCW/kg0xmJbGbx6VBXkiLXotToBQFyJiK0uoEjECkNkAcVdplrKzohO4CmoYKaWDoIwMDgyCxsIxfe5zhbcoCVzZgSkMpIpRTawNeDgH5AiDWuMisE4lIHGLTm6itHoWyWDE/m7sGRcp9bHKCa35Rf3VnVt4pSFs9gwNfRYf6DcrZ9Z1ImzX0+yhyw12OEkj4Eyf+408MADowhrqiBk7uIQMZrFO+CYQFEWLVQpCrKasExoIeofVhFHoJdz8dbrIANUBmKuYwRI58kug7G3mWBty16sOcZgPaQOKZOylNdfauSWpQWcNcSIlgNkcwhFGRjLV/QUxEkiO+ILdYHRqY29JDMGgI6uoMdy7rymTjCgmIPujTiuAT1RkqmFZawJjdStAxwYq3GBI1Lm6ItnJ0K4r0rKhNaZ39D6sG6OGZMucWaiHjDyCED86qraTEEmWWiQLihHATB4PsjAy+HEgCn9w44wt86k1Cjk4MojEkHC6RXEUaH+PNorhEo8hlV1XooFMdG/8AjwTIy5d13rMxMZvfEGIo4+hWActaIAESqHdXa7tVZ+APrRSYZk+wHun0zYwKfZoas51yDHAP9XsufRF5p1vYwDTJnib+EwCe8Wu8fIm8wJSEOyWNoGcGcTwBAf8YJcNJus2Ty16aSXmGBnaVonLNS/VUKDdctlZHU6QlRbkiIvnWd1BI+1UEgtf6PB1XrY2BJB0KFU5DvZa1LfYWp/lNDMYSqWbO7nzLZA9VWk5qG2aHJuPhgIrqrrd2xmVxGAmyJ5TE1nMI3C4anl+dzAJwPeR1zYxiT2U3si2UswNa9uQFkeWewjNM0DELxV/lLEav5lBrUe3Qwh6Zo8nBEEn6JWg4c8oulD2Gi5vTPSeFMRouyjT/uRTyyO2XQ4ztXKpzOw9nH1Jwi5vE9vsVVrqlMQPlj49Ekkx25wT6LbV96dJFISoAi7JAifO91kKHTYS7T97W8Ce6KbeqB8pEfyg7V8ydWHmfGuUe54RMxz15pdceqURmM8nsAFYY9STtQfj8qmTmaAcASoSRdVWik2e54+xwWOXY352TAAIlj7VoRFk4jeigOxagzxVIhXm6UvWlOa1bKgcfUBS9K0zmOiwUYrW5158ySEseXKClN8o2lBfuNoQbJEKRyU1d9v20eIQjujz8TMRz/Y5UkeFESjg8FDJbZOpKJUMnfJkSHxSM+DYCLeBGwUS8jqXfxykpbu/AgaX4ddkEhaIp6Dw3E0cj1iknWcAiU80ZJOuPEfQqBoPaI8waG+IN4taYAYHXCOVJ2h4cYDc8y3OiBaqMVRPGRzEvg1u4LisSn9BIYCsicYRRtZYI94AfbOIjN5YorJhqfOU4+xR4rVG3u6CQJLWAN5WCdtqAsUMMBrVO7cRk5Tvw9goqCaqc64UxYcxySGwkp44iBQB+Tch/d230o6SZ2ydzrzZuS6k+UNmBAhyGATm03qQO0lnGECoyUfUbWrOFMF3OcElfQpMI+2qaIrcw1NYSgRa95okrjZaNfJIuy5uh0qmVxHbVHG8xQatJ7Go0cqrsZ/0/Fyf/3i9f70+CX/vyyvr6/fXX337beX+6vrb8Z/q/X2+ZtvLi5fPf35/++eyNHwRQAA'
    ),
    'ERGIC3': (
        'H4sIAAAAAAAC/+2c3Y9UR5LF/xWLZ2umqut73lKxq9C+7MNMvo1GCEPbRmuDF3qktUbzv++9cX4nbzXQGOPCtAEM5e77mRkRGXHiRGT968HLn3/85vkPD/7y1YP//Gv+V2wefP3Vg++un10/fPqkDv7333I1/1lf7U6n9Xz28fcvnv84n7tazb++vHn04mb6dbPbba9WV1fToetnT3Rgd9hvtnXNi0d1bD398ujly+sfv/nh5/kR+df4fnOcH/PN0+c3P/90PR/86cXzm+unzx4+fv7k6bPv5pPz7S8fv3j6080yrF7D2myPu+3hlWve+qzr/3v+7OV06u9/14gPx6819vVx94+vv9LRzdWGo5vTZhzdrX10v1qNo8d5zvXTabv20cN2zdHDbjWee9ytTjp63B+24+hxxRiOx72v3e1W220dnX46jKP71UZj2O3XV6dxdJoGR68O63H0dODaw3o1rj2sTzz3cLVdjk6z+3qo7B//uC3Q2zqeJXZ+9jV1L+dK0P96cKfS3qqppy8fPn707Pmzp48fzQZ68+Kf11/09476+/fXX70u9t1hc9r+WrF/++iHl2+S+5DaPZP7dnPQE6afjvv7q43t9nA67l/Rxovrm0dPn10/efj02c2L589+hTr2p/ujjtPhuLlj0sf91f7t/v5h/MffHj57fvPwyfW3syh+hQwOF5PBZr053TGDzXGzftV3TUN7eT39ffjj9ZOnj24m/T25fvzo548y9HP1bfb7NRM6rrZvUOp2su83TvPqcFrtri5mnceLOYvddnXXglpPC3d/Ife2Xt1P97Y9rfe87TRZzUd3b7vt/i5trHeHw6W0sd5/mOWxQ8Kb0+qu9b7e7I4Xm8dm9RvncTju7z1YWe3vson9evKfl3Iq69PVR1uMb57eab8+XMxnbrd/ALC6v7orzp9Wh/WHifOH6a3vMdvT4a4VfrzabD4QJjlsj4ePopnt5i7N7HdXp+NljHSeyeFybuN0l9s47KYIcKkhn4glt4e8DG4a/l2AffLfp4tEgukd23cX0fFweI/IvN3d5aZ26/1qd6l5nHbvrurd7r0SqDtzieNps/sg63ayAgD5blqgx9+Ikba71Z0z2By3FzLsCeqvLpmfTk7odPzk2YLfJ9cvWe4+dVleAkSe1heX+/6L3C+b/O3X7yT3w0eV++60/fwY3pL78Z7b++b4acr99MXPXFbufsLdEHCW+2n1Re4Xlvv6neS+/pzkvt8O+YzU7TK+47RZb1YfG2efx8rT5xErS+5fMPll5b45/CI2PE0nT5svcv/tcl/vf5W9z3K/P/n8Bypk/FGq/6WNL97nl+W+Ox1eJ+IOq9Xu4tr4pHiC7foPEYNnuR8+9Xz1D+WTjp+4T7qjgeIPpaPPlmlwtfEjyP20Oe33F2/72r1729dnQ2KWqO+5GzprrDtdvZkkWa+u9vvt1ReI9572c7U+vJf9SO6biy/Vzfrzwtjvvazfthy2l1fLh1oX20/Mr0oBu4+pgM+l4/9tK+DyGGJz+HQjw3F9cQUcL66A/b1VAO3N92sFnC6/eWLz7goYff2f9haw1d0KeG1bx33dvfLBFPB+ot7f0VK7Xu2nV14e2ezu6cbGj4phZlGfPnf7/WUF7H5zCL1DAevV9MrLJ1f3dRPvr4KWv8sKKAVsv3jw30vUX2p0v+jkf6OE73ndbXNYbX5hk6DmcS/rWKNH4BZleec8Npvj5vDF4j8orJm/W+PbHx49+5/5+1AmqX897/n47unzZ7e+ZmOtr1LhzPIVG0e+UeX6fx/+9MM/Z3E/aNl6j9Z669F7a5k9p/9NRyKmvzGfj9anw2063+fL5uum/6arptv0J6ab5rumUznd0HI6MF8WdaGuqM9xT+gJdUfTC1IvmD6no9N1OT+r1W/T26fLpkPzmeknLmXQMY9wul2vbXVkvrmuy+n+mF83/Ta9KObjdVmrK+a3TLfXxfXY+SXTLfXsVqPLXsOZh5F1YT0r6sb5s82HSlSZXNTq+dPDchZcCWGW5Pzs+c6oN84HkhG1qKHWHRJsaaDklfMPvfRU+kAV83t7HZj/el56Zp2ZBVpCnpVSEilxzdfN0y7NtBJ3PWYWQ8zH5/vqBfOAp4HNA53FUuf12C6RzefKSqS20ncNaFbIJJtJxLMkS5+zwKaf5pHM19RESwFlfTWY0l0JGzHUhb3e3uths4BrbmWPs+JrjGWrs/hqHhL3PMfGsSYllhalx7KWXmOcVTDbfc0ua8KWHGMuO50uLeHPEpsNKmvi81Pm36Sy+Zr5Y5ZKCUfjzdSEQwOUXufpzteXMuqaWR7zfdODUmqqw5FaALy8hFDLo+SXNb353nncpbvUHTZNmVCevUNiqnHqTEmnVKKXRsiWohZPqXsRfdRqnR9Q18jiOnY3a7i03iWdebYypUyt7CxDKyl0zbDG26RTzbeNqdfVGvYyhdBqjtJQXVQqqTfMI6mhz08vVdYhOYOsRSS1tK5rpcuaJYtmdgmtLijPoIf1cjLya7Y1OQC8TpbFlC3Ms6nx9jZEqtFKK8uRTI2jvEF9lrTrLjkwS7icSik4pf/E9ek9UoYMsoy7hleubLqoxFeLPvRqregmtcnXNGkON99LDSFVzq4j5EBymWL3GO3yGj7QsrKnDWJMLb2wv9B1vp2RK+505B1yI6Erhz2V+iRw6a0+5SvnE+Hjcv01JG4IbKa8acdHdqKcpte73VamJhFygbWmm3xsV2wKxpsV2eaVHPN8kO58WTKYRBmlRWmmzpWNSGwYOCuka0w1q/FOeQ4FQVmiXLn8v9zH/NQ6VN5YLq+F3AdRsGkF1krRWkVEEbqm4zHrU/fJv1XYVlQPrpCf7nKrcpu9e6b4Rp0NSaQrxujosOoSV90xPyCJQbK++RbNULJNm1oIV9Qo5fuIkaHlq3he8UZG13guXq3Cj8JljaLCTDpyaxrIQWORdZeFyZ/XmOw1WCIKcDVUgZgKaKWImvw87SY7FcCRvCuIdLQjGUjsevEsdvyB7AC0JLjEBwFeMyfMaIkInymg1pkawGw+EqximDy6o3CwXmr68q34owQKNj05BQgkJ4u0bgfEhTxMU5TvTa5uWUn4d4CTJJB6cBLMw7ADx4awMGLMIlu3P5KhJwDV7pQzNlWBBaGH6AP6CmKBvspvY7OzEqRL6QdQi0y7YlsOC+JFHb+tuUjAQoZlYaE1IBMqyyoQU36osANuWsusbvo1/5L/dwFgBXuZTyr0NZAjqKgCsrGiLk375azP0OqQD+leAdF9VSR4h7nKZSuMKHTY1ZAdJGlC3TYPswJgT1kgaMj+o6bS5R84n43AolEoYEuRr4xC2YPmpIVVWAUAKlUpR3FuIxGSGfXgjLBgWRTGLu/d5Qh6d3SPhlsqm26C4nWDgLbSjSHBmpaAghYHGKwJkzP5coeCNYqPWsJaAkHWogXchOMKV5VjEfZIUDNRudfrKmoTQmXEiWeVXQeQVcErw8BARk7UxwWFDKgErFi8gOwKr/ZldUxKkXBCTjqUovk9qUxH4pMhgWW0LN70RyGqJl4DeNXWSTsThd62dfThjMW23oatAwvPbD1tZcBPiZJnMEabetrUBZnAVALoHqUCkBKOFPDKV4xdBhw29v6qsSdLjoUO0ujhbBuRhpJo5NuBEs0z5f8OWimTEWRNBd7saSNU6lxpZoxQpIiV+BXkKSgkcNW0+MssYQaGLJqDmiZSuQqRXmLNgauUhciaIxxPFbPIO5oXfmuGYMrw5FWa4E6Ttz/7I+xYjlzjUS5NekaWKzri9niSnLgWJqhHZxbPXysghQPCHkpRSLlDibMByVrDpoXdl79Kh4lkrIrWAUxSZsU05c+hGN4hRFrNPc3dQApknJEsHaakcG4pCnk0bhalkrgTpaD4Ag3beTBURnPMFAFQFylv7di7cp5CRLhmLbWCwaKBjE3gpSTc0hewuYEdFZgxhwZoEbYyMjVi1hSbSJAw7imnIXpCxI8wq5wa'
        'EL2P2C6hiipjmpqiUu8O9ZOYVskrtAxATam4YyooAl6s/uuYtGORgFF6CSozL/ig9dKMKMEmEoiTXVs4i1FZG8mdYLbGLWpLaEgILRWSZpUlQBaIxi0MOUiRJTUht/Q8IDlMxNXgmsEh+RlLSaga0CdQ7gXJi1MLX8Sk8gSHTyApMxb9opGRV0v4wqgpU2OcATZjBQhghrkxY+WxRsjLGseN7gH9eLY02zkyB60AKLIWVnVKGmSL3fPuZIdKI6BJC1OiLCSsFEeQAi6p4FcDo5PREwoFlRsUUiQmlKwYhXPMJ3l/KDXpkp0xiuyiwYRqjaV4oYCRRVeJXbE0oOgawaizdMSWiAXuNjuAhMiQdOYNjarHCd0UBSJuBsomlLGmLg7eCfYeysOTS0sBf2hjlf9bCN3g9fLuQtQVTMTrscx4pZKnAmAwoCJoZVNwKSNAmKrpjr/NKkaZcGnhRbh4bdYo4nF+JUPT80UCKi+sNK5bSGOla7imnszrUwzAm5C74sZ6OvGGbhPkE9PUpEGRrXqOKGl+Dq24MEzpCF5eSaxhG96jMvA0jWH04uUvksirFNcjTgxvZLAL3x7kdKbfAxIdxq+7aOFQ3pRMLsjGXJssztyYgi6sJ8DUYXZ8dmYJECP6cDhGNoiziREgmTGhtw+uex4EZGHgKiCBSRJz8JgNelXeKuyzOqIkeMfi07Kb7fFV6WgDLVSOuJGpRDidA/N38HZoBuak4f3P7H3hpUSQpBdSJxRBmZAyyzGZLaSWJVNjhShKp2yO1MMU0jACwFHiyg0ZYoQmUiAogRZGKQZfgKZyg11+zLUzRV5RM2TA5VwFncOuA48XnRKN+JEg7DQJz4GkkW11kuFyEK521QiowDUHH01GdRBIs3NUS47mICiYSgFngbWjqNVhGgyzE+4HIWJwxthMX9gDwGCujPwqrE6VbxpwXw6KuB82mYQVTdIzIA1gWxasjEdpPGaRziVUprDr6EqiRZiEPHKQ2wk0ymZcV4LHVPqekLGlCPBikD2B23i7tA94U/XF3AKUF+6FZMJFPyEiOPZUlRRSNE3zCmG3JaiTBZlwTbylx+b6JzYtZUO422NQrAjgnbg+DRNOjqyOOolFRc2p4UO6amC8KUeBLEjutBBCIZ34FGQQrmKA8cnnwpOHh01cl9yWYlDAVIi7tYFiVsrN4KvEYQQcD7XRZAmmOXXYrQQbyFEElWjUC0UIOUNWLNgAgQ1XqVgJU0DiT7lczBPUI4hD6xIa0bCQIKlsCT+alKRg6YUGG1X9BP4IRAX1Cgq4Sh71zCV+DLBDVUHABmGMVdtMDrUOqY3r1+JrA5sLeYDk/FbVgU12M0HYCfmeGHhmVB26VeufUUPgf8UxQg40nB8DAaRC05PpQHKRObtc19tIEqCEKwh13gIEDij4hmeFZep0B7j8bripCiJDBtX0qinJZcdZJ8VrbG7czeY2V/7bK2wutbakB+SMzW0LGOnnDFfP4fYGmxsLm5tvInNJ10Ha2V9huHreZriWpQ6de5vhEsfwGp0bEDG1EuVxLDVFRBo+iMhKappL+FoL5EqgMkEMNZnYx/S+lGcExJqbB/Ks6NdtyBRrlZ44jVGxrTvydcrpvL7B7cDzjv4RUnMK5xRCR2U7jKXEGts/0t9S0khT1q77dH12slr4EQF294gQJoKaIAkNZJxDEUITTbWkuWa5xJi5BSKoBgpbxJughW5YGLz8ZQZPvCGMWEczg8DrZ+arsbeFSPQwikXKW8RdQNzZSsHJVKvOiDvIEzB9d/nIJcPWRilVJ9wK1YC4sgN4nBy+ji4kuXMch0pCnXYNxlzIWk4gRoaXrY3hpAM3Ssy2+IUEsjbT7mbgR9sA3UXijuFLBP2sYkDqoF+dUJZ0B1QwXZMUAVzRdJnQnKqL04AZit+dOnH2M4ZIVhF4UEIUSzHNP1F8ANXJkchRDTBj84Zx6nmb1b1lo17wcpwuSJpVezP8JfumFHKLZTZRJoerFhf1B6SLVAvd/ZqVdtPLWrdgSpbRsNJc6GXKcMSXWCqqeWalBcQ6tnqLBG9ugOt+DMjevqgtkCkNz4ocIrxXkaMBOntzTxjUZ/aR43a74nRlIRRu1QUBCZUOO2rpac09eH1p6xuNDpSIg0FRoFNOp+zNtrV0tizUEIVlIy+XnKlLaf20UaNKU3zhkhopQJgzp2YIiUYPyFK+VlvDKMrTjNIoFSrsiT5OKvD0VMQSscbqEQWQZ4RvmoWRBYYJp6YOmQ5fIDy10PwisSWO0ZBgkyTP9OjN0tBWE/QjnVOwdJ/koFUJ6ZAPcAFt9AmEi/jQuSQg5W2SzI9KQXfM5l3VsxLd5DRcb6cfzqSp279gyJWGmuxt5mbtADu1MZpimFfKyajZqHk9Nq+KcLcdWIqASpA6K3Z2ekRAea42hlkkgTWV/fpSV6CdI0ks+iJct5JRKYRn9Zyak9cmaAY/bx5ltK+1s/TLfWu0TanRhk5Yr5hO0ZgyUbq8UC2LrL2UX0yTJXTzyXUP6J00Tg5MyUzc9hY8fAAjFkU3YA/XpZu7z2iDcVtxc4SDhqcMaguIAbpc2HOQoSCGvYApO8MfRLbWetKw22yZ6d6bnmMJDIfeuquYLoHD4alfyeFsLL9AYtRBoDr74EDI/cPdMIQ9mmdbjDhnQh+roTlFJiDUCv2jxlS7B5m0bHvUpWKBBmL+nHU2eyLajbv7AkWW0mrWzfvLKzUDWhoG9NjRfNUGQ9Df8ifPMyUcojK+874X+8KzFlNVaXHrS57k8oDjf9DvcN71Am/w+ljirBkgCK2W1uhvzbP8SL0JNAAE9cHbDQCNfjRKADEYRgUvzHZJuEBggH0MqNHDCx1G7paubph1orFQaW9S6qDZwLdQDBEObeZ8SAXb8jpzwMYyScnIFLpbYl08JJhSDwHDDb6vjeIRJDDVI6pEaI78PN1OyfAayHk0/yuH6SN5BerAE7hvF3bnLAlUlkmPEJyZmyL1RLpgA5q3DwdB/E/8jMzD7WweqNC2MdHo4Yhmrjpgi5SHqfEHvsqN59HdTwL4EVXeSSuDXutgmoQxl+wA6QPRmkrxiuddsrToIJHxOpF8bjzs7vtq7ni2aEmY1CdF7o4DFBrsBoNKR7VKvWY6yQv9n9Ec1c0xKykhVEP5n88A1hxPKUfR+2i1Tktx4FXR141KLYUOd7N1wVQ1YLVhK2G0QfOg3VmMqlAfhac+aNaBDGhIpU+X/RzquOqjjYN0L6EbWeH0ogUdO330tvOz0QieYmlz6iM3KivuFE7SlgVjDsfgyhidOmPFSNuMAuKRuox9vzJVKA6ZHM0JckQUmCjTKBkcdYFlxahG41iDoY5GEE+JVtjBp3jfDQxALA6UecRofl6iw9isQxm3494Nh3GL9k/yoEt5n90Xg/4f1IBBozvrupNhZ+55FuPDoUW1OArwTlqVazZXOkaPPRbIHhBY62CXFIkAePKMeJRvx/2HI8PSZg3B33IhqRiG3GSwUlqAcLwBI9wSzs/Yq57mdvZmPTlyFzbRPydtrQ2TXeqrbhFWAUHDoITkvjaZCsEYECqi1XhIpJNcDsuLZqiRaIuZp8RDBcCtIuxgIU1AgO4UYrU4fwnS4RHbujaXjZYq153dg0m21LzxhqIRMznDQ6qKY94UQ5u26LSxiUGNlW7VokrL1g9YRrIhRyPa9W07an1WW3iMlo9O63QDZJFj9cXdqUWhjaI6P+IK6NghulnszsOholzu83YeEo9iQsVONLvITDO17tcwE/haayJ9kLcZxdutgJTFAmgvMMYFo8mjedOlG9PPycS4RSaaTmqv/cEYPd5YirMs1H628Lpj1a0cXuUoNQfbW48WCpEIyJhdCN5909ksF2OnBntxsi8bqGApWKutDxzUnNV1w1hTaKoEaQVaf9R70lvMBuFIlBvgDRNuYkPo/4ZJgq4bRdv0Zphw27E3wGnbX1NVij2r0E9LKgDfR1Cm1d6QVWiC3QbRxnF2WlLt'
        'NGenFpgYlR6NG4c7StZwoHJV2tfYh8W4YxSHFtADdIazn2F0h7jhzE7R+Ie2f2wvBnBL40S6iel9xhmwK03lcy3mrg1d0emjCkrrpO8kXr2bgfYZg8B0Myw1Cq9edUT3EbaETtLR0FsrvVElz3qcXagZxF6OtmH3v1Ly8mZG73COIeYWY59Xg6MMqiwER+9qNYqh/zGWncUu55slpKHWbYgDxrs+Tb/AAJmulgMRCY999Nw3Mkg3GQ4+KNp55jJcmuk72kRifOZZXjoS4aCV1j2/3r+Lgggkbal2Jy3+o6gbNDexfyPSbFd67xlOQiVHj5Q2pVg68MwA4cnk8sHmubTsxNgamqM8wO48d9E5BA0xsI0gqDo4hrr2QwMf5CqVHLcks0f8bKdYjryG7LKNotHYzQkccdE7B/ls8056rchzYRgLDPSztmFvSOgjHw1nQjnAv3sbiPTFvXS2jkgMrKCAWRg70Gn+i256AODFxouOZfYYnet2ErcWqmOME9LRz+hqtUlDd3VJRuIzOo00/KNr1WXvHHQp0MK0z4KfuuVJntTcSW7mSfFoSdOSqSVjCdehlAXxtQg5Ks9KkMgGIN0CEpIdxZ0MpIcbcfqgdNObdbIvQmDbETVptqTF2It41izZzBY1I5ZRp/PREOqG2DCXqeYN71rG/rvbDIfldu9YBwO4apLeBtpopWQ/jjVDtuRmC2/H0A5zkhJzZMKbMb60gKrn2eZQWs+Fp0crYCRbTEjg+7IL3B7QqEJ9am6VceN3H1t1KHc2szR91N5HpkVpe+yWzm4lsz7MbDV17bl9Sp1CAGJZSH3JAWTI6ETsZ/VY5xItzyq0dXBs0YUBE5FOM0HH1Tfzl6OLzt8cAI8G0wyT34cn9NdrOHr5WdrFA79BZps5QHOM3Hx8mvl2jb55syq7+ElgugmOHDtR7WwrZMzfL/Pt9c3j76+fPPz2xfMf5y84+f7m5qeXf/nzn19cv7z50/w9Vz9+88Ofnr/47sG//x+xag5xwIoAAA=='
    ),
    'YTHDF1': (
        'H4sIAAAAAAAC/8Wd27Nc5XHF/xWKZ2Lr6Ab4rauTdJ7ykPRLyuWiBBxAFZCIJFeFcvl/z+xev9V75iDwBdsRR/KZmT17f5e+rl79+Q8fvv3hu89ff/vhbz748L/63/75X+8+/OiDD7++f3X/2csvjzf/5d//sx4df+6efvr82SfHp1988+b1d8dnjx8dL9++e/Hm3eXl8yd3nz57+vjTy1v3r76cNx7fPb98ba5582Le+6e7y6sXb9/ef/f5tz8c96j/yG+ezG0/f/n63Q/f3x9vfv/m9bv7l68+++L1ly9ffX18eHz/7RdvXn7/7hxXz7iefPzoyZNPH1zzs/e6/9/Xr95ePvrtb88hH789//jJ09999MHl3cePHj/+5OPj3cePnnzy6GPevXvyyfOn8+7dk0+fPvG7z559rGvvnj1/dLfvfvL8+UeswZO73/3udoDvWbSrT6/X78ndzWcz8D98+JOL8LMzf/n2sy9evHr96uUXL44df/fm9/f/b+vxx48+eP80nv6l0/jqxbdv/8x5/PUj/vT9I/70cv3jZ3/HEd+s/NNHf7d5fPLxRcMfzOMy4Lf3l5/Pvrv/8uWLd/dffvbl/RcvfvgLJvTk8S8UpadPP/n5CT39uQk9+0dM6K8f+qP3Dv3uskKfPnr8Nx/6L1brp3d/akLPf2ZCT/72e3H3Jyb07Jdqy9Nn75/Q3ZOnzz65+0dp/d/A3h5Kcnigr7598eq/L4++u8zi8vLN/dcvX7+6cUZP5Yz45HREH+PI7//ns++//f0x/A8jsqvy8k9n1OXV5Z+MrL78yTz+Ny8fHK+qa97rjstleXzx+Ln87bnF5Yt1uebyc9zyuGNd7nd593gRxxePf+dhOXe5vLj85YqY+8Vx7+PmlZdvH3+PweQM4Hi/az6em2lM84XLDeahxy/HRI5h1nzGLY+La0aUM995N1NP0u2O18fwj7HPnI+x5wxuJjXLcdz8ctmxUMeVOZPuY3SXex7PO8YxD+iZVM28Y0Z7THSumdnMU4+PUg8/HjNzn9GFrjz241iy1gYcX5x/uN9clbsIEfogtYnHbBn7MdTSnhwLcjy3WotSx64du1le8nnIsZjHt2Jea4dr3ixtk0ailY155NywZiHnetbu2EDti+Y9wnF8Z0bfHenFnnlbEI45zLwjWD4/kT2docQs/DHGkhDovxwpPF7OzLRPxxOOJRnZOqY4T5j9vnzhkIX5vTT2WeYZdsytZhlHtvXlmVrN5KPY3gpvcGl3RxBHjuYWl09GdI7PRq675/Us0azvsXCMfu6Qmlaynt51iV5o1Wd9pEgdGuM8ROo4z4kZ6ezcDDdGcxqJnW2TVB2CwUaM2h2zTK2fRjSrMsquec7L+QbPmrHOY7TyPSNprcnMdcQ1+V6hx8dWaQIjbdou9mKkegZw7N7s9qztPGE0ulbXRoxT9yuGitrLih0vR0dCYiclHjWZKZRMQZU2MGYHU5I5MxxT1bJlmursCNaIfxqjlZLjsH62rOss1MjsGLbE5B0/c+OURo25GRXquQBjEhIDqefxlNECaVQElialHbI5tj4xM+3guYF5xvxK+UZ0RvJS9naenKlHjtCmNj61NBrzWPWQtLXM6iE+iQWf7dMCayMxtbPX2qDQDMf+2XPMht3+kYEfxR67u8ZyNmFuMOIiGYrAuI7ozIKkbNYxp4rmsXie2eH5jjQo0oY1cEDzSIloe3zFUP29sA7obiFtPgbR2os+1XEWdR5eMmMjFPPsGjfELZkGXs7OYUxr73ocyi35thiOtoxM2PpKKGb8OVOZ2Y9y7lLPDBILIWMkfZOXa6l3s8Jj4zWHUw/xGLsFObeXFc5Yp5+B6z0kLkMeHx2RFh9PT4neKLqmNbqBdJQUuSTtsswyq7PiWdbkYl2y5FYV3iQawWy0CjWSlYWDDwy2hEKmIbmwNGO5xiK2kTJnSrUk1Q5okkilZCXGSRGKaEdncIFnOpZx/LEigfFhxGwskkwcccG4FgUVjd4VUqaFDpu3wDJqmLK046QmIpotn6tLn7YNotyfYjesY6OvMk2Fvyc8YRrY+2BT1+kqkBo1IhSZqyT3mIg+g6BwPBrEGbIfstE2NqEgR06yFbJk2N1PkNLF+6Np49FK0RtREzGxfdwIkVZEi1xapHmHAGoWb1ZaYpDn5SPa4101nlw9IJgtLQVx6n5RolyyQKmgfESyWmuMyBSxhDSdwIHrLSeEnCPQM18ZytnwmZwsjm4hJ5/y44Gz0uaPldZ2JS5fO6m9bgVhpVh5Vk4Dls2fSzcQnXU8xIRAVBGmbnB4D01rnGkRG0jN7KwDm59pOVNIf7oUQsJMyU5hqsJePdYaebwhF5bXKctPZit1k7HEZizE+e/NWvIXZC3xMGvpm6ylfmHWojn8GVnLn5m0xHW+Yht1ZixOWbADtylLPEhZ8kcpi77w45RFkYF0Kic6SnkGxisr2WSwDldv/8yjFJcHaW2O/N4IL1Ifp/Di51r50Gy8glcy5CvhVeBgg9q34msrmEUKlPKk+qsVlQuaTEn2e7Zow4po+ay5umTJJkgYz1+EkbLOyrdizVvof+VTxjiQtQcJjgzDbLFskaT7jCCiSfpZzXwYy002hWtXiop+aAsJxfFdxNEkj2PNFCLONvrpsyrttEvhwblLEpGUbZ9lS2UuYwuDf8qeO/FGGQ5fHTVNLtfEbsXyyW0oBZI5JN0lEtOA5a5SoaAsPiENnktaRVSkvGsEuGsnkSzq7MLCK7JFRKbcX7GidtzZywgDgakAnLlNM+K+CsucJ5ZCX99AIQliGWR4VxmGhivHrUBWsi5tIMidhPvYjFIEsBEQTiJkZUaGpFYJZtGGPxwhhS4RgEFA0PLL17r6wHzEiXhMPsb2j+hjPGbghA9KyzEefWU8CD4U7mCvbcQUW6TAArs7zfrKYihIE8RWu+9ganllMVKjC+QrbDHiyuHN3hbIUJ6WNh44vevFU15cTM+RGmEjSStuQ1mnIhiBGPNLy9DIi1jl5LkrsfjYxgJmIJksAAWhN06Ey7iNEomKdXjzu8JTrIrWgYwzFiYZEK7HyEg04hQN2cmHotEKhizEmlj1xiBlG1rxIMxQ1KsIWGu/DsW/TWQs6W6A0lNEpcne9lMslKbmzQiK6LHD5lEwVcvi3cRBKSFKb6dj/TyFwSFQslpKUwqBx7j9tCKBMxWZn5weY7xWJa1Wrh9G1vpGlbxYk6ATDmjTpFVzfZ2BY94Ow1gLiLaCrsLr3vpeHlNYldvQMXsxxMFWHniuXaogAXK0B3ypQJ48pWxXT4hhrpT7RQPHoCvpkJY1o+k1P2OIFRVMkAiq5zSnlfgb0wYzbIWYs3CAbPMEQRH4Ma0ibkGxNSEkCWDkvj2yqGfjihm8cg6yEcWgQPwFNqnsTRjB5NdNsNxBaI80ZnoN0ThBA0DNtdik4OEUTIxvBQKbNIVv8V+xnumlRIQIfrWtwnnGx7XS/Eowbr0J7JPkonwzwOnAAVIA64MkJw1t6aGkClKZvLb6XQ9lFTlMWdWfSHMw6i6W3Aisx2k4iYkoiQG12reqFs1xJkRgJlugmCrAIQQvbcRlXDbZIiA+RTGp/cdjyDLJYZVFViFF68HzW8fi7dhLAK+0weiVJFlVR7NKmBooC8RqIrdab5inQLWzSV0BYKY9w5csaorPVpadrkAIqs4yoGf/KTQvVGMD4w0n5+R4M0mF00o5CFYFfoR3OB2bCR3QEgLmhAOodSWN/UBbcqNotuR0BZieNsQF0DkZoQA9GzNygJCEV7D1LmbF/khiZkJKglShIZEvywyRCEBlVe0yk1OSzCQxAAYFr6JIZ8uRiGUUiAB7oChM1hS4f2LUxMq1agipKtB8u6dkBvSqKIf0SnhX2u/wCJIvuVlKYtG7ukI0CpR68+dW3CEVlkQaD6AAoEqEDFLzv6G6l9BHY6ADdRY6C7SoBFBBQSlTU+1GgTVhl9Q1TghzAYGJIEiWR0MB1EAnVTWW'
        'h29yuAmonT4FBRxJxIk2qjBUrTJFOsHiesGZte7I0adwyZQlDFL4qx/VfmZbXczx0kw8GWtZla7igrBioPmKbux5mzTKSWEao3AMT9go5GYc7uxErAcn1Z4QVDENBZvCyG0ej6ktNDqAYOWzVFvNRTVG1wwMCcpWwJxGiIVdJWvuKlZcgYtBcLzmZYpgsj/SFnZPxUe8VxkX1Wprm2rxaUXjudqughQVOYE11BQbHKqAJxojiplPLX+i4We9TiAgEYRi39JKdxJnIFAu24Yw/wJXwd+hviI7GCovwXNK5IihpSqn05l8ggx+rpc9ccae5L9b5MIsCShStElEShrKsLR8JEIGK6RLvRdo/KnfqaHWeU2dFQOnzbubvS5A9SLto94voiuHH34M1QS7F/ZlK0wEQh4bNAXVlIXyo8xrYATgKnrWFQoRHCzLUrQW/wrTF/hSacC5KRSVR1mSIizQ1o5KOJbKtWT+CURPxLyVyF48gvEHyo6YxZkfKGCARMK4XLsYrYNrUJZSNO4YPdcYJ0ngFWBqq7oWXRofmGoUwhVl44uSG6p/Derrjc8tTNnBHCMvsymkdrW4rEtPLZMCo2eC7EHlkgwknRSolCp/6EKb1mE5ERI2781ExSciJdaCFnm/TCTaDea1iGbKw8mLGgggE1ac7CVrtroEuytZKYeGDpjBQuB8wBYgG8/ToaOmY2OtO+BZFDiRRUDwnniFbEhyITNJvNVGPjXlIqOnuptkCSdCqiAYJoQVR2uHjdrgubfsqP0CdiYAaRCstRlUZhNcdQlgphAYGKKKEeWIBXhKgWHuNCg6NmG9fAOUke4tP4QTSwEY8kINCOJ6Gzk/WL31MoBviO+SYCbDOZatqt4tFwODXTBnZXdYAEe2qwwVsNRYc6Ao+EPYKmyiiiJGl8ZwFQ5NuuEbBbBa2oKUU/wEUqPSB9wtpJj8HXg9DYh41xtguyztnQ7coOqR3hCtd7meX6AjSuUw3USUqQ+DfQPVPWt7yq3KNRe4H4q6iKsdUphnBUYK8QLDmsAE7CNZj6EKgsog/GqFjVrzMj8HrLqgJYohodAdYMqJuIK+vkYF0vQ0YccNdawAHm33zrpZksckJhRUu+sEREIouUN5hX1Rzr4NdhvZBJ0U37Ih25kFJVdLCE565rwaXg5IE0SwgitiLXMBEzQdDydnpeAvoURJeZVKEMKRtpojY+Qoq05OjzmelJTw967gUxBBC8ksWW70MB1tqACuYo9Vd6O1G3ZR7aajHTB1dH9FxesZUOHT/2oDY3PFNNtMTqoh9TgdgB9iCoy9sxSQDE+8Wf70/oHKYhqPops6/XEstik2mDOoIOdR1JmO2MZ+JQUvvEH4c++A6rm5gD0/WhVKY+3SbMt3l3k9wBKqOrseC57BY/RQZZ26K9UZ82GIYINhVG4xICidVq5RhqFAVgKZWLGYxF5M09hyLqYCspvQcep3KF86S480ZU/IuCk1zszww/JKmPK1ymZhBGiTJlmmZiyObbyWUcmEiHohZzGBvqx3nCiYSloG64s8vGu9lGyhy0eUHrEBhH/OTTb7BblUUEU+ZmcKRQNQgjx7vZ9N/Lq+uoYNU96wKcwHyoEmXCsmeIwmBj8JhY/1A0HEdlJVKEI3cCTiCGrRAcEozgxZsoW5ATyRnnKJC67Qh1mSMIfgirOyUVuBYofR+EWLc8GUOkOatuPSPlxFqe1i+SJVBsOglZz6EMtUTjGRsHwkTsKlqIKeQKtjAeEp7CBkMGBlqByCjzGjbfZYJsqe0Vf8a4yUMMQ02YOldo3ZDBCQZvkHKr+SYZmYaldJCiJjCDnE4/bSZuTpm4Sj8eRG+nFUyJPVDwiU3SmTSp21eK0Fl5Rr80rSZTHAuJOS4Ci3exYMk1v9cZEEn13LiBB4Sm2iljkkU11pfhg4wlpG0Zucs60pQoR7c1xJ0JJ/VE6BSHUSL0CfT4a1iaqueJpLfsOl+akaXrgYHjfF8HhPMbxPJo3zktwCTlwVPOu64FnbhXBVDX9P2RPau6vhZfbke6vhrjY/qIt0bV3EMfTaUEadZ4EkTOuGS9cm2MjcNCmIbgcaLmx24t5gb0iqoTKbgZvmw5NsUsO8zm/Lzn4tg7KzcKlbYEBQBWW+Ti5q6xq5UV8t3Xbn0eeWnLVwt/+8nybhLRHv9UYymtpum0qcCj6rnWTWlWDIXIV5ESbZPJAEowg3kkABHFPGYAxVG0aDxfFzkgBD8ioNv6VFEDOsRLSdoolsYuu3M5mwNzqlyLwEOjfCwSEVypY5VJWleKuJVKfUwDbBDdMy195rSwlUTbVAtUjZcrWA9MMlXopV6sERFNI3Uz55IQ2C0Wb3YRTL/SJLNlv+M2WdDsdPLoqUqa2uNzGaymUn46La+BxjV7Hb9SyTT8/2lFn9bVRpGikg3Rm3PgHkpYszDBy7Kx4uidKNRUGOjMitXNTXwH3PSiqdTeXygYI8VVOpDWmrafxRIRBwSS1HzoFDzDB1HSme0FJBKS+7E8l1rIVxAIOO23uZc6/xpkE7xW1mvNlZwB24ytqBxHz/dqZLnr7AKfU4uVz8ebpzKiE2KTZfhTNLNtJXsguExU1tq4gMjKXR4OUkrE2UamopxnMa+wUDt83QA1iEeIt0n+8GFVDiRwbSafpq+Ndculsk+X2YjbBMPEG35MsMAgtrPhlQZIEVNSUI9+8RTLvYbNPn4lxvxSjpF4KpR1uZmrfsynqLx+RdTUenlG95x4qUE2YnZTCFn27Qy7MartKBqYzwyQgCgX1URTUR8WREkSk3tk/ZWZrERSa+I9UDk8d4COVmQvLBrKtg7JpIfEX3Mp544+KwDUt2dVZstNI6EelWphsuYJztRqKhLYMLVlT9mMFlvES4Fo6rM7c7cbd+0cgHJL5TnZYX41aMpThEE4u1LW/kQmqkguHgAxqQS5F0YZoX6/wRxZPECbSr3h6ZjeKXIEB6hhPP0/LCKgVhMW21mm7GvEK03OQF2S5oLVXXoFL5LNqwXCmfwn4AAPq3XHVXeYYgrJpNAHSiHcPzaodcYVbUFBxohgIZOGnm7tGUSrrmVuZW+EajFkVXmKmNKAvVf9UYHQAknQr0Z6QtkNtOjF6rJxAH5S47hXA3RBI/YO2Mkfs4iehlThkcmTTWvBWCdAWERobeDPGssbdLXu5Ec1vRdnzGtmw4VjDKSTPsMvxWFZaFc9IW+0rr+0FgWz/iC3pVKJO4qn9DW4ylLUadoW3e0Bbzusnzhuc5teJ62POS/X4SsBmULiZSj82HbQP5kA7WV3SwqLNvAAQ4rs1FbUwFIiRnBcbnompvhxPEF/JVdt8drrUd54bkyvhqAiYP8hnh3MW2pMMdMKJ90RC2LQHuSseXt0O2grWXi2+ubcInGsRwqZATC6BSkn+45TkoGnBZm5yiIA9hzUXwG1xge8uw/grAFADBllsgCvSrZVSQ0/e6p/5xbi7wwn0uDzqi3sdGhqYfBvevBHXPDYiHTPUTIiCMu+bX9tJ8LaQAWYStnQ+ElBx3u4MfUhaXCkY3mGs5pktZCpsOVdNWgrxcSp7OOIjEziqXPbGjA+1OnuY+oNy6LwPAXOVF+/BBP41nge65ueukuk62l+0nOQyvOguD3uj42YameBgyEHTEudHt9HgHcK5huRLO36uOITxQh4nnP8Zh+n04jEUuNiJakrgBxrLouu2xVpuhOW0Wcb3V9t9/wX/WJqL3s0yCVWqYVUnHkZBZ2pVJ1Muq4FMkuPX6n4qTtqf2Z844AW4ACQoLXZpg54YjOVuzhcoE0nSzEuXECFd8naMtiloiWaQ9mrM+F603lwh3lFN0DZgmakFY2BEfXCb2uLAXa+vM3HVLxyY3ZcMuvjPoNj56oRgBrZsiUO/cgGSrg2tSpBDlxXFi4+hkIqtNjahSm0zYXScSbM9gkqEBiTAL2Kh9upuThuu47gKLJVM6Johtj1fcYSI9zTCL8dGdoYI/fEBqR+EjHVwVodDA4SAq'
        'vrVxJHtEaLq5DtsNBvZFDbfKOHapZENd3vw1OsLxMdVnzWijid7bqzTrChV0kmsyqrjG7hkDpGwfG7O0Bdgrsb1oHGtCKSCcrZp6JZqsw/I0bN60EwbHCARJhmuQcfbXqMsA4kvSS9pnbg5HSpW0VTz8xpb+zS0r8LalVsXZlpo+NcZ9ZbfNQw+ySbel2k/WFR/7Z9thbiDTB71DDfBRu6lp7OoKQM0fm3AC4WsoPU53nb5+AVQRbG8AVBYQisQ1cTyuYrg0kR9Y2ZKS+6s754k/crtNM7ea7jNQ1jWHVcQ9Gj4zZ3kcBMJlAoMz4Dz7a8h5miYz/Za0d+UJDORZT053WW/VNZa4H06RMXIUv9yx4hIrwFb1eV6C3NF5clCtVWt6fCAjG08ot2IoDDF8kj72RLM54VxaJvrEc5cXRD+cDzYIs+9yD9KA41+nLWrrIJQ2nwaG00yff4N5iBt6+3bi9bpvOQAff0BnDSsEeK7f9hCXk5963rE4dkTzMCku6FeNdG+LlN6HztBUD9S1i+3bNvUSRX7UMms5BCfXb5u3zoam2tem+8UCr0t4J9WAjLSF94CQ6MgIsmaXDXCc1ON2GgoR05a218cHXGH6WNzpYdq+T0yANkcUBxUv+AxmpgACr5UTiIYhgsxveT191JN5IrqFCJs+OcS0xTZRXayMckOVCcUaw5IsBct27KFi3QvPpBnTuh999+4pLbPhZIYbMmRv8BU+UqLhli/ED2rm5iZ6SZoTgwT7h88gcfDuPm2CwHat2McgUVpvH3sU57EuNAkttEkDrxmg2VvDkMT3WnNZsSV8tL2povCzZF0erPIOE6soihiNOam1cZ56or4F2t3Nzg4XRM0E6JMEpKVrk6+dP8PW8gEUZoGYr22cf+P4bbxxWAmngMZKc3jcA1U4LziG5rwHTfPAlEs7gIfarvxfqY5KlhylgZRC/aCPnZMO3aZ+8jJNx0+4NHJJUJxji7Zi38OkKt8FpwiJ2yc5bEbd9FHUnnVn29Bx1qg5JIeufjfdMzYa8wFtI66ZsoJqrMPWSx8dw9k9nHTjfiYOLpEjLzNnRMXdHtWk6QNv7f7k5ECGbalxt0XsCVTt5hc3/3PgjJlTfI+uLouQufxBoXTzZS/LFdJo+EH8GBE00iNL9Y4UpybpCBzabzhXRa8aQhx9UwWFr7bC1lT7zlCDan3k2dNzcupYmDSHmC5bfeKO0kKCtmaZFXvIHUeYmbRa8H+gVRdSVybwUVSmHEg1SZk8C3DWCpPzMYK9rXMnFQPuT58R9LYWiQVToNjiyRUN6wxV9ByyuVIPV7qV4OYBLszy+74PeQ4uqsZf2EX13pNfI0FFY09aDkAd2BPNgL5L+jSjXSTi+B3wlObETaPSHGEA0WONw1lZhZcBW8odRdpcFiB3BxtfZ09fNltdC/SYUnJGyWbG6gPOXJNky9adVdPOrpOFDOkjCV/EnDKeL/DLaKJWBb9HW0f5fKZOC7grEPqgFuwTxs9VbqgshG/VQh1nHKbWTF6mQb/k0uxdzHMD27YVrL46O1OIBMzK8vQ+wQ0D7F4T59nTQk3A0GphTajNTcNoi99OOTc1nioX7t2rZeIqIKnoMzfvPo+Yo9ndmL7B+jZmv5A4OU6adZfbW0w1UjTnFeY0wYjSOqd7cC7WHp/mA6FYIDast1+a6MlEI+q/mx7pyF3XIUW9B5u4ObKCYKfMBeQhl68fZzR/df/ui2/uv/zsK/4/Db559+77t7/59a/f3L9996vjfOrvPv/2V6/ffP3hH/8PFpPQOSthAAA='
    ),
    'MTG2': (
        'H4sIAAAAAAAC/91d244c15H8FYHPgt33i98SuYt82n1Y55shEBQ5kghTpJakgRUM//tWZUTkqe7pGZo2L7K5MjlT3V19Tp68RkbW/vXJu19//v7Nqyd/+ObJf2Vsnnz7zZMf717fPX35Yr70n//9x1jNf9ar9fq0nl99/tPbNz/Pr21W86/v3j97+3769bBZn7ar1X66dPf6RV3YrLb747re8/ZZXZt/efbu3d3P37/6db5F/I//tD3Nt/n+5Zv3v/5yN1/85e2b93cvXz99/ubFy9c/zi/OH3/3/O3LX96PZWUta3tcnTbbq/c8eq+7/3vz+t300p/+hBVvTt9y7fvjd99+M189b3frQ109bw+bna7uT6sNru7Pu7OuHk97Xj2eD32H02G7w9XT9LquntfrM66eN/q2zWp1OG2/pbgOp+++u9zMpXzn1S5fXYr6cPlabfKvT64EtjscztvtoxJ/6v/xx6ev37x/+uLuh5ev717Mb3757unzZ6/fvH75/NmsKj88e/Xu7pYwz48Jc7fab7/77m/ffnN/Vcf1dr/5XKvabT7VEZ82h4d2cDhOJnK1g2lp7+6m/57+fPfi5bP3dy+mxT9/9uvHLH33zy39H9PDzXHS5JvbHPb2qJFd7e7927/c/UuY3s2T3Z02+93HbvmhA92v7+35dDqeV7h6Oq32X1AS6+3qIXucZLP/LPZ43u63639uXx86y9Vxv37ISg+r8/lz7et8+oKau9qdudvNavfQbte79frz7Pa4We8/sIOHff358Hl8/fT9x9PjEjxt++r50BI8HB5Y62Sbu9PnWOvsZLf8/s1pt9W5blen3Yc80um03u4+mUfa7G944e169zW88MOBp/a8/4x7vvDCp/XXjkeb4/TehyVx+JySOJxp29NPx/NXl8T+9JgdHL+QTpzXh68uid12/4gkTp9VEvsts7Vp7avfps2s18f1I/I5fxr57E9HymcK+Juvv+eHosR5u96vP5GfeCBDXXfGsznsv7p1HFbbRyRx/PSSWGj8b0oSk06cH5DEVMscP2/msNjz8fQlo8R598iev1TmcN6vfsMeYZbEp/EI5/Vqtf4tafzxgWzpfF6dDqdPjststv8kdLHbb7n5/eb4ZVPpEsn5E6nBcbv9OqnQ7naoX6/Oq/1u/aVc3Off87IgPD62580nV/LVP7T5/fYzWfjhkc1v/9UP/LT9iDi+Xteuv+aeD9sHPO56vVnvDv9GBjgdSONEx9UDe95u18fVv9Ger3HxuSn1w6tnr/88N/GmHU+/vr378eWb1xf9qQ36f3xl9KZ2bAPe/e/TX179Zd7qEwtPt+k/9+mfNIvIzMB/Nr9gbpE2vyumd9R7dG1+eXpT3SGcb/Xp89Ov038RuuP00/Tq9Ov0Vk/+FPUFH3zX/HXzX9Ol6dr0rVjx/MmcPxLzUs3qbtN75k9kbWFaTG0mp2vTp6Yr0wXHt84vTb/Mu55u67WlrC3Vl00Libowf3iWC/+undctbd5vfRT3r03XEuadzLfBGkqQ+DkgMwgOYqzb1cbSg2/En5Jk7W6+67Rqy7pnQLpeW6w9lbggmtpEzOvBlixx+3oJC6hbz+ssoTmkz1vNZ4Y9R23AamHTP5MwpjXUzupQovRlXt98Ul6SrJt7yXd+3Uu4dfdZxjG/d35bHYGV9ObTmX+fX5jVYN7m9LPjjtOt5x+zXp2vWV2Yt14XM/HOOonEaub7O++HS6WguGFpdq1rflOtbf6aeketqNYQpWfUjYBSlpiwplpWnVjUSrB1D2zD68txN77X68b1O7583kvWPrAH/IyVJYQMM5jPv/QZAih1cJ5P3RjCKblAslp53SxKxlYCLwGUBuAu81fNnzWuCq+VjXjputenAyoTkF4JYPFPfRWFxjvXvSGs2kEJWIqC9+PzaTxjSN34Fb3WWWfrcupq4m6QG/4te4fI69goZa4CrgFn4zoqKVpQ03gADhFDo6xkjsOqryhlq+VDWYxm7fAB+HQENXM2jbKqkorXidH0aomzZcK9zjssz1DrmBWDKy1vZgtvQ+uFi4lyIfAkpQZwI/Nby2jxWn2wlmHZb6AfKG9UvgSmUEdkOIFS0XJ39cHZW0Dj5y16QrtKUHAoOLpyEh70hE5niQsw0SxLoi8Jmmnd3WCicM+lDfNGaZwVhrDlkmzCWOtGZQfJbeBYQ2EsjdcCpwFtxH7kRrl7+AX4C8NXJV2/wa+VGlCXZhfLFZWDpYcrL10LLz2qI7XyW4ktBQJe0v/hyxK+AncoXcE9a01QsAq69JZlCCWZ8s3lrc0zZf/1W0fB0jmE7VmbDW6mHHi9nwGI4iu1rIBRMdNpmU6b8VJrnFK5MMR9bK6WZbDD0pv5vQiDiGAVVB0rCSYKZTEGE0YIm7dkOFssDb494PYCmh9Xq6iP8fDLW4TrqOHlShx0/wwCQdErrpUvNcSV5DINwQQKPr8Lts246qWAXKYp1EJyFZ1Lq3FSWatiLmLUVStrQZxV9KQqzkdC40akScqN0Z3WgGBe28ertfTScxozboy9Jnwkjdzgh1pwMGrchJKCfJCGIApAsjjXeZMpT82gjPDs+EStFEaMvAEOAyJCShKKWjC7uoHRH0B8+MKE+dYdIYb5N9peeSl4d4fhMGOpqMMcA+mVHO7IvuiS4FUTDjul7vcUTeqOMEZFg/1j7cjY4OlvKLwvFB4iYi7dCp/BLETLYAyxxTKQil/oe3AjtUIEReMr0HdrfUdOXVHLEPtKaWAhnki/uczkgZrJVUTrOk91GGbljp4X8rLhHpLmV0uie7B2D/64e8i46R4yTT5KZqF069I7dPJ36R28pcVF2pV3SC4yg97T2jvUt154h6AoKaDF3xf/LMJz/U+/V0DG34HXe4OmDXZKcbFBGm6lcpV4Gj15xlCHCOXdiw3GYoNlA5cbZHiib6ILXPhAY03GWJNhHagrhUF8QvLg3BzTq4pU9TISA3w1QlNZLz2Yq2Sh0TqCSddf0kQmO8hJ+MH66uhyDD8icYfjCB1KeR0UfkjmUXPyZqjdUEjRIyElCHmqCrFMdHkb5Bhw9ZAWggEdF6MLfDZNINptIsmq2FDFabIIStTDTFjgglEE0JR4ckrfLLFmCELVouPDiKWMgvWNMmB8Ub2AtbGGdnkKHEgyapaBQrKM5HLJicSQ8YBrHcmjijGnN0aYxHfXcWWiwGZtCVSBWmHJxCLx/db1ahCcUBk7Bx1oO+wbRoFgTAzCWHjLvXaCXE4FVVlC2QgosOBO/s1wXd8IcQMQMMYbEyiBDEARfb4p3UuyFEGZAFkmNLn/RN77E8jmWJF0/eE8XFVvOHYAK8rlqOqK2tSqqlwtc8Rzpo8eIx0rkZQKV8UIPaDzpliwzbJTpOySHUJP0FxdYT0BIzkXCXUoBfGGGrDMCtsu2AjLYHBzLMdbG/kupsNhyuKcGskKiCU3dAwe1Sl3B0CU8j5BjakPwJe4JbMseG+mmc5blFNIVyicj8INNwN0lMDdkM3T57NKr5ug9AAYksztHJkPlAWlRcqqFLiidYb4iCmPg4mHsW6FGdPuDeZiDDT0EdA1Z0iB45az4Ncx6aoSjSdKaaWihOEOOJPykCUP2irsRWhZyaxeYjID6QauwvSGKbJaEpRVi8CW6VPg9NNkBYYzdho+PYgcIGoyGH+kFKKyR6RTydIdeA3ra6SFKFEp9mSGDTkx/4fHSSO8BsNCQQ8/nHUPYX0pQQNZrTsQNEjmTgIbGSSxA+C3EKzrZ2eehGQ1larBRqBjzqgM60Aymu2ugFoBmGDG3nVVEr6oCK+smNk9cBNgtQEtBzxKu1qEFpbfyBCSVoYF0mTo6YzigX6zFgKoBTce+JCCYsILB8MVc2O6ZqDNOGueLIIGAIVQYhk8WYfrwJfCar2hWOOX6jzKYGIUr3E/Pb0qXoVUX6SntSnmJa4cmsXrzfQ0L9JTu/dnscyMgXML+QlB4yacJakpFCWFBcAOhm2QJSFVBEVa'
        'dCioQuIONacFGpNQWCYthJhZAfYsBZAg0TEg5peTrp0zFvHLGQ3tIhq2E+fBJIWn9I3ldOmQyX0xCjXWTGwiVV2aqg9XfOPxlvp3ULzxB3mJC/Et0ILHoSieiogNLTlRvBGmeb3e51fxUcU8OxxMVRn+mGNZNxKsDzHRqkAGg1CJSgE/wiB4BzljJFNSrmigMdmZ6ERGrRUpAOHn0X4CMIQ3eHTi5+gtUfW4MIYzZIhVDuBnRkbklNArpxAWOT4RgJHFVchAdti7rSKCCmsKrhAR6wO6EXo1G4lydococdYRBOaIjQzQYVlE5xJ0EMZG3Y5GtxaFc+UWSRi3y+YYRangBqgr/AgKL3olX3glu+eVot0bZBWCDVVy00dCgQgwNaRGfOWqnrznlxpjiPb50GvKsOGikdwG+5LOuoP5GDFZOuCg1Si5dVsktzbsGRqGfKPcNezZ254V1pHWEsJA9i27peBgzmqPBu2McaZ6Ym23ToxmabdMLWDNYwWMaax9oB/pAn2RFRBBN4HjyDA9Hy8gXC4ThrlwmajK3JYFhOpT7MMJ/xuBn3v1A9Ocq8QddcM9EXf9NeoHX9YPTnB0IWfr+gF9P/TMHK2uLmI+UD24qocWGcPQKBLVeIa5Wwi7SGYjGcI3kZERi7/we0oGGTvlKZEmMaFWKlcpfzYokADQkJOlynX8UtaoMlDtTuC3yBKQfKHhrVI81bk1lsNYMvyMufAUJ2yEAh94bik3t0jBM3VjfDE2fVhUwaAEvyDWogHTfe9K70kiAD7GKroZAZ1tpz7Gdzn1LhrXZA3IeGlS6lSSW8WUEnVHqkWTY7IYzHLZbhO4iILDoHOowrori7qlBENKADIHj64zOudBA7fcDTtaToDPVZUAOW6cOFFiVX4GcMVsoOSdH5h+NgBBBFTgOtAO8xT64yhdGBvDBEOwe4qcie17dHC7+Ge3NSIWzYSKCsTejapAIVRaVKkMsFx2ftQ+YW0eAynNDjQ8LvX5QrAi28bJzjObrwTlmkTBTo7BHyDMqBOcakGzg9WdIiekib4vIbNUX5Lq4N12dfVpuSosiAVqCo43ulsWGEhXCRnKazP3CfoqVspCyJxCIv6IE4+B43iI++LsZMC/mzrsSS2nn4cCsPXAPJRdJqKNLtwngZRULeL0c+opUQecOQnaJE4PYiR0gMug/oESuHJOYq7wzcYeRTIpDVRbrL2ZFbM9ogYrj9uVKY4UXumMGX0v2iipWCd7QwZVn2DwYj6L8ivV6RzJqUAFZxFCpBeZlcpbE5oeshK1cyPVSwKjgSakVmd0DAkxTpxhmuwXqCK/p7AgmSKpTeondK+cqiD1p2GayanTURu1CjD1LVvpei6ZugI40vZiQG3N3rBQV5j4S8eclMRi1N28BQJ0sJoU9gs8n8iZETK3foW5hvoKjIJKpbqvRP5L0CcD0GZZT35bsDtAaE+FC64i6FCbmW5Td5HAgYAVy4IaHVVny5+8L4KQyD/Qqwx6YqdyoVJDow39HaHbPDoxa7ABQp3EHrWhVIeyYm/z36LLJILYuG9nAq4OAyRKtAdmYsTS2dtmv4w9EZOqePc2mxhEXhVcJZSGYZ7AHYMHeg9ceGQzUZyJsvEEmcg4i2ACeqgyYQ7GnCgEddBRgwKJYOJG1ArlpC3KhFA+J3BXsSj0tzvQ4ZFs1K2xZELdZEeabsjMNJA8ODfuDBDKCKx7G2wcQa9c9AY4QCO4Ym2uDAbs1FTd6PokD9UIZHqjuRcfVSnC5pd7I3naXzoJjDp5hB8qNzMIJzSOSI+9k3XDrJtIBY6EJa9bt9mjYX1CrLY8XQYDZLhsZzl7DQSZG7EWnrHsPiW/Ck0ZJ1o4GFhqAKkddgm+Al+IYaPdQhC+7UwaFHqJ9LoprSXOakrpyQxKbqG+3xW78L1iXjgqcYAliPDe1Ea1GhmlrLHDEL9voNhJxqbJV/moWkZF4KTJKu+gAybhkn6OZc+o7Ehog9Oi3sB/mORHEhgs4TIe4I7gFYSRU8MAmIOLEEEfJ8dhqv6RJiHVVXABpbR9lZAB5LViXgY004OlJfJltVxFgG3Ekwg4MzqChIQ7XLmfskd2871bNHKx1iyQKvlCb0hhR2xqJVvmatqqGyJ6jSuVZY9N5CKxK+CaUYcgM2YOKVk3myNE1bQQzRJgFfws+os2hKcSh+U1+ipq5aNT1hQIlYvW3t0WPVYWfJYktJDv6kvQjFiduwqmZp7YSKaaIrOgpnl8DPdk8afN29o7XBI/GJ1BQgRUDWYuYcJQQydYYTamm9GUclBKCBGlUlHWBFgaieH0RUChyRI3lcEEJVhNdD4ulieMk/AXYFF2yCyaceVB4CvkrELsIFFPgC9QCYZeu2qPZqE7gV5miEUIMt6wWRyu1iWxCpGtkuzlSmsgDfkXpjhsiIEoCljXaUxQbxO/ioQN2g2B3RgOd/lv9kvWIFUMUDUHMKwLrl9iqA37LKwJDGizd3OiSXuukMYkOJQ/1/GqQScTD/Y8xByV2F3ocjN/VGupdsfhjcw/mnnNXEtTDUmeWFJ11UQ3qhGLfPZJgAcxeVQe1nxOgc8LVmlc0exGEtDoxgWr1Np083GS3TWrdKyCXXtf0KqWjbm0BxpzdsEqRf5xzSolxBz5Ad7YINnRq/Qy2Z8gIZ/scBkuySa0F190X4Kqw0kPJ6eezsG8m5xGQ2zaEH0K4xqAcmOhpsZf5Y+UeFCYAszQvjfO7ahNA9Ll6P2Qt8mmF+A7IzOTSmIcA+pyi/1iJD3RZBp1a3rvwhSEntjiE+oyGVnYqeRSxCEnuyqFWoqIBkwSnpXIAkke3eQB/ONiBUTz9QsNt2YoZBKob/KQWnyG1pJ4HeaDBwPsHfwtz6af+Jh+AoovrEVFuYgvOHpRmzBmlfQ5xiAuhgwONTrIN0c32D6TjnuPeKgMAhFr4Dc0DsZpg9uyrjBIn1Xd6tZAo6iaJhSDkgaeTO8ps+aXpomgPHxLKjGIJYX3AgUbjS0mBs6E/JqUCsQ5mX6yxXsP1b9B01K7NiR6aZMNjWw2q7P8BPKqGZ+MC7pA88pJpaFxc8nE7DC7xzb3YAiw5IA6dTcOPV5Uh3mV6UQHMyK27EIN7zkSsBuTAXbtw6MTMH8wAQslYNkJ2ODQ9n+IOs39BegBOG7hwwlEESYn8DXOmnz2a+avO/g3oQpn4cFdcwEoxCU2u3Db/Ev/o6oudNW6+6DJPuvr6ddzD9kNnHt09AdkbB+R5HaktMtIeYNjzdQ7rznWdpPEIgpyNgW5OdZxwbFGJqZYSZoFJzCWjHRf9ouzRzHGP73spTDBOAImCKyZus36X4M193WaRJhbnfgL3r1fFRX3CO1JcOn2vEtcuIDluIvd43tfs+pJpm9hS60JB1yMu9gtQnuKuRk3xl3udeaHzMyXHEN5QOugHwLIVXz0WIqL/u2cGiRWZ65Zv+zph/Qeg3A2NFBSixzoYzSZ02uc+WsOe7c8aYt8L3t6aHu2RsUFUUKoBznN4gYv2OMR3ahtJ+Uin6nGMYIbHNsE7u1keziGidTvNhaGHOZlLiMw2pkykLtUJD7jvEAz5m0UkZr5SuGN4myorOIhEAwnG1StdmY51mGd4YBM0MskWmgechcnujSiNwEUJgpiDKjnozpC9KtoQi9hK++ehsBUIksKSF23KsMcaAEkGJoEHFQxpt6YCYagCSHlkIS8r9KPsosQZZTj1EqYu+RTj4drr79ATfR2ZOr0SVAcX2fjkQAogbAxZADaNup6aroGNV3RUvxfEvzCu/tQvQ0jChHiMHYiQmg71O5n77WSZpfkZLBqGAs6cHXT2aoahaQ8f6i2j24wsleItr961GIWknQX1OSuJuDQSS0ZraQeO3P2k9krIvaRAiOofaymvVtXtijpkjTn0NxChpqZpln+ZAuHMTI6owzO9RkJKLksUg3Xs9W5WWnKNUM+XNPl5JEvOvTN3yZe56qzrLuGhDeJjogGQn9JmsJgEakbq5kX'
        '4jAkPseikEeHVf1Njn1AIMSY2OsIcihYk3JedbA1O8ljgWsisRun1lggF4wHF8dON21B4wckLbG41SFHP35AE/9sk0hZyIWEB8d7RBJQywaIhqux5ANERkETwQcioGcY7OO4asZmDZgANWsGGnw9mUQkWalRqNxbFwexOIkJNPYE6p5IRJcZxDJx9IcAFruYy7sAWHwBsORDaeMVwDKyxs7OQ2D+/ex8MBLyHsLSeLjdRFg+PJlnpvGLG+TnhdgEPIgud5Mrp5kJgQxj2iav+MVooKcMbUmWM7b7RG3K29M2doMttyAkumaobDCnHuHLLTXjgpc45m1C02WO2l6MZ7xbzLm8XpASKB+h3fpBKTYwKBePGT3VHpdWGCaPDO6duaLmqlyPf4kB7rKHQyfZkzQc474g37n1+I+6GmjfEDCEu7LRLUb4wny2kX1uUgkppYjRemyOkZFmijWkVGQ/h4GWB6oQQ4wpXwratiddqfUQlI5UdMXwZq35wNSQQFqIqsKHmgTlnf1dwplgFxdDWDhmmAubXEwtADvoWKIf/2A9bdbdIw5ljvEBPuhhIFfUpCWtgKNemUon2ZwIDUCqLediZxGgEq243Q270/2EmhxhVjNhimthVDuwNIpnzgcMOIlzzkfE8OEfXKMeFOHAoF0hScMR3W6InimhCWja0sThD+SNYrxxxt8EE/ApFj04qSekVCqGuBPeeCnJr4s+J4q+1BCuQp7wrhCBOcRS0OwZCwY9zqYTvVAiRYvvkIx2U4BW4iO6wCM3H8DZSGJ85mOU+FgL+Cw+9YYDtRniwwpy53OjWBg2pux6yJELnmXN0Klc8tklPWlLGJP26pqbcw7L6ckdIIFSj5AO8FPM5PUEAE240gcULuM9TGtkpjZlKkgxdnpNE7WQ5W7tyEktaboQUigOizgqwRKh9Ao9/BjOzkykzcGEsfEsG1RHLPo0dcCBw6BSkCtD2oJrApoFC5xJ885YoIAAqZaCEj6WmHxeUdOryPbX4xU6myfKvGDXEA43UWysn+bTJCNsfsxekhinp630/Lv7QLb5VCdavffQqGveqgn5eloOMIUeRyauhJlrU04ggkKqbd2PBCKjlM/hygVnrvkPICOKEMbD4YNWmKA27B2a3AzCUsbH+aQmOTmSz4cwiO2WzWIlYhHyRzR8VXO00mBg5hOixPcj44Au3H2IrZmkfOSajw4o9EvgPkodlXPqSKQmyJuAjrDJpoh8HEKSdspglrqzKFUuNgzJg+3PQjZLtQWYZCwlwLLTxE4O+Mq1LZpwECaLJYuVDCW6Vqa8Rnb8eNhCXMH3olRoquwyy7W4hOv8YpTGb/zP1IulD7xBoHikB5ujB2vLEiEve7DEov7OEuGiUlHIfng60m5NR2qZcbsJa7eF56PzEQ+WCMtTtM6sOH4V1uAIyb+qcUNkB5daeT9rAyuDS0uRpMN7Uks9N5qp60EQYuBoPkDsCOKs1rwSTTczERPplScfBFlAw4H59vCesadlwnDJY2RSSrjAvcf5czGnTaCRoI0p2OuBRGp+Bb0DYV49BUwPzuLzxZKkVeMAbz91Q8DHCPIY+KVTcZm5N8trAD2ZeohkWKcXfFKIaFs87Gwum3BckwdraxapngyyfrTK6H+R5Kfnf9CTmFhmKXajC1bh9AhpmCKPNQdAz+oR+d17qCrQdQfFN4Vb9EM2stk/OsPBX9F8ENmMpgaykTUPbJrPriEEkmraCyJelKAq15r3yRBG4otKNwGQevKoaxpTQ75IHWwxBO0a9MXfenqn0cmYN+NZz1DkOBzx81gQsdk3J/TJakATSiLpavabLoNB1zAoZv0mpEN6ppf3SAJJ5hoUYI+CHPIe2jUf86CkyAmm6QEEZtokNdBmnMWmole0ASDkjkl/8agHNQ88aLYYCBv4aF33k0r66T+COxjcNdB/H+64ng1MDf/dfrhIPgx3KIO+eLZIPDAbaPZBrMN7NhDOWM+h6fH1MR3oWoUvpwMztAxXN1Kj5B/7f/b3/F/eApquB9krRrBev57KtCvJC0RGGaHsW+HGoudpvOdHiK5On5ufmvzD3fvnP929ePoD//+O/vT+/S/v/vD737+9e/f+d/MTzn/+/tXv3rz98cnf/h/L1q3SzXQAAA=='
    ),
    'TMPRSS3': (
        'H4sIAAAAAAAC/+2dW69cx42F/4rh50xy7pe8EZwBn2YwiPkWBIJsH9vC2JJHUoAJgvz3ObvWt1gtWbKRxE7sQI4dSX26d9euzSIXFxepP3/86k/ffPri649/+9HH/Z///btPPrn8+Fcfffzlw/OHJ88+P179j//6pM6Of85vzs7v1k8/++rli2+On12cH3989frpy9ePf7y6uLw9v7s6e3zp4fnneuH+5vziYr3n5dP12r+dP/7p6atXD998+vWfjmvU7/Kry7vjOp8+e/H6T98+HC9++/LF64dnz5989uLzZ8+/PH54fP7VZy+ffft6r6vXum6uri7vrt56z/de6+H/Xjx/9fij3/9eS75/XPLxu4vb2/s//Oqj9er17fmNXr2+u7nzqzfXV+d69ebm7pJX787Oz2/Xq3dnF+cXfvXi7OZar15cXF751cuzy0u9enl+fzev3t9qDY/bdz7Xvb46u9er19f7Co9v4Qp31xc38+r9Be+9P7u69qv39xda2f3Z2eUtr95fX15c6dXrq7m3++v7qwu9enN2ffGHP7y5oW8+5GPHTn968rwfP/vGz9ZG//njtx7a1eXjF97+4FN/9urJZ0+fv3j+7LOnh4l+8fTrVw/veIBXf9sDvPnXfoB/+dVH393325u7q7dP28uH10+fPX/4/Mmz569fvnj+V2z8T31yHq9x/r4bubq+v/leA3qS//7Jk+cvXj/5/OGL4/Z++vv6uRrU+fnde3bx7uL25vyfbQ6/rHN49b7NvLx/vOL1P9mn/dKC0vl3fdrN2dntu3d4h/q/Zodfv/zjw4eo/3cEjZvri6vzD4b99+/7zfW79v0RIb/XoTwu9UfZ97vzu9ubX6hdvnd/rm4fjeXsJ8EAx37dfXe/Lm7ubWW315fvXtXd/c3NxfnP8rRc/wt4qfdaw9r3i59lSvGvv++Xv8hU7j07zHV/Xjt89YuMv7+kHb7+QEf8oxHO2vebD5b9E+/w7Qfs/uN6lHO/en/zfft+92Hf/959fz8ZsHb4/oPP/ml3+PLsww7/M3zH5fmHTObHjZU/vO/314+bdPEhc/9HY5S175cf/MwP7Pvfnt+cn5+fXVz9OBnk/eXl3fnfuorL27vLH2kVd+zb/fXZ3dWPsEOX59cXZx/w2k+aeR+agi++fvr8fx636vxx1391FBu/fPbi+RvygjNpSPjJlhbcIiV5+N8n3379x2O7P87IzM6MyuO3x//H8efu7NJrEf34C++rOv7Q9fj7ePwtnz3+ULXe59ci6/hsnnxW36HPrv91+HuDz+g7el3vcQXHG47FFGvRZ/v0e9fH3/Ed33kt12f3d6S+4/GXjuNmY9+Yb0JfGicX69MF+wtmo04+G/tL2691sVp/NvzZnk05vmItP3zZPrmJinetTwt7fAB87rjL2G84rsPF663daC1i7+TjYtdS9k0dF337BnzzvRfx5lPt2ZDHFfspeMe/YxHeuGbNcfodsZ++Xvu+6+21xF7zsrJ6x8b94EP0+97x2XcZSs4+swvHY9WF9YD12rEn63gdS+t1t+u51tr6WkaxNm4diD7ee5ylxxs//l1bu45erEOmTapjn0tXe7zq8cyX2R7fHrHM9vjdcfW1gOOV5KHrWrV2NlJmdhz6Wmf6uHTEOiTHv7reYWGldx3/risd6z2Wtu4i1jtivbWPXx/fpX2I5RmKY9zHohtrXbd0bIveH8sDrCX0utay8Mf/jhfln9b2+87XlQ9rPa6Q+s3ay8Cb1fpI5bruemntytri0u96vb3Wva9nuQxpHeLD0OToQpt43HJ4jyu0gFxrKe3r2r9135WygeNe1tNdSzqusn5dS47lgvSJKBaRa9+K9a3TvGzi2Nl1fjgfem+tp1DLKNYO+0aKfVseJGyV67mWNnzdR2OkoSeQ+rXXoo/3sYxa36AvZhnr0MteUgsP7UkvF1jaGv3ci9WTKmLH4cI4WWHfsB5ahBzfciU6+HJxyy5Pfll3tr7ruNla+yxr1TMJXOTa+XUTchWYxbEuPEwpHOmeddbX3S+Hs8z2sJKqIIysOwpdc32i1/4vRz9XlYUdxrg2DX9HrDv+wZsdj6DXOfKtLQs4Tp/fh7dc7tPBbD3L5QFkPn5GibWmDWVdp1jqcuHHiZQBJntXHMj0G/WQfWQwPmxNCy2H8OW41vdm1g6yy0+uK8qbtRauOzlO+LLC3l593cDhIRUSw2deDlTuj6O4dmXZsPDL8cB9JG0HuBWds2WBy/hLRyr8wfSjZWXrHkIrLrmk5Jkojh/btfw6YV3Hf7mNsIOubW1yCdrewD/I1dlLtbZ+7cjyVWvJa0k9LmX5hOaoHVuqHe7U/WDTepZ6UKk9OA637Al79glc4ULPM3UY9NC11UWQUugFFjWoIYJDIGfJ4SEmrx8td7u2EEsMhSctdoUwLqYNaOGkdZF1DZ1i+YVQEJEtHT9pueoVSBSsZDCh3SEIrD0Nu8GW248m3ModAThKAb3WiS7gIQ4Z4INHX6dxfUBPWDd03ENh8YezNgbR3a3fC5rIp68HLTuVR67g2IX9/noMy3iSZ4dRtfdpYvsySfnyBC/qiGdgfwr0lTKRtUF6bsuQEriwAtNaiOy7A4jPVYssYNxEAGrWaktHZPnrauJx6EeyyVi4SR6oDXvTvioxleNBFxCXl7DTTswwsCiOaIXPEMBnLUk2UXxX681rBwEGYK7jPjt86A4AIC8H+FzbFAu4yUjwjnxhyYFypvTHxDuE3pDrIJFWcID0hLXX7Koj5FpgjhdTtF6wIDf+LftjnjYbpccb2ohWHG9lUK14uQJns3FruSHvJ1RWmWP0yn2EKXr2ERwhH5xCC8KO2XjlIJdbLwrRAWeFFcZVkeFwnsZ7K+SsX+RfCvjkeG5f2HKPcu/CoaXoW4TRnoDu/V/bCVYwcpXbbMWmVtTQ+4TYl4Ws2+Q92i45wACHhhK/FRyEdksPAEy7MoTiG7Txxwp0ZRCZopv+WY/NH8DRhRLkxswIGGMphSEAEjlpbHtiDIKd63zIt6wnyEFa53ed6faS5I1BvAAdHGoQMdatNMFzeRQBe7Dtii7sr5GF8g/+NKCSKNEOmN1OEpvcNI2e0knhsnudc2VKIMk0zh8f0hxsIowCe27kk4IB8jfpX8OcSA1Q5Gu0vUo97BLWdrWd1cKwZeQmdLhhlJCSHAmPbPyroNNeqd7VCsg1oKXwJWvZoaXZLxXRor0Z3kjt3jrTzjlOtzoHGuj9SkDG80Xas/H8FM4UFpXNyvh0s2urBFzwVGUSIIZaUibTOrZQIeuHOrKg4A6nYJxY/Jqemg6kEgWsSgmzfBOYXcbomyC5AO6XmRHhNDtf7bRohQlFY85gseOqpOr6trXlEAM2NVv1gn4VGJehGNhhULhy+1JeXcqvuwnPEDjCtCtTWm6AmMkbmoNnzLjeIqpAnpNz0WQSAC9wo6JNQ0QIiQAGEtgexGOfCdIieELOmtY8ZyQxFN6l68SghS7TM8owhH3Z0wIX67yHs9oyNtem2Q7XG+ASZHccDZLOIi0MArsQqbgGHUD8FVTFG3+cXFh4WJTosuEiP9TO8K3YgjEBCE2HKMNcZNVCuuavdOMYQSguiryELQqF5LQjE1GwToWisdwiCJT0pJVis1F+cHPkzEYWTtREV6VxKfsemGuwU5AlAirlmwtvCHG6zTKG8LZ3UgbQOlkFk8TZLsd8iOKYFBxKS/ggCGPNE8DKKiC1FCnWDbPJRCd4OHsWnT1ZPHmYb2FzpTJgyIqcJzTGWAl7pTyKXGa8fEGfEUSVawGEBRSUByhMyD577rMAimsnw+ljCOzIU9T827oAcLlI1YaBM2KsybqEDRa+zvECTlKU8iloxE5w+M9c20JP2qhFxRD4AtQjd7GWIaCTfLniio8w2YWcOysNrQQUtPxKNI8LRNwbQEaXAUyTVIusWyiNCCSyTI5KID75TTiowceSPyv4iTjav8qBQDcsDA/zl2KaKgykiNsB'
        'XlGWXDowOEJDWR41ALq3o8WVesE7wwgQapLNJ6kxjIpvX45S+HAFBTYr0lFEPBJ1DXGg2ihlm22TGjp6pY9yCHB95TsEzcju9LCa+NNUoHDxzdLMTIipJbMwH8hWhFOLSWvgY7Qi7pzMCu5EG3VcVO/XWWjYjQROCjcsQBpkv2K4fBDCS7DZCdc4V9MJsqMSkQxKabM1Y1EAH4VxHRKRYcuHcJMB2nSIxTQns9ncIominiDBNey04MQVu0o/4xMJZIc4FSUN5uihfR3GNnZpLqmEilx1CDOOS5powe8qAef/o8kHq05yJZ0pmSosUhLuOS0U/mI+rQBaKj3AYtnXCt6LvjHAcoGvcfoF4ujwXZv/VMRYOVFPiiTqNMzIjzUfVLp4gyntTYEm5M0orAz9RfjTNy9PHn7oKsYoOJJUNGAXIw8dO9JLwrKyxjqpFol3hA1jf+3+hC/TLLfh7/CNSvgg/sNcBNWpaidzodIHAB/IGKY9q2yfPahG0EHLcUAk5wcGBLYPpBn+d/8fZHCBwxrCDiLMR6gFPwTKWginT6Cg4Bs2SX0r2rmgz5gCagKfgzxKhZuYxb3vP9ysiZHTZcLVDf/BMvt0mRR5AQmCQwKF7dq6yCGtNGalxE/YAZZNojrLDooNjWWUi61ig+zMm5fRDShcE5h5TOWz2jz/fnMdKiFM8cew7GQ1Pb95azmNv+j5xFvLefPh4dJ3heMdq+mTksD7dmWWAaGkos17lrG2Ip28wLbgi/Yyhq/K/XBAnbFzmxCVqKg70RgCeAXsEM6PdLoAGl0GQFSgyOcirUyGIpm0AwRXoTNKP22lRe8jLHYaFjnkuFo30UoXLUpQ6Stxd2x2yqZd8wbRYjRARwc4IGjsEgwMcEDvhSsHKl7KX+5yO8WfiZoihjvG08PucWi0Pwt0y1WppDYFGIUAFw8W1CuoTmFd+wdeLExWZgI3Ey4YSWnhI0S0Wk9KFerY1Dp1TpUKrEHwOqVl4dFTSxD8yFOoTEA3z2UyWiFXRLij9VIfgLwsgFHsGxnGLmWkifaw1iRw1DJaB50NTprVjpBBxbpULjv1XG13iEENIGL0VLyXMbgIH2NU0a7/nAC+5FgSQLfMocr5V9t4rStQJtQQHKOu6PAth6U4QnFSOgQZOJhqWEuBuTnfCyGZ+WTPUAdMuQ//LeTJsViyH0DcyjEs3GlERWTc1C+dvQln+DQ3vqZsYVFw3mv/1i2Cd4qqh1eb8K8CG+mCTdtnKq46n8E9LuIuwgFX6+sYMsoxF1mW0nplwOhppJypMZXBW/AvYa1Uu0YopRP1+aaYb8dKma1z8m6XmoKiEnoasRmmD3iu1i5RvW6gk2BRQ68kRRfdvfhj2TgKGtXc5cXLHHMH56BNaXQaBAEgOOu9NDhFsa3g7Lx01yu9/jTGqSHQBYs2o+DsOKzOWYaAOqzNPTclQoeDYH9FOQXlmDYCiaIiQpiA2IgpLyimdW6LtA5Ru6DbacpEkB4uvwv8xCmIDpIw5zjIdOYJj9ygrFrBhEk1SQOrLWYiH8qT875Zxwks3VNu0bMWzpSHDeuFUMQ4f5EHxNmT8yYqrzYpX5sD3vFx8LKZWryrV6eaGvmaQpHRiDjVHg9plYrUEOHU9ntfI/jivik14xyoUQ2JZ1RqWenmDsPpqdJqEUFkbqo7S46kZFoAPwpn1tZ2tExc0AWGLMLFpzJyJd3ocAgRWSDc1DwySa9kChaahKtYEiGQnED0e4cazRHBjR0Iy0xF+igdQ4y1jrnVfTp/InTFqhi21ciBwtWWADcjd8l02pjsSpZVbqgUAWpoSAJipyalaeK8TkdBxyIJHHJZzq0cEJbltWVIpgVZUCK/wT2hYpE3JYPw2aIGvcvcMWy3xUUJlSiWBPcF5gC6FqRb+ciXvQpnsUeTFPZkK4fW7SpbDyvRash0itYQCY73crkU0VgSgdEVodGD2JXVhHQn9D3Capj9nCVLoYakTPyTGa4wxxEEFE6nnn7D78UWPEoCoLwpOKGDU8koYpuSzye6qhV3cGOEWwcrp4YWEZbR7WSBnASi/EDlkWQ4wUItlAMf5Z37RDBHDR9q0udUxzmNvKgXWXGzEH2OGwz/y/KiTqkNl+USBKcICyHcBLS21A/kHCrZTY4MEgF3UcvapbQAgsA217baDLNZ+m+KQIASLdY1LVGKvGK+L9GdcXgtBE5kOV27iOv6lwUgShFqS1ZUyXTdtI18xRKnQfsUdkay6Ifpm08EA72rllCJw5w4oUrnT3zWkn5+uIXw5ELkq0ZcadlSTYmd6q25mfFBAfyLLUIZfVJbIYW41AqlSo7rCVpoM5AkZEqDYOjngRElXAdpi4Rg9SMtVpvUzLdu7bIEheR8yNBxRzQCIHIRvKXW2C4b5pQbmjJT5fBMFhOLsW84P3t6KgaUPCg77TSuJrtEVskmiX0sRPlKxmvSlDZm53RQakPwXn40YcV9jvqjgLOlMEIuqTot0CTnBmoYznRW7NAQ9tD2vAGAaOVOrpPnJMRpDql9qJQHsnWYn2B+UmIbERWlX8V+i5ygqDstfpeMJS1TtEo0+S5llgTkQUF8UpR0B6IYQvqKDwHHFRB7VtfGG/9A/KOSnSoKGiZT35LL4jgXm9oWRQSVN4BwWGckSC4Wktp1hAuzs/nN8sFSkn0MTAmVYYaHiK1hdt6JdlZk9qSB4ZKgHgnIN40WhXjA8M1N94BTvycnwdfiG+oszeFhqyL7RzgUJjcIL4m0XNaFuIudHH5peEFosEBGg/GZpJGTQ3iK6JkNbgN0uTRLx9OK23LN3k6kJx1B+FP0PqBkSqnLIQBg/g1PcgpYMmCounDLRtHuUYNTZJVlnURTZjbXBJkltT0YQ8Vgn9bcYE4m1Cg+9S6ROEY1INEWpcGzxCERDsgVeg4StGmRykSShZo6rRohoIQY/L7Q7AnbtKXQzszGDbgN4O3XTtoD/LPc8pzGKTkJjzHMEw4KroQHLTA3ODMmekqX6Iie4zmgapVHtwtz096mLNcQeFpKariwgoJzqkhfoVvcRjVoU4RrF6HQDmRhnXrvRCy2+6eet70vIdYidjesNbA8rO0lSCI0QlNQPUfeKUgnlemYioILwj0Fvsrdl9eTbhjdJ2UXKt9BSdOnaB2plgtfloJoNMiDQ0oyATQyUVcqlHSFA1pMlR4lEu2Ols7YHNF8mBBvCzbS915WTltUPBw3wCWsTzOZQKZglrEGO7oEU1P0HVGOm0JMuoukQQWodWohxJgYx+f6DUK4zQjViK9FWTblHMJTItN3gc1RIs1JZ8TWkVv/sdvrHL+NBMPV7NrYwh0e9KRhONRZEpVA4o6mOp/Of9CloxZwG6wpo8RwXbNRmjpqCJWIDd/LchQzUJMVBymFa2e5z7t3mgwz7TjMN6Pk7DghUIP+OHYKxxe0vkqC5rSPsCeuxj1mhbXBoIZLo8Pq4Edyes9Qq9Sp+rDKfbSKPoooVrhlT6tSWZ5prbT4uW7TZBMGpamVCxqRLhhw5xo53Zbh8j9y1k6rx2MrUeiYcXdrjj6DvMT02NzZ9tbKDeV+mgKR221EAadYWltqj8bbjZzo+UenQI3B8USoEPVh0E+SU0BtOux0xOnEhBt+53/dtklLgSk5BH0xQAYUNDuqIuVVFb6tgwkbJtyX5QCSmIxHWGgCstc0PVjFDP6gbooPZT1p7OzU/gd+a1Ciu6bQyrvhoChkuGQynSXF4aNiTA4+ADGGojMj6rrz9OyY/4U8o1yKyiW2oKPbVHYbK0bHlhaVJXGAMhqq1V8JlNoyplEN+rMxTXXmlE18GqCQ4qWFQ1YGuvQXduSUI01BbxkwZ+Gd/4vcAKrdroqqMdpNr0SG1QtB8cmdz+2agulRmjdq+2BV2qgB13Qrba3xsHxmPms3XCj61tDvZbYjpgY3sFeomnbiCh/OcDnVPHbVaAJVXMly1ZEKZXWZa+JkZG54WHgq0S/T5zMtNdOO2+Zv1DlEFavQ27pP'
        'ceSz7n8ylLdNoHUS5out2my3TUS4MZwD6CBJ43Gd1DqTdpuCOELZoloiQBxBoJ9yuQM3B5rToEnDqCs0RbV77LBGvYAcQp5uAj2lX32j2wKMml1rmg7oGLNAMnKiUCKTM9ZzECfDVZKh/sFwVtsW0NU0uoaFKtPBYb3rdFOQ0/aMIag+yR6E/5zsNx3Moq7IVnt+vsIw9lPTQJZU5cutt8Q71BU7gegO83p+YOZirPCmGBUBoQL3pvgIQWWLET5sl8raUyyk4LeoMwduWaC+E0EJUYtSU2zEWpRhDbyjrJMhGbZwmtCZoxZvS7HdbUW836NHdg9NsJ3uPWqr58viMioU+O5RbdJTZDY1dyG3ExtatYyeVmJiT+7WdtrhczOJk8YN70LSTq+tQJq7iawpUA0GyqqAOIiKEQxTmaT93r06lmX0UPvTMhYTKtJf0O44XmRfZzknHrFkIP9Id5Wv1K2oojhLRJZBxM6C4xllEjudkJ6Rw/4Ok+gBEOF6aLpSjBtOS6tOund25pP2C26S3d1+osokKa3phcgZ2GAdgAW9wzMgVERSOQJnGt3oGJShBEKd3Xgcrn64c1U0kW3NMjakMQTwHCaOvueyEsr9dlNLqGmKQ+dr1VubjTF/6u11BOApIBlmlVUbm4dj+G7ycTmwLRgLelNTbHy8NY/G7QfuqXO32BQjekjPmK6jLM9VKKtXkN7g87DNEftP4/Bu/FGKsZlazLQA1SOZi5GdWQzmZlZwo56DrUQt3LW75wl99O+6sQD1Ct0jTdOm4nVZkOEZS5Ygj65s2lWHDqAfhqFClnIEU3CYYsJjUUQjUsCv6QjO4Kaiw9JiYornoIfp4VAjhmI+rbeunu8ymdUDLp2ZHUKBsqe2xEllnWzd4vQB3uTmDtlBzxVqstyU+/Duzm2mlXNCSTMOZzqH3cFD9lfTdr3QhQmZnhJZzEimJAvLEyZo9zyRebdRVbgmFZ4E4DEx3a5mALWFJ2siC8LvKRpRWY4t4GgpJycNGGU6jbwI02fsTlqZHq7ZmZmomrYRj1rac2/CSFuKTqopNG0GCcCIMCwRfc/cG0ZnRH937o31VSjFZ+7NLKNjGp9OUus35970zhuQzdSkaUoyh+V+e+7NbspvauxDy5zMvSmPWhrSqGLgYUxVwXXOcgdR2uFOk/Sej+IFj4Q0RrNlyixiWlxR+diNpKv2OwmksQyFlU1ydGaaqCD0Sm9w79lVsZnGdL9IeP6UOzDcFjRC/3buZdHZ9pKuKZoZszDV3avu0Q3stKeCXTOmAWRK57V4otzPeWddPTVr7aeL7Ejf6rt9AmG1uw7HsBJBN4PzH/SyIn7fVrzTB2AMgFzT6sSteLcUdbcvOILbe2qRMZMkUKXN3BQqD9BbU5GQ/tNa32DQQpv04Zy3mUkhRXoGpyZlMp4+/2pIRR5woPZu08Onn0MyahrVTwgZkcgmZH8I03eTYvSOdajISKnFs62TviXLbIqFxgj5yYvTjXuWtJgplYwTQUVZcAT12QF6MtMxyhEN+umTYUSyeOJS2VfZjmybNH82uJfxB2adTT6qX8t8Z7hTzALsqdVaiTlD7oJC8ExD2ICh3E2THrI0ypmZ9uE5K00rPfMtXEbqPJlLRlUO5t5Npv65FWCM//JQtSEqNn1/Ut6uaQ46keF6jkOPgpXshkMBnzOdOz3avEAluOWyCW9qhVHgZXoPIPTia3o5YOPbimFita4XzkxNk3qGFx9CqWiWSeMnrF802Tbk4Z5aE658TIsr1GvNOD33pznyYbPprtXd/A/kdtW9d/4/4pPRarnjs+e7xBqowkcFhATRKYGrheE0tJGel4u3FqV4Q9EmcNxRC5cbcnvqCdaqaqgDlRf34YOUR6kUe6xni7WxCIt0EBOSAlzqopzZnTh6E9luFwl3XM9teNQVWXiOwiIoWDWXOOmrbsJDz9AIa2hXphsj/beCZubFWXPeM+dotDJuaCHHnJbTGnlUUsNBX9o1dSebdbsEPYEp3EUIocDEkMaWHV1MmETmPs0BvFBcNpv+HkwH/VfDminPYX7NxnT5nlmGTI4aSeCAumQWTnqu65ugrk6HGSZTwpyp1okK5+07rdOJgN4gHW6rosu4hS1uF/9jS4jiZEwFU7Cm5T0mfLRH8Q0tT8JDxzYaE4PQ2jkfhBDDN2G2PDNzfLilKY3iJ8lJG2rbCInz4CyEVKFhZ0cdnNCzihcGx8tdGKkZKFh2nMYMKkKqMFQnuAy9lK7uYmqMjDZc+6VBJLBlJpw2neHMY6mmCXeT9+m5cwK+PKGcrFNF63Il1qm+vWfmzuHp0gkfoEm4ggoeTKIUHAyHU1eYfYUbNviVd4YpoWLSiGPsNNmGJ20wCDZIMRUfYw9LjBGgza9OuZ1KtMfJAi2DOmG7cEg3i8bYqPHJfZkxQinU82G/2ACJYjTKFLdH1Zuzf5aaezDrgGcZYiND4W5dyBjRFeU/6PAR/fcenuufs+WoCwLrc7so8T6mJ4sdUQdB5LRqdgzt5QYCng7BEPXRpkf20BraEdqST8/3olkKmsgVD9hmS2x8dRrTZ/KaZ+54Ou7J9LK2ZJ+iBE8OlneeokcB9VZyefTYMGueCWnGxdOu2/O+6JmigNfu1IoTPsLF15OBWuXW5dqjvRhGY9FmefZXzyRgZwXVU7OvmDlGq+mCKO55Ce3AZWY0PEMmBzRKYF/YBYiGcbOwvm4P8IgMuwGPjqM93uMcGVAQphbrja5bF9ybwl2bc1P2gPRFGddMY51pSR7h58HY5BZhkncqmeEy+klbj6fMuQfN0zo8p9YqR5yAH7bZkXDPZ9L/uMWFLnMmRVP6OHO0/T1S9pwWiJzJQ56QAOE8URz4MOTYzJYZye1QMNUOsBbeEakl2vWUl1FI0HyMCpWB7Gnpd3hToz1s0Qq79ARkTSGedppyZ7aBc05zbXv+amGEHqY0Ko2kQ7Ny041+WMMQAMojPBwSTUp4kGabVXFrvdXzy8v0CFwVAtg1srKwBEy0ZU3dJTxrpmcQWPRuE2SUJzcdWx7twZEgwXjXP6BNyFmE6x6JPtPy4GroSgKZB4u01BmpPLMAldsqS267PQWKQvtqOSkdRBYBtf+OADQMFaPEmqldGTP1onfWOaMzRN80faWShdAO5KqaLtE0mVt+jFRr+lIdDtqDVehMaboXGW8Q0w84J75Nlu26oaW5yHHowE112zN8Z4TX1lIyFtSDMTzEKNRRLBsjMicy5pweLTyka2aZPc3xM9gyGYaSVk/NXzpRpojoQHWefJIl554GlB6HagZ/z4mN7b/3PCIiUA2NJe8VaF73DMc3pryfEHBDueWpMHgiRJgn9YzgcOUQwXlLEJIZo0H1yF7uR9d3i/goMCav3zaDmma00eC88oz28EDX9tGMsBwVQTYlZOyKplBGteKn3JbMdL32HLzwjAzoA/MxMvs9OiwR/FghObMWGZuQM58jnSfTq003VHoSENVKfpduluzcfymE6uHjCWZIFB2gHSPiYY5HurF5I7ppOhD0Sv/1AjT1N52SnrXQ4+qtCDHpqmGySRCpGdxq0qWsBQGCzlAazxWmSSN6T0wgqJkq98x/T8a1hgnSMQDV1XtosjErZQl3ih+XOP72oC8eXn/21cPnT754+eKb46+v+er1629f/fY3v3n58Or1rx+ev3r45tOvf/3i5Zcf/+X/AQjhZTSYiQAA'
    ),
    'CDC25B': (
        'H4sIAAAAAAAC/+1d2Y5dR3L8FYHPwszdl3lL5xj55gdPvg0EgiJbEmGqSZM9gIXB/LtvZURkndvd1NBCS6Zlcrm8fdY6VblGRh7+/dmHn3789u2bZ3/66pn/2Tf7f3n29VfPvr+5vXn++tXY+K//9pdYjV/r1Xqz2Y29L394//bHsW+zGj9+uHvx/u7y4/Z4OhyPm8uWm9tx6va0Ohy3qzri/YvatL788OLDh5sfv33z07hA/Lv/sD2Ni3z7+u3dT+9uxsZ379/e3by+ff7y7avXt9+PneP0Dy/fv353NweVNajNbn8+rO4d87PXuvmvt7cfLrv++tft8XzYri/jG1+O2/U3X381th0Pm01tOx53Z2477dbYdtodtrXtdLn36fT1eMjVdnPWtt1hg22780nbjrsttp0Oe207H9e1bb0663rrzX6HbduTzl0fN7je+rTaaNt5g+M269WK2zabE6632R4P2nauZxtfjjr38rgY83az0nHb3QrX2+63R27b7Q/n2rY7nHSP3WmDexzW6/0331xP+ZQBzuly5xSHy5lXu2oh/v7s3qJud7vN/vBPpeL1h+cvX9y+vX398sUQ3+9evPlwc2+BKZCXL8f19vxlgT9xgS9a+803//j6q4crczzvz6f/Hyvz+a7C7nDcr88/uwrP/c9/eX779u75q5vvXt/evPrEVTnvj9tzzfb+dPyiL5+4UvvLTT6yUtvz6bR6Cn05Hy7Df+iqDrsvK/OLdGgGDv+Tlbl7/7ebLzHEk8UQj7qY3eq03T6Nyuy26199ZX6f6rE7rU7rw6/gYsZE/QYzVnnH+HJ5jl88i7v9x8z64XQ+nn+d2Tnsf8OnXh8/ooW7S5y33v8qT7g5UiI/fR2228dHedrvj/9kHT498Nmt7wc+h91h/cWyP5lNGav1RMHQ/sjA5zdfmd3u97oy6ycKU4/7z1pnPvNV2DzNKmxXh9+t5Vp4ud3/8mo9UZy6XX8+cernqx/n3eoy4Kea8c1nox/b8xHb9uvV/0n/sV5fZmK1eqqV+ZJN/8KVGZj8d29e3P7HqLRc1uXy4/ub71+/vV3A86c9SjTc0dD8kZWam/98/u7N38ZyPEtPD89wj8uXcMu8fI5/7fIRYWPv+FJHRYy9l0PG5sBJdaL7+OJ2OdLdufNy7LgmrmPje5rXlQ37xt7kpe1ybl3uMpao+2Vdq0Z12eXjnnXdHCMaf8Zx41aX3eO64wTsq031UDUsqzFoVIG785nqTz1X/VPbL7fOdFw6Qp9jeI65qnkxbUleaBxm4+zxFGPsVnsvu+tWkZqEMQdjbMYHGc80tuDely1Z23Ocd9mOg2vSnePmTNQdsyZk3NrxTOOUy2GaPasFq8mPMZRao7EFS1APPk7kOMdzea1U7cIE1oTWYDArdfoYZ210THtNXGiZxsGZmBDHVBqEzXllq4kaByYX1Ooz+dQ1tcbZqLkdczcGL4kbjxA1kzgCwgABdKyrtl/94S4uuDs/c26zGgakmBLFawfk7jKLWOt6JEkJHz1qygz3xtXnZfHdWmTqzKQc9ibKp/EAqwvG3DvvaTY1LCjkGIVBZU1jMw2O3/oLFyYov3OAnDuqm0uh6kvJyVgdn2pkQ7vdNY3QbNqCWleHVpbAaHoxNqdwO23LHEUpp5WZKqXWuCDJAUGFBI5fl2fI+iyRHapc0pr41r8uR43DLoI+fuUQrmEz+hem1ssyjQPG1WuOuaG0YGwtoYTUjt8JKziuO4zpODzxDZNlpXQl1MnDoZ21DSOr++UY4Pg9blYGKx2WM2VCa7/2JSe5LuF9Bs1iTUs9ltdJNcBxgbL4ZUH1SGNqhlJGlgk2jLpMVO0ZR2Yp+xh5zVEdPR6zprBmyfpZSmetdo/Zy9pU+8ugjEHXdb1PKeMyRld6bWXe4TfG3nEPN8xvmYAaKRavBB7z65jbcmR1eI22js3EWo8RcfFrIjChJcRWV8UAYMVKBTTNZWCxwvWINYNRJgnbOVm4PW+qpcVh9DVjLq1E3RwTDRdswa2B82sKAjqBbUZ9qieGzeS9ytdg+WCn6o8ZTzH4r4BfiTLDic3QCehS8AyHd6gTas+Q+OzVodoY3UcNwugCyg2X8MFVBNUIvjHr2lii0ipHNJGl6NRLKPWYZ8xxPV9NkFPvxtkuEa2Ao+cHNzUYj3KLtIic34QzD14bQsDYQcsBkeL01bPUjpS9oc25/3dcsES2jocnhItFiJSaKKM9waoaFrC0LmGPawW5hAGNSwyRyl6LU+rNLVTNhHq4Q9esRDdpCErBYAs8EVy0ctAQIeZILW49RmlHfZ1ClYgHYE8wp2YcCfSnYhTIQjrtQ+C6Na6yTEO7YaikLAgxsMiOYcOMls3CWgfse425dDoo5uWZUgocMqX1BXc0LiICNi6bVRyAZYMZCYh4cDKdegSzE7A8EhGofT2iyQzzoRjTyS65DL7jmTC7zuXAsXPv4q8vTuvPPsf7hKc+GQf2If0RmBU8HowK1TY6nyiDmIjlaHxqvhlCw/qWXFXY6DLanP+ayLIM5pBW6j2MmHmHgbD7Tktd+mLaTmNl1F6YK0Rd2G2uOYFzpFFJRNhUA6hYyS2uZDJEnCDIctsOpwRlYCZNs8UYnUNmyI9Fh77C0NdTQ+HqNkzKhvzRoY7nxwSWeYVNwx0YWFcMwJlPuA+jfWQuUpbdrCOjGQaZ7kajAwWntQroAV1DJCJJCxkoMwWPUEPa/5IGBSzJ0daKpGLWZBRVT+kYJ6Ma2o5yrrR/JVAK9/hQNH31YEFrzdt4ywYMU105GY2W5JYDqtQkmMbBRzjWGjmpIkf80T/4hMojQlCkTvcp6b1yn9Hus44r/zQ0pn6mm6hkzmgjsGAl+0z+2nfmo74T2hjKcjym7yz5NOd9rn1nryweO5QpenLdDXmEJVPxh74T+h7w/BDkhe+cQ0wIJmwIcsQeYjJZtnbvwYlyOFUGA0bZtBbxEsMafzKuhUBlB6XII2s2DYsByafLrcsvvHws/tV3U0hhi2nBcmUlxAjXa3kYtCWnRSHFw2mBb0OeUu4rOwuhiU2aBIXNsgmlPOXO65KyHbXTmZbElZ5fK3xpB5RY2bYU3q5CQWSO1HfcKqX1UidpPfTvsdWHtyxjYAj/Gdw7Y9uQnCP8vKc/eU9/HKJRIkH98bYEjJaQeBi8zT17VeIHvCRhRS2nvZK5YmolO2FJQKkEzXOaK87z0lzFA3NlC3MVj5mrgLlyptAQ/TZXspq07tNcNZTxYBw5x2HXZtOvxhE9DqYFDm+Q7fIfHUe2l3EacGaH8KTOSMCMYBLiPsaqVFUIr9EGQPqRK8KHQlDpz+TBw4Qx0KdSk4IJHhN0RKkIP434Ru01JE3061bPj6QQ3g/QWCJ11h0QxSOAYaSPQBMe3BCg0LoE8wHOWQqlwTj1E/9FnmLMyWkMO5gwbDBKMQJdmB1lcEHzBB2rUxw5PrKNmCnVx5xTMh6W5YVyQV3b3t5P7QIThwwUCohosx2nImZlzzD9FTAvTD8stHnreMpHykFjMuUlafkemGF4S4HCGmQyvEG8XwESM3t8hn7w1AKnBD5haxHbQJSSmASwipI0hCbAFODfkKkA0yIgYrAT5dmZPAk1QX6P4AjJsuCWsRLG8yIU19IwMemHCAAxgtgzuerP5d8OO5kawRlO83ll/hF2pqzyDPdiwrhtuGQ+nSDWwn4uDZcpAX8Ylv68/byyW5YtE04I+cp+PhruLYaRsp/B4ZT5DOI6OZMVDosJpaQaIlVWDlC50k4jAAqbWJEAsmEm6QxS0q7WBKk01sSuHXM+6pjp2225MhJsroxMOSSOUyLPVg7BqQ95BaCGIIarxCBDSeSD8VjSDcDePyopM0Tw+5LSdZ6FpGAxAQs7cRNpKCG+h5KSC0nxn/O0ec/P+s/42Vz4WVV0rueObpDmHy4QQc1EOazRkh6v3YsMrjwyEBxgsKzNmdElAFIM5IuB9HDpkeeImakDgCQ2RiEWhhMw'
        '0fIqvBSSMhhDwzBSbhNot0y8joMtabE2HGVcCyZODLmBlRu1QecjrAXE1cgW44jIHjUBNJQAtDkb99WyX0GZ093lx3KxXKYXMZHMeATJJA4k1AsuduHuOsyzZS5mV04XiQI8/yIXs4ljCge4xjEX3u6jOCbtpt/HMZd/218jq0TcQz33DpacuiyEA1ALwq/W1HtVF2Y0XY147GPxg/4AYDaeXPMDeaRBRi2xBNFgmwnjh/A2VgkcaAKEEZASgip6WhZUECyH0EM9rbvKXsjsAzEnoUB4BpbEgEgKUYX4w8Zlcg6NeAfEpsxZz5Dqq2UQgZQyskyocoFGJsCwFKmMuBHbV0SULKrQGTDGMpZHgt6LlpS4MOSwDLtp4MHyTLncstZMERmsE1dWCdJYqJkLJYg2SSxwxbPUPabgSJuo7MGKM0oUBo0xYdesnxHeJciPHIPoLwxpyK9iGonrwVbLlGGVPJXnGTAWFWQcvpZ4p11XyYjeoUgH+8qBatmJUSOz4S1Q+3JobapmT4TbumB0T50Qznfgj2SIKKPWEBJOrVBNq6cxIuY/RFYjeT8ZYdh2JfdlZKYOu2Q0YkkJSVNdiWgq8WZWIl3hLyL6bOuhAiVjBmQziJONdRJWvsiGMHABjKk5TB/JDsztgB1aTFxeOYwJsgnUsaSM2cyHVLIdjTLxgQg+u6q2NH1w+7AN2fEIkgIWXLXTMbFOwGnOXgBYBl6BcmBax0FMKxU5To6DdWrN4MNVsPNcIE2KgcrMINdgIIas3FkyDZb5Jk7Nyk47GyxYJcXkEqQICLDrCQ9n0AGhju1vi2bzUewz7/nbT/G2qBsuwM+rwqETWAxgItH8lc4uU9nl/bqhdd0wGvtMciaQtcHfsuj/wNvS8LCOO72tN/bl1x+EF+cMkugkVMzDlE1PzJwhYc1TstQLwBwocPY0eZdinPU3VChxnmaowQExbKDtNS2wK+AUqZpCBoqxhkHDjIAYjr5r5pCWOkOBUE0TSDxC/FpmSboBlL7MtgnXoowyE3qRh8y13t6pBGcOZtibLULGVZkRLix5M2CnOMTPBJaHkURjQuAJsbHMA+aL0gjEzfQTyEFdWQ2tHCtNmP5gbZRsHweWyxoUUl2wDsocqWhNfploSpCwpG/reBJ0JBM/iPObemisLOLCOUOtminEH2kW43YFqMwRO2gGEqFqAN1MkNUXcmbwLpA0coBoS83azihwJeVIEWiSkCdCklFYHBEPHEfAukHqUcNY0uZUPofLRRaSwGhY4SDIwQICoGNWClMwWFcceH3y2yZVqqz0JI54TwykhzmLN4qkOqovftEtwfJnE1Fi2nyCUlg+Vr56plit8gUpERVN7zkEaRAP3orNB3PlyoD5OdvieGBhRYGUaCvbTXHrxE0i1cyyN1LTXQxMMOY4IbAP8nHCoQgOgWHXxLkUjVHkTPJcOuxV7sp1pPoFANU5Stbqu1pmzFHlA3WKDIUJGye9A+kbWY4dApkq1IzOKV8mBg9XkLkFF0sZVU87Mx5pO3XIWd0GKjSrrQw+xay0KT1B6gNw8mC1XYUh9yZ7RlMkmuAGohKiOfGUwJ00cXJMSQniNw0KaxIUPiaYAM/prlL0LpgQnml6dGMsI3vIyCRJovHGR4KulgRVk7lj5Ef7B+/MlJJEKzg88oxrBJA3WHfQtlzSa1QNU5Dmir0JLkx7RSEhZSsJ8xK4L9eOwI8cPKM8JhNEPF2KBcwig8kNosAAW0dpQGqDsJ4U7mziIWsNQbidlBcASojEVGKB0zTxP/riQV5PKs6HW6TaEUAMUo41BfDeiBrw6WCS2yTqoOLEYrXL1XIXFo1bQtKPcByqHWLJzSJEF/nBEmRRG3mZYpJlNS1p4h2sCuCKrmGQjZgqPjPRkPtVjCRmIdMWlgAUUEcLCOEI0FyNjgWWhPIS2YucLFWQUId1EcpGBSceMQWMgks7Y0hnIS/JqYho+Bq5LJBFZGCyMKIuRhLYyobELRtto4KzJm5JtgaTW6bCRkEH34uFE3BpAhZAsml0kobwnvTHMK2lsrUQnpFLxq8ADdPxrDDSm5vdh6dmjsucmcaaNya32MXEA4jPuiR5lSmqjtSSdtZYDpF7ddRsZPGVWaB4IGQoWL7rqqPqZDlhCUJjQjATJCkEWOKqdO6G7I8lHkyZ8ndS+d3pnUngzsZTONVMlo2sqFZaDZf9CPTyZFYCwg8ZRiGxoYpfKEUkl844FURzvUPh7IyfS2EddhAEi1wQ6yjGpEgQvciYoLK1ZLPlgGUEFgEzm22J8oXPlI3kK1ZXhO+Ei3QZJD6TUlDLF8jgAJdge0geyPQiDN8Oz0zMculqCN6HS9NSAnIJMYAx/cm0YvIYES2mAIkgddPIbHQl9gh8nTw8kylvUCXnGFQYEHdVOCGYEcYujpDFYlBO2qlI7VTeOZ2kscAcKEcyFX2plCmbIAyYWJti92Yede5AlaOrAEwGIYVcKJd2OhZdVjQUi4Z71ORgKTicSLVASSKlHYt3NMfIg8YkpFzib3/ab1/8zu48AL7DTFSNPSnEA4QxgBzqsWBNtgv2xHstRVjNDnlgTcUN40VoJ5sc0CiRbHoIvSOEJyI2o4tm0EID0NcgqbBgSUFO3dPE+SCnQMO3RVYcs7vK3KbksJIrUA1VMyodcTIa5BS46d4EmuTjZP9tEiyRBODszNRU+mahrPNKtsaFanosxtCcJPN1hT2mz87NVFliLZXsUXU+kEZMM4tnzV4OUxJQYiqkFiqhIMFd0h/EhXikzXQ0hb2IVs+ENlUUcLAcWDuJXkM9exP4cKMQGEHmXjN2GD+m3CpYWWhDQcw7g3DmrXh8OOmJ7UeLMEAC1gcY0qZ4yaEgiig000fZ9aU6M+hTGOLy+2z2EUeZySFLBa4gv0ssyHLhIOEkIlRIM/JdOYTOOIiZqW4QgqTpVZV8K1VSpUoix5SJJpT6k/3Xml4jVN469TWF9Z3nEHtgTbPxY7LaiY71lCnL6oCdsuCzeVTcfrblwCm7arfqBPUOraNZf0rC5bpInA6F4GoHgfXsFovuAVWbg7MXrm2cioixsOmsahkltMVBnBitC0wAexnoU9Ta0dkyzRVCFfl30m8l2VhcarTKrqZzqbfMD5NVUh7ETtFuf4mZHoktHp1UKrVHk4q36DCCVLuAnEC2KwF9pTs5lbmTNQIWKsBEp7TTgAeLOmGNvKnlxGSOghhlCtlNGkeGDF3QSslFLmYfoTxoPKGQxNQAqBS+S4PNL2RZUI1SZLiop4riIvKEPC5Ca3YyQNEjfTKy2H2AmmFz3Fz5Vigc4YrIdQhzNIk52oqFvaYS36Q5cqWqbX2QAUiOomM3Y3LCUAqTmSm3xygPmEuI3AAfaqY40LtdGWagsx1mP2oiUJcIg33RBjObqWiiXCvLztY9+avoml5bQ2I+prqt1t5CYJZPwSdQQ7jc1DupOmqEWk6MpGq3ZszEBDOSLWtEu5Lt4+heVmdysmVIJYRG2JwxtasnVs2rXYwwFRS6QyJMHJ/ZBWhs7CJxfgGtisMXrsqjggK1704uUsy2QwncbH/TPjBKA538MYuOHcIpPLZF8EK4jGkQOzWZRgNjJqRD8mV3moBjOHsfZoO4q7UHhBew/jLEuGecmjY5UR6LoauERDyNjEwjUAtbbg0mqflrsj3Z2SaUnLVe0iWdtEA2CGsDtJK9u91OKmyesAqnyLtHRXh7uNAXtvUgn8qO6tggw3YG2m0xetmgxEfWMqfirVnkIYNVAZ6q2E2BF9ZMx9kYN6MM+iNTLYEmXj6PXdmwe+xKZFYdItRR3buoZGwNTWJTjBoa7WWxM83msmQj4IxVG6kkgqFaIaCQJJeR6BjtkRhwtKZAjKa/qKeKCTKpZ0fFfHaskD9j3qWj7uC1rpOoiqP6NrdxvsQfCFsMO9tHG97HETBRAh3wWoawqdxGgqFSbCPlnQwk6xiKfeMgEQN+XKbfal41VV2V68slsKGccYQJiGDj'
        'mgodDRHhOJpuaxxJzf3q77PQrdsGuDgI2bPGALOTbLwxIFiCVTIHwBXvYDC6TFH1hHyzUMeVpNFh9CCzzg7RbqtlAZJmW5kgw8QZ5S0rdwyEog1lZxiiy5ryJZZoJuGALy4x63qyHl2BKI9MmUjviIoVabGzyir6ZLZn8/vEQjHVgUg3ExEOMs5MlKin3mCwIIp1UbqrEGFqlUXNjwaEqUoCqrBOT8mJV92WoIaBGT0zO773QDygBbQvcJ0VVhSxwYVLVaXIxOZjd7bb78BQYqrQhRg2K0WUvQVLRf1ZJJ8w9utCJuRRKFDXiqMrtuyomh6ApHSjOKhWOIOaZYTTyG6KcbTsxhRjwGfFjcFvegMjhGWdOGTyTTTEOmdP93w1wqJQpD5Y4nDIDdhnxzdMoDvNu5ygbgw6TjEphByqHTm6GiNYKlKhCMR80vpUdzDXWgeG0c29c93E3VLfXwSLY0AyqVfBoNCa9dPxrDMqsdbi1DrTJbIywSCctCaKNWqBYi6Ei9HUPexkqIJTzDKl8lqhNCoOlK+iIcoucbIJUM1MJEiquX1Gqd78y5kaq8CiKBmVYCf3Ry2Vi3A1BZYYKTCpTlJXM7LC5GAJSw7amE2Es1401519pNFgNL0IozHmouEtFTMbJ+FSeYCHIMRwYdOCpicOY9M9o2wjTKOR1H6/TSdCSidMVRfSbYXGMKeaoB4a3SHysITqMdLbTEg6mjwB+TI1vc74EFRoUz0M0J6FUl43tkiE7Jx14iiaK40sejGJ6CypzYjKaEwXsDRmqlt0Y/IDJlWXhF4tbis25ztsFvdSNBnWv7xrR5y6pjIkM6EA+VbvnFCXzHzlQeonnwk/Oq2beUHkg4g8kcycFGFYFqZdTA5Idg9Wd0J+dEGKVfjFcEmL7WySAH8dZoSmhm0hkhL285G4hMxCdTrxZrVoEt5UF6ZeExbNWE29RSRMvIVGHunH++VKdAqAaQWAszain1m3Fh9k+Y1NF1AvluT40hO9o0qPSnK43p9gXe3j23uCqxCCddQkknz9kLALvRGm8YQk1MI6V3bV1kjhUUqG6maTXwRxWUebnbk4ybmp9IzmQO86YLXC9U42hG94cpbE9RINVSHYKM8FJ+HXWCJm8UsIrqu/1G3xHrDu++k8jImgHDs0XYSHmKg44wRI7AIL4ftZ0q2Ne3OOfNFKF94v+VDBTVQEELQInuEZQxEpKqwsH0MT2iUzpQa6RJezMMnE8h5pC19Qi/PqtQoOws0nU4vt0VcS5RUaz7ecPHwjkXk+yixOdXI99kaiFKJhfA3Co28kcnKDiN1d99XzrULZSHWQj95JqliFCuoZO5jKoLQ27nOQvui2g8Mi/Em2pt6Wlf2GhRBg6JP6INjbmyHjJIgqzKBL0PvNssmZ1qYIMqp+fuLokf1yk+QY1FDtjW0YGx7QeRCEK9VZ0y9AzBTXn9VMrqLJD7hFdwgzVOTralwoPrAam40Rwa6iFFBDJNtYBE6+GZJJKZsPtIys1HSrA19saZZtRjNlIGLWwvWixMk0THVUkGLClY9lJ35nFmqxZzeS9Nfmu2pEq7RZmNCcmDVBddGuQ0cm2umiQttpM9+mkCq9qCNYZZJkPyZBLMb5Zp2Ges8arb6L/uLZ1Xb5A3EXJ/+xpYpxN3EyV9esCl1KkxSvuXK6WQ2SObBoRDyWLSZSQRJzWSEnDJmMCMisZgzPtiG9nGS8efe7m7uXP9y8ev4d/+O+H+7u3n340x//+P7mw90fbm7rP+T7w9v33z/7x38Dqlvd7BBwAAA='
    ),
    'ZNF512B': (
        'H4sIAAAAAAAC/+1cXW8cxxH8K4aeHZvHI3l3ees0kn7LQ9JPCQyBks4SEYlUSAaIYeS/Z6erqmcp0U4UW3ZsiDQp3n7Mzkx/VVf3+tsnd9+8eXbz+slvP3vylz/+4Xxz+rsnn3/25OXx+vj06sU4+vs//jlOxtfmcLE7ORlnn7+6vXkzzp3Wx7v7y9v75ePF9nC+XHK2HDpev8CBi8NyV11ze1nHfrNZPl3e3R3fPHv9zRgj/uSvtvsxzrOrm/tv3h7Hwbe3N/fHq+unz29eXF2/HCfH/XfPb6/e3s95Zc1re3HY7/fvXPO9Yx3/eXN9t5z661/nlMdcTzbbw1effzaOXmy2Jwcc3Zyd7Pro4ewUR09PTs509PR0t+XR7W4ePd/veHS33+joMuw5jm5PL8776Fbjbs/OTvroxWbPo7vNPLrbc2bbw2aro8skNzh6drrp+Z5tT094dLm4j54ddHS3udDRZaEc4WI134v97gJHd6sV77Z7Xrs7P+057PZbjrs7nJ/q6GG/4f4eDtuTr756KKhHlGd1dq1HY+zVuRLgt0++Uxm+VwOu7p4+v7y+ub56fjlU//72H8dPevEz68W/Pv/sPXHud7tlpA8V59eXr+8+yfMnl+e5VnE4nG+/R54XH1WenyT3gyzxYve45A6b02W/PlniL0Kem1PO93B6tvkgKQ8pnP3qpfxrtM+xs5/s8/9WnmO0RyW3OVlmerp5R3KLiO6Oy39P3xxfXF3eH188fXF8fvnNh4hw90mEY9PPP7ZJ7s/2jwt2s93s99tPJvmLdLEjTf369eX13xYRbBZpLh9vjy+vbq4fZKznyFh5prPV3QlZj+Pfn759/Y8hxiduYbF8uZunj78yzMPM+aFOjr/SlwuWo56xXDu+zJfLxvG6Od2WM7n8sZw3i7okMP4Yejm0DDcujfG9/FlnzJYB6uQYtQ7jtvFxOWV1QWLAHBNNx4TGLzfOvWYU46KoaVkuD/aa4BhtuS7rNJ4zPtfsagxML+reGs5rOG4BllBjjYea1RZgcxxbETEew60ILH0MNf7KesjYlZroGMJDe1lLHTvH6QQ3GoOOrRq32RAH5OD8jxvhdRazMT4D2+M4FDVxLHWsOfHMWPY0A5uAaeM+r8H4iwKCxBYJjPvSuZ9RH7Kmn+OP1VrH7HJ8Q6PGekv2Y0CsCAK3uqs2BGvXhmNmlpIN5jFUK3ldhmRXt9a1IYFaCQg6UreVTkKgkJVxx0uFcWNN5Oc7jXWZRIk9yxKXUdQOfajFS09hZctWlxQMgy4PgSXg07DlugdfQ9vSykQyx88iB9jH+JBWn3AkSgTQV6oBvYQ5LC9c/qG8Rp3AB6ubU6vUVWUZY5Ay9gxMBEqleZWch/2On+VXfUdafw8jqXFr7oHFUUFqxeWiEnfVZtSejP1N047Ub7PoT6XWtY42TXioWpBBweUfIKPQMe73vCDwgHD6G5ljicvpXB2uEXOTA0lYRakrH08nZ84N5bEy7NrJ0oqEY7beyaEsw3lCsF42CYODHOojLLcejalnefqgUWv99NjUrHLqJSG4kfEwrrsWAy/d7hmei+o5xBRavmcFjWmpEBe3Q4Y/5Ej3CV9Cp0cxJKdFx7n24fCJ2sKaLZXXMSjjHxda67CeHRQDzmjsEK5PxrZ2i6VUFXZ4qzafZobIMSRhihQQtVHGMfRyDA8HzjlnQFFM9uISbmkull7Twi9HiCpNiZpXMPgyioUk4JBalJQRkjAYI/xQGDjs2g+XLo3lhgI7ggV2X1r9/i9LTkGzr/8ytAA92BjOOImWZtm4BZVGul8/GXyKaytCsTkhHHiWfnYNmdQzGF9ZeJZqZ8OdsgqEj9SGldigADUq9yakfEPqDPg4C2EYZk8IganDtsb9mBclB42hpx56C5lmhQzIyxk6GPIZvDkxbElQmbEaWhpBGOQFVw/H6LyZgjHGnp4aTcCJvSAY/oJ3QlRNl++hO80EgCtfXBE8AfgqVJUvhx5bRSF+1SYO116rN8ypfDS9d8nBZOUGffD2sN72NoaFlDFp2DN3DQiNjg+CNWkiwqTgFQEozFw6FtBr7Z1DEyOpAvLZ3CG6F4fmwsap7pnu7cbpGSAU3F3SKkMmmq0ZA4M17oXPcXogGopMrOAqVDQDcbWViBYK/8CwA6+U2MIMugu6G0hLChVTbMadZaDAGOV7nA9UTMAayrs45mvyDgn0DClBLMZYE66FEcSmy6gdXoDwj/5x7r3Qhgl1ly2VRvO5CWWHX0vZSM8LcIQRxILxy61xQPtl7qNhrwvZANHVdsPiIVCgYeg1/bnEkVwWME3hlkI68B3CGgqryRkFY0gZQioQIf+wmcVlB01hSAbxkiznQvhJkUpddJdSGkYNRFFsGgMmnleXcAlBhNYOCWoGV4BnGKAIhG0MPQnkyaSpRMUMEtGQnsiMppUzmGpvEP0RsOuMHNyDLayVE8EFc2Egc8Ab6AzRj3ImREdfaVtZSrbyh+ySgTUK6xCYOl0z9CdTCJKeVzGpnxMAGYKiIReSq2UBE3nOaAunoiBEFC5tsIzGxUbAHkIc9Ak0zDCeraQPiRz2IrVfsgbsSRLVJiboGKVUk84WcBMYVAkzkQUjH7GrMGDl78yZo/Py9i8m5ACBI59VQpwEvMi1kwk27RyZE44DgiclU97OQsJzGra0mnrulSXVI+Ank5Oo6FlpA6+BqgDGJNSUFtyoOoq3EHLzeQ/z62RkAXHCW+UAIG7jlbm6Mt65cro2jhqPjhrvDfodFz7y+A+4Mr/7yveufWxR/uioP++V8R82Pz7u5tsPvFIu4L8bMn7ItT+56gkTA+oFEkLRh/QqBBwA0sBL6TJu72CySmKKR4FHLYhK18IEomGQ7hZwsAb2AH3l+QgpGLpXcBwBwTuijJAiKC4vSPaNzA0wAcMXXVYyymYlpYmsiZNjRDDxmb2vRJcFXFI4lW6QHBSAsKW0XvmGCAbGGWwNEYKgqCFVRMqs7BtejygE6ZSZIHQCZVAGZlNtyCFkNkk1YVCaXDOiZip0RoMLxVgr0AH/W7isfDVQmlUOY1xjNlTgFk/BUcSVtCAaA9WmK7UGJwj0H4h9WiCTShKxzFSJVZP4BIJ3hUjEHpA9DPpULyE90x64MhVyL8wzQUI0IZ0sCQiQMgWCRdG4mP1I7N5aVFtF6wP9W3GQoAJJNhI4wEJwhqWhKADIGSmHWpHHDPaK1SJTijXMh+mTZJqcfJBKBj4D9IwgPZZKpr3BcpKaJUZNgaiJFOomI09BqiGDeUoKPHlnrp4NsNKILpRne9P/KeK36AyoB3QGRsu8LWmrrMEwr8TNKb6FlQWTSK1TeiTlGJOJ4hgkKAI6QaESJUfyX0yispIZsMPg/uhxaxwcDXLCE71F23Z0vt/GjSTEs4mPwKIIM1uvga/gZ8M7R5InN2YR0eUIUyWH26aSU4rhSFUjUDBJScGw03QVWC+JWpYfjI9qvda4E5NKB1iekLZYl3AmD9jHoyl/E8FAXJyA/WCfIVaT56T7ZHWB2wWf28jbO2wwvjVpZPTH9FnMZOCIQLdUWpYi8JrhDbEyICaROnQaCoK/iHFYD1LhJM2Y1G7YtFmz57zNvOt3AWrITWEAbgdPI4FAVXOuCz5cdbVETkFBI0Al5hMsGZkrD56pl1EgIi2jgwwLaSZ9YsTxZAo6ydAw8UQsklorrvR9xeJZT1LUtthtsBOQvQ4TR0zqj+U0mBRSllR9NXUWJRjRr8lyoAsEiNH3aG8mLZF7wN51el3PooVatDtela2iM91UQosACFI2WZrhMkDdJ8vIrL6akAIZXya80S4P5HCADs9obUFGaAmNb/enWm3SFiuMKVk12oYbw5cyVIASeTeyUqg6hQg9pclIhl32htR4Vi0RMZEVGdkBx3BITbt2iIRfN5OrI2TpaGytlhSuIfEMcpnQFxcolC8rtWt4ANGDEMto9Oy5ct+AkLDNMEuhERJmQqlJSoPPMIJdaGea9oD5'
        'cINMb5TY5KaQp2mKQFw5mQbVLxUJGKl7yx4QqdmVOW10PQcKpzI2lBNlq0zlGMRM8ErCBkz+xUeFsgk4DowIn7dCyIR4EHubWEoluCngyblJLc+uwkQVnYrVAKYpgBAiN0jCrAkKb9IrLAmqUEF0mHv3Roh2ZamdKCsEe5JwL7E/SX+iQMdQzkpob/xMGlMlGmUUxm4C2ZATtVorLiAxWDVXJViz7UJMxiTkPEABO6ASyW89R4SkpdBfsoAAjx8SlZJkUZcpiKdqbZopkONZNmsmMBsxtFwqCz/RtgKhJ9lO4kkUu1IA/t1Lc31pzkv/06gfcKl/jFE/cOD/YQ72o476S7r0R5Nt/m+Xfth0Uyz4x9Cb/BEn/VNd/XHW93PZ5rv78DEloiINw2rFpEQWwNzX1GhYZYBG8qn2NTD3Fix8qtWEQZx9fa5C6WPXJGmCRGcUUhJ045iuMdPkKuJinNRdKBauj7hwVo+cwjoPryFBwJ7O6Da5OUOOP8fhDItl4jjxcJzUOACQdY0q9Kl6iPtcBaGsynTJs7Z+OvinMZKIFRXBzbxxOWuD5hMXqPJLdBpNuQeZOMFnFh2r2xSwSDSvMBbDuCGDZ2an8rqyPyepkQ2/VL5fU1XaU+ZHq9YcVbGZzzKlJjxadbMYG0OIG5QSA/LU3ibTEPYKWifurFC7T3ARTGlAEYEXFnPJNEd0CCtt6pBhIcqiiSMmYKnWF+W8kwYx4vbZDFv6wXq+++ycY4MZvwGg1ZyAjj4ycczDi5my+TCBfzT8sRZbRk6UCE7QITFsLlgcVyePof3RZquBI8vMzj66cQYQN1SnA1Mhsh8kgfqlVr1zzWmVK4noivFcrbHblRxbkLlim1OqB5attfjsyXYO9gGxjMnkMdVDo0eTnUALs8VkZVbNtCZWnb166h4OEd90l+iwVssQZqfGB6OeogiCei17xtUvRMaK6pw0cpVfZtc180wxcdrbUI28Da8aBEUXmVJFmgqyj5RlpBYt01dSSS5eJQA4sGSDEEOLejjYdoDapSVyqhagOvyMhAuTvUiWe1APKhULkTWoUbOS0LIy1mXYOEHLYZc73YaRDmVHtYPZVGMmGG6WSMjuhnIrtQmIzIRdd79ePmBuaG50VAyfKjKEiA6WySvtLC/iov1MZLsYBHAcqZJMaP2hVvuk68gWONZmaFVCaAaJANo+mRR3i4jRo5l3o7uxYwysNSjPPkDlTLSZkmGDEhpqFwh5xhLbVI5mh1bFBKbHbLSJSbep/KRCjrFzZHYWVd+PGiEsZ47PhmGz1sRQY6W4DrZI0fjJ9RAmsR6VpLe6e81md1YQcZARNbWYdo8LSAdTY6x1ESZV4OsGyBRfpl4ZuN6mgVydCGkpfkdvkUCruxFSAZ65uHXzu6tsCg/L9k7qKUtvDKlubcicO7rIVS1T+zRpRlc/EqttZEzYDalA3g22s4QsjsvpdUh9dVeHqAP2MKY3iBFqNQYkJ6IgeU7+v0nXCDKR4irInLhwa4XC6noXv40pwgrNJpcnnkM0hhpQ1gwri08M8OXGVKfuLh22zqprWHKhQIxt6mCo1G9ts0+cLTdGelQaQdeXq/qbCrZqwO3XkNBTCveipn3UrAo9BLHn7P9Sr+C8WO/8BPvZSVijeqldY1dRNswxkVsmqMNKkkJt8D0OtpK1MyB7OqtCuCZJFnW/vrbBm3lUfYed/a7F4CUrWzHPzhd8XBzv6myQ7Jzvc3j38K/hAYtcqo/F6hWxLsvCeNzVRhsmbKKqhzVKNzLUsWrr6u6u7GeoLU8P6lcVVuouBo97irqmURedXQR80hpWME1ZXdfvhMEd9csL8eDOeO/XnKY//Gc+wPluU0t/aoKxDhldS5utcmSzk8X0iFh12xLVoKBmLEHxxTK8wuLrqx/++fC3q5gaHcTZ64+R6jP9XM5dqCUhPCBqBhvcvYvpztea8J4SiW/2ZvJ1JWQLRGfMOOhnuiGfSo1uchaXyz/IZNgx6L4qe2IlcrLqDup2m5g1YxSKiXdlnNmtd3AtE+YRneGlNFO+6tHRXlVBNZOiBg8p8iUFUAepdxbUBt0vPHRzLbyZd38yW7ZzNkJHtzvqbQGbpTOpoErLwZZd1RyorGwfJlg3wWZkxWzYDjXqg/VHx46xYSLnm0mm16n40kRXm61zMHXxd5XsnZ+cP65qspo0PR9eQAFldiu1r4/04dXpvvP9IXg8+U7Z6qkadj3S/Pedh9rq9H/30PkL/45Xsb8+3j9/dXzx9Gv+/81e3d+/vfvtl1/eHu/uvxiv47959vqLm9uXT/71b731nTw4TQAA'
    ),
    'OGFR': (
        'H4sIAAAAAAAC/9VbXW8cxxH8KwKfBfu+eDz6rdFI+i0BnH4zDIIiTxIRiVRIGohg+L9np6uqdynJAmzLjk2Lx7u53dmZ/qyuHv948vD+7Yu7NyffPDv5Z/z925Pnz05eHW+PFzfXY+hv//hXrOpnv9qdr8e3V6/v796O7zar8fHh8fL+cfq43xxWu/XZYRo63l5jYL2bbqxr7i9rbD19uHx4OL598eb9mCK+9dfbw5jmxc3d4/t3xzH47v7u8Xhze3F1d31z+2p8OW5/uLq/efc4LytrVZvz6d/6g2s+O9fxv3e3D9NX331XKz5sT5/X2k9X29X3z5/V6Nnp9gyjZ/vVqUYPmx1HD9tNj56fHjjD+X6/5eh6EtZ5ja5Xp3tdu17v9huMrvdrPW292WxXz1tc33//dDNP5FurXX77kajn72qTP578rMA+K6Wbh4ury9u725ury2Eaj/c/HP8Ssvvp+bOPtrw9W+32v3jLLy/fPHy859PDYf//2sfudPo9fJl9nG3Otl9QSz+z4sP55mx99mVWjNhS7za7g9a22624tt20nh49yAb/3JZZ8vlCGv20N+5X52uM7jf79Z9XEuf7s9V2/8Ut5S9oE+dnu+1u+4Ekpi0/HKd/F2+P1zeXj8fri+vj1eX7X2Acq9WnNr/b/Kk2v15tD9vd6R+0+d9J89v94XOb355PNvnzmz/74pv/Q81+u9/+hs0f/iqbbx3/CrMf+O7lm8vbfw88DNB2f3x1c3f7BOptAaX5zQzzTomoj/+5ePfmh7HVk/R0C5te3N1svKal+XhXQ4E/0wWWkc73MY3XT9g0wbhkzGDTBdPd08s0Y0yvmGZMj2sip0fkmGF6M30Z0zwxpovxoDEFPozrM6Zvre7CpzHFmHLcGzkekWOh46Ja2TRDLWqM+LSI6f5w7GzcOA1MN4yH2rg/sEbeOp6amKPeTuviw8ZrLXYaHEvFpSWXMdW0hPGYWVBjR2PKTOOOp7eB5ce4fuwqanAsaLpp7De447GS8SgKp1ZXIswSYT9pvMn6O1Y3xGh1d1BHtaooGdbLuDylqcRfgyohxGmKsbehOo+acGiy1Iq1W8mp1FcaHRvP0oAnNJBDJ+Od1S6HGdWKqKT6LsekQwQhhbpDk+Mbqie46VrruGpIsVaGvUMqVnoaH0oXtYPJHn2Mynggk0iqfEh96KJmHfP2rOOWHI+G/koZsE4fkw3J0JhKWWV+sHeopSb3stvA1mDHw+hgOF7idxgv1QE/Szx23BWhfUH9JaKap2asfZfEhlTHf2VJdRuUXv9xFBaVdXtJoL6q+RwbhjdFuXxtNmAKY7kStsMtyiCG2gxGCFfxoEYMt1pJwbOsGNfAQShYrCnrYqecxkqjHkqDkzIrkDCWcKCkWBeXzYxP+Kg4MO5xuhivok0HTIGT1GcIK+Hi9RkPiYodJcX6NuFPWfZVYaoWpjmGNBcLcz0ps9edCEfGh5QpIHCElgS/s5oUHsu11UNrCGvrCzBWkQE+O+/OMfWQccYsB7zlbzilgllc++lva0nOeTgj4k5ZUEkxqcBKBfyFF0JsDjsbTjFeITYoObBq3sl/VhF5PNCUXWKRYyg/XEBZ1gK1xvmrDmQeeIMHhGRlvTMYii+nTU4V3juvNzRpZ66I5KD3FUY7T7oRHzkLqNIiXEDm48yqrbZyD4RJ99S7lL9JVa4fGGHJb3ZNZyCj3hB2YKFQazJsGZ5Z7j1bD5ZU0Qt25ti1no4g5rSjbN0bBZ2UuNOBzWQqfR/WgfjFbbpihDPJUbRMYp9/5cJL8KmbFIa9BpnQA6Kd7d/peYgNlbErcfMNTAerRQzGwPIqihWZp8KnYAfyBMMmhoISByJATgzjVuEdiSvo/jJzPBfKpFgTi8fKayyVazJiRnGVosuvkztBbo3ko4EfEKor7xBEQI/YisKYc7GmYMeo19EJKkhCm8orVrkxmSMUwCqGl/SBTAoFlhPig4INkIIpwbsSSs3HKO5Gb5ufUtC0DBe5XuEeedEF/yw6kiVfky7GNTDVYxJq2lxWCxBXeK7AjEsisJcBYGh8xGTmdK16fKk8IST4UArIcoSWXA7BxAmIUEkssvIffM2wYiaR+h7WKKU5AEwAjMP8AbuAF4gaBDPTAOADm4NhzBv0YP1gjeYJFgGPZWlGY8M6h88MlRScr9SNK9KkRib+LhywhkzJBq4DNIxEXmjD5kQBUFMwpww7g3f90itqxPC0LGtLZlwzrp64y1qmgjwMBsgGcHrUOqEwwAyULDKgcgWjNMb5YaxQgUv5tLKa2BSEYJeQKFBjqoYylGGBvXIRXVfgS8Ab1B/JAiPkxK6SoKqwQHHD0FZhpWdkBZhdqBhhHOA0vL0rNKbg2nmBK4Q/AGA+zLSYMP1huRjaX22OpRZ2yFjU0ckEk1K/RieGu4VghiIB8wuqvuCsRM0OG0fJRrBdHlJeXzpQTKzKmIp2gFUGsDSTCaQMblHjWSMeFpbYL+A1DSdChWwbE1WpGonjRlPhJagXgHNstoDEhoU1AKADsRL7ySrwgL0qAzIUl+2ztlIihJRQfXB17bDcFJGWITQIKWX5MvBJMmCbYp9Cv5ZYAQ56NmQQQuTyFtMHVxhB6ZDMDAhXWJGzflc8JUg1oUYrbyjbZ+E1LrPyguFxKcaCr7gJGm/8gzSC8kbBgvQKiIg0VjHRSIZekRIiiILszCszzrYxmOQow8vWAXngKcTt6dEoNLqAQKiAeAIVoIGZcGCAioaM51VGc1WF5ciroMATDK5R4ChSHDRgYl/e7kgKpDYqRTCkFE2ByO/K7Ab2hEaTqPsdyyULYAQxNsODhUBRZLG8FHjxBpsMNVSWijywCu0rJddsGyaeVGFKT3Fpo3aVJDFcqIIluzXEaz8Bd6Qi2LA4F8nD4paWxMqSHu9NYXX5XJaH9Fjcjc/0GTg6Fm1w5bJnwXwRgnRCFDWZ3rUrv0EUS/k/KQhFC1XQAW3KjIn8KmUxeMCiXDIj1DS6t5Kfu9IISuogS8cnZhfT5aBJ/pGKgnhMJFDJOol5TGRm3VisCcKcyTodwBIxixguAD4gewTf2hHILZJcpQ46UDFiRvRGaD+TNqWw/Ah+EQlW0iQzwIRtgJoVFeDTC/CVM/iSmz3FXr0eUaAVPCqFsu6xhjBMvAhPFh9AMPCvnwZYgJqIPEqj2QsN8Ku0SFYP2Yv75A+JOERN1mNR4hYqKxmDXQQcNyRdsq+BUMXIBBiPEFajAB2lVYhPpkM1IQWYqr2mx1IyMkbtFNQ3pqlCJsxpjbMyQ2RapvKeCiSaBMQJwXfyZWkDnybeLF/EfHQNgUfypaSxHIslWEIpSzDhJOqDjAdyGeMAolSDxCqMFXMJ3hsNGBAHgEnQDSWiROALkN4mcGiKvUHzZx2Y6kWgukVGQIDGk0wsnHSEnAN8pjaGqlsWsAlAKBqQMRUgoEIzMz083/tKAFSwnik5lxeG+D5Rlkwx2ZSUNT4MeoK3OwkulM4VmlmYa5e5zOgpgYWq2mKYHU/mR76T2pweTKwBFUtCll2vlkJQahMwsPqUBzhrI5eTENyJdvV2AVGVlaoFHVhSdpRQUSDaJMBfIvAI44SM3AMTMc+wv8KUVPemGDNDI4vcEtcKBEKyUnxyxagUO8omSYjUDFJjgDmMu7ARQxeDUm10gBKdkkYMdpFWaGRE93RK9+IaDP0QEhAiMLhq2m7tuNhMZi+FA+9EjJAGsAxc59k1X+mOLKupM0OaGt02lcRzl0vdrBnpkvkHZraGEmSDQN7QhZHUUjsUJJ/RHz2yW1JoJUKcascgDZKkcdovmQy28QT6cpaTMKFK0gWbQagPt2CkByqk4iuPoY8Aj/CmiIgdhdst5GHWwa5uBkfjIqVcIYN1DOpsQHFWaGX1lQ48WjIdxVz1paprsa5kl9Vtm2EwgKMAXRWs0V05dt+cTVLFYLXEgkYHgwNXqO6DN40LoshVIYDDZxPMOuApCnQ5R/oRrq6esKpf'
        'Ex6HqxHDoEOaLA8MKCjJQLE7miKkE1Xix5jKFlhmgakwIkoL1z1BVU6fN2agfIKq4tOgSiHRnnBriU5eiDuaQRNTTZLDY9vuI9BU5juDpgQtZcmA/uS3xeMUFpseDJEiEEhiI9aTLmd5ZbT+OZKwgmJhze0Ce3WaRZ1h4rphxSQjnLxYqm9dsgS492Z+jD7QyC3Yj7ImVlBbVLQhmc6ix3ymcNiJUePCGk2gtiGsJOZCXkQTb4GCgpm5LkfjRuWJnDRYgBkBu+K5C0MwxicdCeyrd2FYXcKUZxTtYADIpLVTDL+4I1PuJ7vjzUKxiKQ0WKumEegxJgAzoYJo4O992iDYV0iBxj40wQSi8MVSDUUwLgHlYFyI8xSFCAcyP3RsnapgPYbSrA2PMcC7yx1MShRQquPcRZ6H6nD2Jwwwjs0PtVhFT1GH7HMtjl20DEhKsL/AToOzEk22r4IbN9TP7NCAHBGXWo7HcoLQxkEvu6n5rowUKtIF97APwhG2+hivY148QIioYZC2BiU767tED8x0jEYkVSSX0Yy4LzhfpklUnOqAixd1Zg5WgMzUjBY6gYHeDuSe3dLIZj11DGLR8gwTHbc4dYNqf26wMCCmzoZkxKIiMP9Akcg3MgYSDEqZFLUtyHpWLPyACYgoWMgDvycXb6a2mrFFbHMrIMSoNBMr02/4wjxIMmbuS1Q8DfFSKWiTIUJ8plxVbLUgjX1EhpymEDsHNb1Ke+i+3YKwhtUQaxHhLZQIyMCIHuR1Uh0m1UqsvgGKGqx60/DdjWY7iLE0Ra0iNrIh2ycEeOyosKC6u1CEDnP0ZlS42qIbYgunn8OfkQ5Wz+zT9/pvvTd+3b329F5W7pa/93N/1zWL6Go2lhUJrXnhDjwvxNNqSQIBXsjaC6ZWwUJ1Yc6lUmfz5bk51/EInhxg0uFBxdQpL5vbzpYqNWn6eoKChc7GMVkaD5rMrq6oZ/NJP5sPg4i+zmADow9bKSHq5EU60S5TFVtaBPqhlCHEop4IMXQCtWgVbAgAVSYqIMFnPCiSWEftFpAgJNtTDbwkgAKtnaIAmQF6g+SKeWLEgHxTAIO4SXFdoUQWlUKv0ETmktcVzcJTfJF9ZkuRkgmEkVWcOqpKRVYe8gK7VHtFz3tZjakkJOujc2hJZl/nL1Qhd7Wa4ktwUNW73+bdBcrFSTPWpWrW+QzUSMqI2FGKs04SnSCEOlF2iFZiNdjoyXSWKXnYMXXiTx0FyQZnBSmjNv4u6HiWxZyUXPePs48TQrFzS2txbASwGKVFdBO3wrqxRAHr25gDlCwxMhK8OUMUMqGaln0Cyxu5sEyjVaQay3z8AiakuKucz1DgRI7zKCt5IWvmXLAPmB0lKw/49JpCp5shEp6fmxkhgI6WcuoQG859Ju9OzEsgA+e0TvLUiKmbI7fwOXn7OD3/8vh49fp4ffGS/yvn68fHdw/ffP31/fHh8atx4v7tizdf3d2/OvnpfwbAb24gOgAA'
    ),
}


def load_genes():
    recs = {}
    for sym, gene_id, gene_set in GENE_LIST:
        raw = gzip.decompress(base64.b64decode(GENE_BLOB[sym]))
        digest = hashlib.sha256(raw).hexdigest()
        assert digest == GENE_SHA256[sym], f"{sym}: embedded gene data is corrupt (sha256 {digest})"
        rec = json.loads(raw)
        assert rec["symbol"] == sym and rec["gene_id"] == gene_id
        # 10_scan_gene.py get_gene(): refuse anything not on a held-out test chromosome
        assert rec["chrom"] in TEST_CHROMS, f"{sym} is on chr{rec['chrom']}, not a test chromosome"
        assert len(rec["seq_plus"]) == rec["region_end"] - rec["region_start"] + 1
        recs[sym] = rec
    return recs


GENES = load_genes()
print(f"{len(GENES)} genes loaded, all checksums match: {', '.join(GENES)}")

Scan code, copied from `10_scan_gene.py`.

In [ ]:
COMP = str.maketrans("ACGTN", "TGCAN")          # 10_scan_gene.py


def revcomp(s):
    # 10_scan_gene.py revcomp(), unchanged
    return s.translate(COMP)[::-1]


def gene_frame(rec):
    # 10_scan_gene.py gene_frame(), unchanged: sequence in transcript orientation (a - strand gene
    # is reverse-complemented), true sites, and the canonical transcript span that gets scanned.
    """Sequence in transcript orientation plus true sites and coordinate maps."""
    lo, hi, strand = rec["region_start"], rec["region_end"], rec["strand"]
    seq = rec["seq_plus"] if strand == 1 else revcomp(rec["seq_plus"])

    def to_idx(genomic):
        return genomic - lo if strand == 1 else hi - genomic

    ex = sorted(rec["exons"], key=lambda e: e[0], reverse=(strand == -1))   # transcript order
    if strand == 1:
        spans = [(to_idx(s), to_idx(e)) for s, e in ex]
    else:
        spans = [(to_idx(e), to_idx(s)) for s, e in ex]
    donors = [end for _, end in spans[:-1]]          # last exon base before each intron
    acceptors = [start for start, _ in spans[1:]]    # first exon base after each intron
    body = (to_idx(rec["start"] if strand == 1 else rec["end"]),
            to_idx(rec["end"] if strand == 1 else rec["start"]) + 1)
    tx = (to_idx(rec["transcript_start"] if strand == 1 else rec["transcript_end"]),
          to_idx(rec["transcript_end"] if strand == 1 else rec["transcript_start"]) + 1)

    def sites_of(exons):
        e = sorted(exons, key=lambda x: x[0], reverse=(strand == -1))
        sp = [(to_idx(a), to_idx(b)) if strand == 1 else (to_idx(b), to_idx(a)) for a, b in e]
        return [x[1] for x in sp[:-1]], [x[0] for x in sp[1:]], sp

    alt_d, alt_a, alt_spans = set(), set(), []
    for t in rec["transcripts"]:
        if t["is_canonical"]:
            continue
        d_, a_, sp_ = sites_of(t["exons"])
        alt_d.update(d_)
        alt_a.update(a_)
        alt_spans += sp_
    return {
        "seq": seq, "donors": donors, "acceptors": acceptors, "exon_spans": spans,
        "body": body, "transcript_span": tx,
        "alt_donors": sorted(alt_d), "alt_acceptors": sorted(alt_a),
        "alt_exon_spans": sorted(set(map(tuple, alt_spans))),
        "donor_motifs": [seq[d + 1:d + 3] for d in donors],
        "acceptor_motifs": [seq[a - 2:a] for a in acceptors],
        "donor_genomic": [lo + d if strand == 1 else hi - d for d in donors],
        "acceptor_genomic": [lo + a if strand == 1 else hi - a for a in acceptors],
    }


def window_pairs(texts, model, score_fn, batch=SCAN_BATCH):
    # 10_scan_gene.py window_pairs(): for each 600-nt text, the last-layer vector of the token
    # covering base 300 (C) and of the token just before it (P). The index logic is 10's
    # (slow-tokenizer branch; NT-v2's tokenizer is slow). Two changes, for the GPU:
    #   * C and P are picked out on the device instead of copying every token vector to numpy;
    #   * the forward pass runs under fp16 autocast on GPU, like Run B's own evaluation.
    # score_fn(C, P) -> (n, 3) class probabilities is applied per batch, inside the same context.
    special = np.array(sorted(set(tok.all_special_ids)))
    out = []
    for b in range(0, len(texts), batch):
        tx = texts[b:b + batch]
        enc = tok(tx, return_tensors="pt", padding=True)
        c_idx = np.empty(len(tx), dtype=np.int64)
        p_idx = np.empty(len(tx), dtype=np.int64)
        for i in range(len(tx)):
            ids = enc["input_ids"][i].numpy()
            body = (enc["attention_mask"][i].numpy() == 1) & ~np.isin(ids, special)
            pieces = tok.convert_ids_to_tokens(ids[body].tolist())
            lens = np.array([len(x) for x in pieces])
            assert int(lens.sum()) == len(tx[i]), "tokens do not cover the window"
            st = np.full(len(ids), -1)
            en = np.full(len(ids), -1)
            en[body] = np.cumsum(lens)
            st[body] = en[body] - lens
            c = np.flatnonzero(body & (st <= CENTER) & (en > CENTER))
            assert len(c) == 1
            bi = np.flatnonzero(body)
            k = int(np.searchsorted(bi, c[0]))
            assert k >= 1, "centre token has no previous token"
            c_idx[i], p_idx[i] = c[0], bi[k - 1]
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            hid = token_states(model, enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
            r = torch.arange(hid.shape[0], device=hid.device)
            C = hid[r, torch.from_numpy(c_idx).to(DEVICE)]
            P = hid[r, torch.from_numpy(p_idx).to(DEVICE)]
            out.append(np.asarray(score_fn(C, P), dtype=np.float32))
    return np.concatenate(out)


def scan_gene_windows(rec, model, score_fn, max_positions=None, log=None):
    # 10_scan_gene.py main(), --context window branch: every base of the canonical transcript span
    # is scored from its own 600-nt window centred on it (seq[p - 300 : p + 300]); everything else
    # stays NaN, and the scoring unit of a scored base is its own position.
    fr = gene_frame(rec)
    seq = fr["seq"]
    e0, e1 = fr["transcript_span"]
    positions = np.arange(e0, e1)          # every base of the canonical span
    if max_positions is not None:
        positions = positions[:max_positions]
    assert positions[0] >= CENTER and positions[-1] - CENTER + WINDOW <= len(seq), "window runs off the sequence"
    scores = np.full((len(seq), 3), np.nan, dtype=np.float32)
    units = np.full(len(seq), -1, dtype=np.int64)
    chunk, t0 = 8 * SCAN_BATCH, time.time()
    for a in range(0, len(positions), chunk):
        ps = positions[a:a + chunk]
        scores[ps] = window_pairs([seq[p - CENTER:p - CENTER + WINDOW] for p in ps], model, score_fn)
        if log:
            done = a + len(ps)
            log(f"    {done:>6,}/{len(positions):,} windows  {done / (time.time() - t0):7.1f} windows/s")
    units[positions] = positions
    return fr, positions, scores, units


def finetuned_scores(C, P):
    # Run B's head reads [centre ++ previous] straight from the backbone: no input scaler and no
    # float16 rounding (those belong to the frozen pipeline's 03 heads). Columns: none, acceptor, donor.
    return torch.softmax(head_b(torch.cat([C, P], dim=-1)).float(), dim=-1).cpu().numpy()


def model_slug(repo):
    # phase1_common.py model_slug()
    return repo.replace("/", "__")


def track_name(slug, variant, context, head_kind="original"):
    # 10_scan_gene.py track_name(), with the new head kind added
    return f"{slug}.{variant}{'.' + head_kind if head_kind != 'original' else ''}.{context}"

Run the scan and write the tracks.

In [ ]:
backbone.eval()
head_b.eval()
TRACK = track_name(model_slug(MODEL_REPO), VARIANT, "window", HEAD_KIND)
TRACK_DIR = OUT_DIR / "finetuned_scan_tracks"
todo = [g[0] for g in GENE_LIST][:SMOKE_GENES] if SMOKE else [g[0] for g in GENE_LIST]
max_pos = SMOKE_POSITIONS if SMOKE else None
print(f"track: {TRACK}\nscanning {len(todo)} gene(s), batch {SCAN_BATCH}, fp16 {USE_AMP}")

t_scan = time.time()
manifest_genes, n_windows = {}, 0
for sym in todo:
    rec = GENES[sym]
    t0 = time.time()
    fr, positions, scores, units = scan_gene_windows(rec, backbone, finetuned_scores, max_pos)
    secs = time.time() - t0
    e0, e1 = fr["transcript_span"]
    assert np.isfinite(scores[positions]).all(), f"{sym}: non-finite scores"
    assert np.allclose(scores[positions].sum(1), 1.0, atol=1e-3), f"{sym}: scores do not sum to 1"
    out = TRACK_DIR / sym
    out.mkdir(parents=True, exist_ok=True)
    # Same files and layout as 10_scan_gene.py writes into data/scan/<GENE>/:
    #   <track>.scores.npy  (L, 3) float32 [none, acceptor, donor], NaN = not scored
    #   <track>.units.npy   (L,) int64 scoring unit per base, -1 = not scored
    np.save(out / f"{TRACK}.scores.npy", scores)
    np.save(out / f"{TRACK}.units.npy", units)
    # The scan_meta.json entry 10 would write for this track (14_import_finetuned_tracks.py merges it).
    meta = {TRACK: {
        "model": MODEL_REPO, "variant": VARIANT, "context": "window", "head": HEAD_KIND,
        "window_bases": WINDOW, "positions_scored": int(len(positions)),
        "forward_passes": int(len(positions)),
        "scored_span_coverage": float(np.isfinite(scores[e0:e1, 0]).mean()),
        "features_cache": None, "scan_seconds": round(secs, 1), "scored_from_cached_features": False,
        "head_trained_on": "class-balanced 600-nt windows (~1/3 per class), the same 8,000 as the "
                           "03 head; the backbone was fine-tuned together with the head "
                           "(kaggle/finetune_scan_genes.ipynb, Run B)",
        "device": str(DEVICE), "fp16_autocast": USE_AMP}}
    (out / "scan_meta.finetuned.json").write_text(json.dumps(meta, indent=2) + "\n")
    n_windows += len(positions)
    manifest_genes[sym] = {
        "length_scanned": len(fr["seq"]), "eval_region_idx": [int(e0), int(e1)],
        "positions_scored": int(len(positions)), "strand": rec["strand"],
        "seq_scanned_sha256": hashlib.sha256(fr["seq"].encode()).hexdigest(),
        "scan_seconds": round(secs, 1)}
    d, a = [p for p in fr["donors"] if units[p] >= 0], [p for p in fr["acceptors"] if units[p] >= 0]
    print(f"  {sym} ({'+' if rec['strand'] == 1 else '-'}): {len(positions):,} windows in {secs:.1f}s "
          f"({len(positions) / secs:.0f} windows/s); p(donor) at true donors "
          f"{[round(float(v), 2) for v in scores[d, 2]]}; p(acceptor) at true acceptors "
          f"{[round(float(v), 2) for v in scores[a, 1]]}", flush=True)
scan_secs = time.time() - t_scan

manifest = {
    "track": TRACK, "head": HEAD_KIND, "smoke": SMOKE, "model": MODEL_REPO,
    "model_revision": MODEL_REVISION, "variant": VARIANT, "context": "window", "window_bases": WINDOW,
    "run_b": {"test_accuracy": run_b["accuracy"], "macro_f1": run_b["macro_f1"],
              "recall": {k: v["recall"] for k, v in run_b["per_class"].items()},
              "best_epoch": best_b["epoch"], "epochs": FT_EPOCHS, "history": hist_b,
              "n_fit": int(len(fit_rows)), "n_val": int(len(val_rows)), "n_test": int(len(y["test"])),
              "same_rows_as_local_pipeline": SAME_SAMPLE_AS_LOCAL},
    "device": str(DEVICE), "gpu": GPU_NAME, "fp16_autocast": USE_AMP,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    "genes": manifest_genes, "windows": int(n_windows), "scan_seconds": round(scan_secs, 1),
    "total_seconds": round(time.time() - T_START, 1),
}
(TRACK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

ZIP_PATH = OUT_DIR / "finetuned_scan_tracks.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(TRACK_DIR.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(TRACK_DIR).as_posix())
with zipfile.ZipFile(ZIP_PATH) as z:
    assert z.testzip() is None
    n_files = len(z.namelist())

if SMOKE:
    print("\nSMOKE RUN: one gene, partial scan, tiny training sample. Not a result. Set SMOKE = False.")
print(f"\nRun B test accuracy {run_b['accuracy']:.4f}; scanned {n_windows:,} windows in {len(todo)} gene(s) "
      f"in {scan_secs:.0f}s ({n_windows / scan_secs:.0f} windows/s)")
print(f"wrote {ZIP_PATH} ({n_files} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB) and {STATE_PATH.name}")
print(f"total time: {time.time() - T_START:.0f}s")